# 1. Objectif, contrat expérimental et stratégie de faisabilité du Dynamic GNN

## 1.1 Motivation

Le notebook précédent a évalué un baseline GraphSAGE causal pour la détection de fraude Mobile Money.

Même si ce modèle exploite uniquement la structure historique disponible avant chaque transaction, ses performances restent très inférieures à celles du baseline tabulaire LightGBM Core :

- GraphSAGE Core Edge - Test PR-AUC : **0.352779**
- LightGBM Core - Test PR-AUC : **0.979749**

Le GraphSAGE 1 couche sélectionné obtient également un rappel Test de seulement **0.164752**, ce qui indique qu'un simple message passing spatial ne permet pas d'exploiter suffisamment l'information disponible dans PaySim.

Ce résultat est cohérent avec la structure particulière du graphe transactionnel :

- la majorité des comptes ont un très faible degré ;
- les comptes sources sont presque toujours nouveaux ;
- très peu de comptes jouent à la fois le rôle de source et de destination ;
- les composantes sont petites, peu profondes et majoritairement constituées de paires ou d'étoiles convergentes ;
- les ensembles Validation et Test correspondent à un contexte fortement inductif avec un cold-start massif.

La question suivante n'est donc pas de construire un GraphSAGE plus profond, mais de déterminer si un **état temporel explicitement dynamique** peut mieux exploiter l'historique transactionnel que le message passing statique.

## 1.2 Question scientifique

La question principale de ce notebook est :

> **Un modèle de graphe temporel dynamique peut-il mieux exploiter l'évolution de l'historique transactionnel qu'un GraphSAGE statique dans le contexte fortement inductif de PaySim ?**

L'objectif n'est pas de chercher artificiellement à dépasser LightGBM.

Le notebook doit permettre de déterminer :

1. si des états temporels de nœuds contiennent une information prédictive utile ;
2. si une mémoire dynamique apporte un avantage lorsque la majorité des comptes sources sont nouveaux ;
3. si l'activité répétée autour de certaines destinations peut être exploitée temporellement ;
4. si la complexité supplémentaire d'un modèle temporel est justifiée par un gain mesurable par rapport à GraphSAGE ;
5. quelles limites viennent du modèle lui-même et quelles limites viennent de la structure particulière de PaySim.

La métrique principale reste la **PR-AUC / Average Precision**, adaptée au très fort déséquilibre de classes du dataset.

## 1.3 Contrat expérimental

Les splits temporels validés dans les notebooks précédents restent inchangés :

- Train : steps 1 à 334
- Validation : steps 335 à 392
- Test : steps 393 à 743

Le Train est utilisé uniquement pour l'apprentissage des paramètres du modèle.

La Validation est utilisée pour :

- sélectionner l'architecture temporelle ;
- choisir les hyperparamètres essentiels ;
- sélectionner le seuil de classification.

Le Test est réservé à une évaluation finale après gel complet des décisions expérimentales.

Aucune architecture, aucun hyperparamètre et aucun seuil ne doit être choisi à partir des résultats Test.

Les comparaisons scientifiques principales seront réalisées entre :

- LightGBM Core ;
- GraphSAGE Core Edge ;
- Dynamic GNN utilisant en priorité les features Graph Core Edge.

Les features Graph Full et PaySim-sensitive ne font pas partie du baseline temporel principal. Elles pourront uniquement faire l'objet d'expériences secondaires clairement identifiées.

## 1.4 Protocole causal strict

Pour chaque step temporel \(t\) :

\[
G_{<t} \rightarrow E_t
\]

où :

- \(G_{<t}\) contient uniquement les événements dont le step est strictement inférieur à \(t\) ;
- \(E_t\) contient toutes les transactions du step \(t\).

À partir des offsets temporels validés :

\[
start = offsets[t-1]
\]

\[
end = offsets[t]
\]

on obtient :

\[
G_{<t} = events[:start]
\]

et :

\[
E_t = events[start:end]
\]

L'ordre de traitement obligatoire est donc :

1. construire ou récupérer l'état temporel correspondant strictement à \(G_{<t}\) ;
2. prédire **toutes** les transactions appartenant à \(E_t\) ;
3. calculer la loss correspondant à \(E_t\) ;
4. effectuer les opérations d'optimisation autorisées ;
5. seulement après la production de toutes les prédictions de \(E_t\), intégrer les événements du step \(t\) dans la mémoire et dans le voisinage temporel ;
6. passer ensuite au step \(t+1\).

Ainsi, deux transactions appartenant au même step PaySim ne doivent jamais pouvoir s'influencer mutuellement à travers une mise à jour de mémoire ou de voisinage temporel.

Cette contrainte s'applique au Train, à la Validation et au Test.

## 1.5 Importance du protocole intra-step

De nombreuses implémentations de GNN temporels traitent les interactions séquentiellement.

Une implémentation naïve pourrait suivre le schéma suivant :

\[
e_{t,1}
\rightarrow mise\ à\ jour
\rightarrow e_{t,2}
\rightarrow mise\ à\ jour
\rightarrow ...
\]

Cette procédure créerait une fuite causale dans notre expérimentation.

En effet, la prédiction de \(e_{t,2}\) utiliserait déjà une information issue de \(e_{t,1}\), alors que les deux événements appartiennent au même step temporel.

Le protocole correct doit être :

\[
State_{<t}
\rightarrow
\{Prediction(e): e \in E_t\}
\rightarrow
Update(E_t)
\rightarrow
State_{\leq t}
\]

Autrement dit, toutes les transactions du step \(t\) doivent être prédites à partir du même état historique antérieur au step.

L'architecture et l'implémentation retenues dans ce notebook devront donc permettre de contrôler explicitement le moment où la mémoire et le voisinage temporel sont mis à jour.

Une librairie qui ne permet pas de garantir cette propriété ne pourra pas être utilisée directement sans adaptation du protocole d'exécution.

## 1.6 Candidats pour la modélisation temporelle

Le notebook n'a pas pour objectif de comparer un grand nombre d'architectures temporelles.

Une seule architecture principale sera retenue après un audit scientifique et technique.

### TGN

Temporal Graph Networks constitue le candidat initial le plus naturel car cette architecture permet notamment :

- de maintenir une mémoire persistante pour les nœuds ;
- de mettre à jour cette mémoire à partir des événements observés ;
- d'utiliser des fonctions de message temporelles ;
- d'intégrer un encodage du temps ;
- d'exploiter un voisinage dynamique ;
- de fonctionner dans un contexte inductif sans nécessiter d'embedding d'identité appris pour chaque `node_id`.

TGN pourrait être particulièrement pertinent si les interactions successives vers certaines destinations permettent de construire progressivement un état utile pour la détection de fraude.

Cependant, son utilisation n'est pas considérée comme acquise.

Le nombre très élevé de nœuds de PaySim et le cold-start massif des comptes sources pourraient rendre une mémoire dense coûteuse tout en étant peu informative pour des comptes qui n'apparaissent qu'une seule fois.

### TGAT

TGAT exploite principalement des voisinages temporels à travers un mécanisme d'attention dépendant du temps, sans reposer de la même manière sur une mémoire persistante pour chaque nœud.

Cette approche pourrait être intéressante si la structure temporelle du voisinage contient une information importante.

Cependant, la très faible profondeur du graphe, le faible degré de la majorité des comptes et le caractère fortement inductif de PaySim pourraient limiter l'intérêt d'un mécanisme d'attention temporelle complexe.

Son coût computationnel devra donc être mis en regard de la quantité réelle d'information temporelle disponible.

### Approche temporelle plus légère

Une architecture plus légère pourra être préférée si l'audit montre que :

- la mémoire dense d'un TGN est trop coûteuse ;
- les voisinages temporels sont trop pauvres pour justifier TGAT ;
- l'information utile est principalement concentrée sur les destinations déjà observées ;
- un état récurrent compact permet de capturer l'historique utile sans message passing temporel complexe.

Une telle alternative ne sera retenue que si elle reste réellement dynamique, causale et capable de représenter l'évolution de l'historique transactionnel.

## 1.7 Critères de sélection de l'architecture

Le choix du modèle temporel reposera sur les critères suivants.

### 1. Correction causale

L'implémentation doit permettre de prédire l'ensemble du step \(E_t\) avant toute intégration des événements de ce même step dans la mémoire ou dans le voisinage temporel.

### 2. Capacité inductive

Les nouveaux nœuds doivent pouvoir être traités sans embeddings d'identité appris à partir de `node_id`.

### 3. Faisabilité mémoire

Le stockage des états temporels doit rester compatible avec environ **9.07 millions de nœuds**.

### 4. Pertinence temporelle

Le modèle doit exploiter une information réellement différente de celle déjà disponible pour le GraphSAGE statique.

### 5. Compatibilité avec la topologie PaySim

L'architecture doit rester pertinente pour un graphe très sparse, dominé par des nouveaux comptes sources et par certaines destinations recevant plusieurs transactions.

### 6. Faisabilité runtime

L'entraînement doit être réalisable soit sur la machine locale, soit sur le GPU Tesla T4 disponible sur Google Colab.

### 7. Reprise après interruption

Les expériences longues doivent pouvoir être sauvegardées et reprises facilement sans dépendre d'un état présent uniquement en RAM.

### 8. Fiabilité de l'implémentation

La priorité sera donnée à des composants temporels suffisamment stables et compatibles avec l'environnement PyTorch / PyG disponible.

La complexité du modèle ne sera jamais considérée comme un avantage en elle-même.

## 1.8 Stratégie d'audit de faisabilité

Avant toute implémentation complète du modèle temporel, la section suivante devra auditer précisément l'environnement d'exécution et les composants disponibles.

L'audit vérifiera notamment :

- la version de Python ;
- la version de PyTorch ;
- la version de PyTorch Geometric ;
- la disponibilité de CUDA ;
- le GPU disponible le cas échéant ;
- la disponibilité de `TGNMemory` ou de composants équivalents ;
- la disponibilité d'un neighbor loader temporel ;
- les éventuelles dépendances optionnelles nécessaires ;
- la manière dont la mémoire des nœuds est représentée ;
- la possibilité de réinitialiser, détacher, sauvegarder et restaurer cette mémoire ;
- le comportement du modèle avec des nœuds jamais vus ;
- le moment exact des mises à jour mémoire ;
- le moment exact des mises à jour du voisinage temporel ;
- la possibilité d'appliquer notre protocole de mise à jour différée par step ;
- le coût RAM / VRAM estimé pour une mémoire associée à environ 9.07 millions de nœuds ;
- la faisabilité réelle d'une exécution CPU ;
- la nécessité éventuelle d'un benchmark GPU sur Colab.

Aucun entraînement complet ne sera lancé avant validation de ces points.

## 1.9 Règle de décision

À l'issue de l'audit de faisabilité, une seule direction principale sera retenue :

- **TGN**, si la mémoire temporelle persistante est techniquement réalisable et scientifiquement pertinente ;
- **TGAT ou une autre approche fondée sur le voisinage temporel**, si cette représentation semble mieux adaptée à la structure de PaySim ;
- **une architecture dynamique plus légère**, si elle constitue le meilleur compromis entre modélisation temporelle, cold-start, mémoire et coût computationnel.

Cette décision devra être prise avant toute observation des performances sur le Test.

In [1]:
from pathlib import Path

# Définir les dossiers
graph_dir = Path("../data/graph")
processed_dir = Path("../data/processed")
gnn_dir = Path("../artifacts/gnn")

# Artefacts nécessaires
required_artifacts = {
    "nodes": graph_dir / "paysim_nodes.parquet",
    "src": graph_dir / "paysim_src.npy",
    "dst": graph_dir / "paysim_dst.npy",
    "step": graph_dir / "paysim_step.npy",
    "label": graph_dir / "paysim_label.npy",
    "offsets": graph_dir / "paysim_step_offsets.npy",
    "graph_metadata": graph_dir / "graph_construction_metadata.json",
    "graph_ready": processed_dir / "paysim_graph_ready.parquet",
    "fe_metadata": processed_dir / "feature_engineering_metadata.json",
    "gnn_summary": gnn_dir / "gnn_baseline_summary.json",
    "gnn_final_analysis": gnn_dir / "graphsage_core_edge_final_analysis.json",
}

# Vérifier les fichiers
for name, path in required_artifacts.items():
    print(f"{name:<20} : {path.exists()}")

print()
print("Tous les artefacts :", all(path.exists() for path in required_artifacts.values()))

nodes                : True
src                  : True
dst                  : True
step                 : True
label                : True
offsets              : True
graph_metadata       : True
graph_ready          : True
fe_metadata          : True
gnn_summary          : True
gnn_final_analysis   : True

Tous les artefacts : True


# 2. Audit de l'environnement et faisabilité d'un GNN temporel

## 2.1 Objectif

Avant de sélectionner définitivement une architecture temporelle, il est nécessaire de vérifier que les composants logiciels et les ressources matérielles disponibles permettent une implémentation correcte et reproductible.

Cette étape est particulièrement importante pour TGN, car cette famille de modèles maintient généralement un état temporel associé aux nœuds.

Dans notre cas, le graphe contient environ **9.07 millions de nœuds**. Même une mémoire de faible dimension peut donc représenter plusieurs centaines de mégaoctets, auxquels doivent s'ajouter :

- les timestamps mémorisés ;
- les buffers internes ;
- les gradients ;
- les paramètres du modèle ;
- les messages temporaires ;
- les batches d'événements ;
- les structures de voisinage temporel.

La faisabilité ne doit donc pas être supposée à partir de la seule disponibilité de PyTorch Geometric.

Cette section vérifie successivement :

1. l'environnement Python / PyTorch / PyG ;
2. la disponibilité des composants temporels nécessaires ;
3. l'interface réelle de `TGNMemory` ;
4. l'interface du loader de voisinage temporel ;
5. les opérations disponibles pour gérer proprement l'état temporel ;
6. le coût théorique d'une mémoire dense sur l'ensemble des nœuds PaySim.

Aucun entraînement ne sera effectué dans cette section.

La décision TGN / TGAT / architecture plus légère ne sera prise qu'après analyse de ces résultats.

In [2]:
import sys
import inspect
import platform
import torch
import torch_geometric

# Environnement logiciel
print("Python :", sys.version.split()[0])
print("OS :", platform.platform())
print("PyTorch :", torch.__version__)
print("PyG :", torch_geometric.__version__)
print("CUDA disponible :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
    print(
        "VRAM totale :",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

# Vérifier les composants temporels
try:
    from torch_geometric.nn.models.tgn import TGNMemory
    print("\nTGNMemory disponible : True")
    print("Signature TGNMemory :")
    print(inspect.signature(TGNMemory))
except Exception as e:
    print("\nTGNMemory disponible : False")
    print(type(e).__name__, ":", e)

try:
    from torch_geometric.loader import LastNeighborLoader
    print("\nLastNeighborLoader disponible : True")
    print("Signature LastNeighborLoader :")
    print(inspect.signature(LastNeighborLoader))
except Exception as e:
    print("\nLastNeighborLoader disponible : False")
    print(type(e).__name__, ":", e)

c:\Users\bouza\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python : 3.13.7
OS : Windows-11-10.0.22631-SP0
PyTorch : 2.12.0+cpu
PyG : 2.7.0
CUDA disponible : False

TGNMemory disponible : True
Signature TGNMemory :
(num_nodes: int, raw_msg_dim: int, memory_dim: int, time_dim: int, message_module: Callable, aggregator_module: Callable)

LastNeighborLoader disponible : False
ImportError : cannot import name 'LastNeighborLoader' from 'torch_geometric.loader' (c:\Users\bouza\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch_geometric\loader\__init__.py)


In [3]:
from torch_geometric.nn.models.tgn import TGNMemory

# Méthodes importantes
methods_to_check = [
    "forward",
    "update_state",
    "reset_state",
    "detach",
    "train",
    "eval",
    "state_dict",
    "load_state_dict",
]

print("Méthodes TGNMemory :")

for method_name in methods_to_check:
    method = getattr(TGNMemory, method_name, None)

    if method is None:
        print(f"{method_name:<18} : False")
    else:
        print(f"{method_name:<18} : True")

        try:
            print("  ", inspect.signature(method))
        except Exception:
            pass

Méthodes TGNMemory :
forward            : True
   (self, n_id: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]
update_state       : True
   (self, src: torch.Tensor, dst: torch.Tensor, t: torch.Tensor, raw_msg: torch.Tensor)
reset_state        : True
   (self)
detach             : True
   (self)
train              : True
   (self, mode: bool = True)
eval               : True
   (self) -> Self
state_dict         : True
   (self, *args, destination=None, prefix='', keep_vars=False)
load_state_dict    : True
   (self, state_dict: collections.abc.Mapping[str, typing.Any], strict: bool = True, assign: bool = False)


In [4]:
import psutil

# Mémoire système
vm = psutil.virtual_memory()

print("RAM totale :", round(vm.total / 1024**3, 2), "GB")
print("RAM disponible :", round(vm.available / 1024**3, 2), "GB")
print("RAM utilisée :", round(vm.used / 1024**3, 2), "GB")
print("Utilisation RAM :", vm.percent, "%")

RAM totale : 15.84 GB
RAM disponible : 4.1 GB
RAM utilisée : 11.74 GB
Utilisation RAM : 74.1 %


In [5]:
import inspect

# Import correct des composants TGN
from torch_geometric.nn.models.tgn import (
    LastNeighborLoader,
    IdentityMessage,
    LastAggregator,
    MeanAggregator,
)

print("LastNeighborLoader disponible : True")
print("Signature :")
print(inspect.signature(LastNeighborLoader))

print()
print("IdentityMessage disponible :", IdentityMessage is not None)
print("LastAggregator disponible :", LastAggregator is not None)
print("MeanAggregator disponible :", MeanAggregator is not None)

LastNeighborLoader disponible : True
Signature :
(num_nodes: int, size: int, device=None)

IdentityMessage disponible : True
LastAggregator disponible : True
MeanAggregator disponible : True


In [6]:
# Taille du graphe validée
num_nodes = 9_073_900

# Dimensions candidates
memory_dims = [8, 16, 32, 64]

print(f"Nombre de nœuds : {num_nodes:,}")
print()

for dim in memory_dims:
    memory_bytes = num_nodes * dim * 4
    timestamp_bytes = num_nodes * 8
    assoc_bytes = num_nodes * 8

    memory_gib = memory_bytes / 1024**3
    timestamp_gib = timestamp_bytes / 1024**3
    assoc_gib = assoc_bytes / 1024**3

    minimum_gib = (
        memory_bytes
        + timestamp_bytes
        + assoc_bytes
    ) / 1024**3

    print(
        f"dim={dim:<2} | "
        f"memory={memory_gib:.3f} GiB | "
        f"last_update={timestamp_gib:.3f} GiB | "
        f"_assoc={assoc_gib:.3f} GiB | "
        f"minimum tensors={minimum_gib:.3f} GiB"
    )

Nombre de nœuds : 9,073,900

dim=8  | memory=0.270 GiB | last_update=0.068 GiB | _assoc=0.068 GiB | minimum tensors=0.406 GiB
dim=16 | memory=0.541 GiB | last_update=0.068 GiB | _assoc=0.068 GiB | minimum tensors=0.676 GiB
dim=32 | memory=1.082 GiB | last_update=0.068 GiB | _assoc=0.068 GiB | minimum tensors=1.217 GiB
dim=64 | memory=2.163 GiB | last_update=0.068 GiB | _assoc=0.068 GiB | minimum tensors=2.299 GiB


In [7]:
import inspect
from torch_geometric.nn.models.tgn import TGNMemory, LastNeighborLoader

# Coût théorique du voisinage temporel
num_nodes = 9_073_900
neighbor_sizes = [2, 5, 10, 20]

print("Coût théorique LastNeighborLoader")
print()

for size in neighbor_sizes:
    neighbors_bytes = num_nodes * size * 8
    edge_ids_bytes = num_nodes * size * 8
    assoc_bytes = num_nodes * 8

    total_gib = (
        neighbors_bytes
        + edge_ids_bytes
        + assoc_bytes
    ) / 1024**3

    print(
        f"size={size:<2} | "
        f"minimum tensors={total_gib:.3f} GiB"
    )

print()
print("Initialisation TGNMemory :")
print(inspect.getsource(TGNMemory._reset_message_store))

print("Initialisation LastNeighborLoader :")
print(inspect.getsource(LastNeighborLoader.__init__))

Coût théorique LastNeighborLoader

size=2  | minimum tensors=0.338 GiB
size=5  | minimum tensors=0.744 GiB
size=10 | minimum tensors=1.420 GiB
size=20 | minimum tensors=2.772 GiB

Initialisation TGNMemory :
    def _reset_message_store(self):
        i = self.memory.new_empty((0, ), device=self.device, dtype=torch.long)
        msg = self.memory.new_empty((0, self.raw_msg_dim), device=self.device)
        # Message store format: (src, dst, t, msg)
        self.msg_s_store = {j: (i, i, i, msg) for j in range(self.num_nodes)}
        self.msg_d_store = {j: (i, i, i, msg) for j in range(self.num_nodes)}

Initialisation LastNeighborLoader :
    def __init__(self, num_nodes: int, size: int, device=None):
        self.size = size

        self.neighbors = torch.empty((num_nodes, size), dtype=torch.long,
                                     device=device)
        self.e_id = torch.empty((num_nodes, size), dtype=torch.long,
                                device=device)
        self._assoc = t

### 2.2 Résultats de l'audit de faisabilité

L'environnement logiciel dispose des principaux composants nécessaires à l'implémentation d'un Temporal Graph Network :

- Python : **3.13.7**
- PyTorch : **2.12.0+cpu**
- PyTorch Geometric : **2.7.0**
- `TGNMemory` : disponible
- `LastNeighborLoader` : disponible
- `IdentityMessage` : disponible
- `LastAggregator` : disponible
- `MeanAggregator` : disponible

L'environnement local ne dispose cependant pas de CUDA. Une exécution locale utiliserait donc exclusivement le CPU.

`TGNMemory` expose également les opérations nécessaires à la gestion d'un état temporel :

- `update_state`
- `reset_state`
- `detach`
- `state_dict`
- `load_state_dict`

L'utilisation conceptuelle de TGN est donc compatible avec l'environnement logiciel disponible.

### Coût de la mémoire dense

Le principal problème identifié concerne cependant le passage à l'échelle.

Pour **9 073 900 nœuds**, le coût minimum des principaux tenseurs persistants de `TGNMemory` est estimé à :

| Dimension mémoire | Coût minimum |
|---:|---:|
| 8 | 0.406 GiB |
| 16 | 0.676 GiB |
| 32 | 1.217 GiB |
| 64 | 2.299 GiB |

Ces valeurs comprennent uniquement :

- le vecteur mémoire des nœuds ;
- `last_update` ;
- `_assoc`.

Elles n'incluent ni les messages temporels, ni les paramètres du modèle, ni les activations, ni les gradients, ni les features transactionnelles.

### Coût du voisinage temporel

`LastNeighborLoader` maintient également des structures denses indexées par tous les nœuds.

Le coût minimum observé est :

| Taille du voisinage | Coût minimum |
|---:|---:|
| 2 | 0.338 GiB |
| 5 | 0.744 GiB |
| 10 | 1.420 GiB |
| 20 | 2.772 GiB |

Une configuration relativement compacte avec une mémoire de dimension 16 et un voisinage de taille 10 nécessiterait donc déjà environ :

\[
0.676 + 1.420 = 2.096\ GiB
\]

de tenseurs persistants minimums.

Cette estimation reste très inférieure au coût réel d'exécution.

### Limitation critique de `TGNMemory`

L'inspection de l'implémentation révèle une limitation plus importante encore.

Lors de l'initialisation de son message store, `TGNMemory` construit deux dictionnaires Python contenant une entrée pour chaque nœud :

- `msg_s_store`
- `msg_d_store`

Avec 9 073 900 nœuds, cela correspond à **18 147 800 entrées de dictionnaires**.

Cette stratégie est acceptable pour des graphes temporels plus petits, mais elle est particulièrement défavorable pour PaySim, dont la majorité des nœuds apparaissent très peu et dont environ 95 % ont un degré total inférieur ou égal à deux.

Une grande quantité de mémoire serait ainsi réservée pour maintenir des structures associées à des comptes disposant de très peu d'historique exploitable.

### Conséquence sur la faisabilité locale

Au moment de l'audit, la machine dispose de :

- RAM totale : **15.84 GB**
- RAM disponible : **4.1 GB**

L'instanciation directe d'un `TGNMemory` et d'un `LastNeighborLoader` standards sur l'ensemble des 9.07 millions de nœuds présenterait donc un risque mémoire important.

Une augmentation de la dimension mémoire ou de la taille du voisinage aggraverait rapidement ce problème.

Le passage à un GPU ne résout pas automatiquement cette limitation, puisque le problème ne concerne pas uniquement le calcul matriciel mais également la représentation globale de l'état temporel et les structures Python associées.

### Compatibilité avec le protocole causal

L'API disponible permet néanmoins de contrôler explicitement les opérations suivantes :

\[
Prediction(E_t)
\rightarrow
update\_state(E_t)
\rightarrow
insert(E_t)
\]

Cette propriété est importante pour notre protocole.

La mémoire et le voisinage ne devront jamais être mis à jour entre deux sous-batches appartenant au même step.

Si un step doit être découpé pour limiter la mémoire de calcul, l'ensemble de ses sous-batches devra être prédit à partir du même état \(State_{<t}\).

Les événements du step ne pourront être intégrés qu'après production de toutes les prédictions de \(E_t\).

### Conclusion de l'audit

L'audit conduit à distinguer deux questions.

**Scientifiquement**, TGN reste un candidat pertinent :

- il représente explicitement l'évolution temporelle ;
- il maintient un état dépendant des événements passés ;
- il est compatible avec un contexte inductif ;
- il peut potentiellement exploiter l'activité répétée des destinations.

**Techniquement**, l'implémentation PyG standard n'est pas adaptée telle quelle à la structure complète de PaySim.

Le principal problème n'est pas uniquement la dimension des embeddings temporels, mais la combinaison :

- d'une mémoire dense pour tous les nœuds ;
- d'un voisinage temporel dense ;
- de message stores initialisés pour tous les comptes ;
- d'un graphe contenant plus de neuf millions de nœuds dont la grande majorité possède très peu d'historique.

Le scénario retenu à ce stade est donc :

> **TGN est pertinent sur le plan scientifique, mais son implémentation standard doit être adaptée ou remplacée par une représentation temporelle plus parcimonieuse avant tout entraînement à grande échelle.**

La prochaine étape doit déterminer si cette adaptation est préférable à une architecture temporelle plus légère directement conçue pour la structure fortement inductive et faiblement récurrente de PaySim.

# 3. Sélection de l'architecture temporelle

## 3.1 Objectif

L'audit précédent montre qu'un modèle temporel doit satisfaire simultanément deux contraintes :

1. représenter explicitement l'évolution de l'historique transactionnel ;
2. rester compatible avec la structure très sparse et fortement inductive de PaySim.

Le choix ne doit donc pas reposer uniquement sur les performances publiées des architectures temporelles classiques, mais sur leur adéquation au graphe étudié.

Trois directions sont considérées :

- TGN standard ;
- TGAT ;
- une architecture temporelle parcimonieuse inspirée de TGN.

L'objectif de cette section est de sélectionner une seule architecture principale avant toute implémentation complète.

## 3.2 TGN standard

TGN est particulièrement adapté aux graphes représentés comme des séquences d'événements temporels.

Son principe général repose sur :

- une mémoire dynamique associée aux nœuds ;
- la génération de messages à partir des interactions ;
- l'agrégation de ces messages ;
- une mise à jour récurrente de la mémoire ;
- éventuellement une agrégation du voisinage temporel pour construire les représentations finales.

Cette approche correspond bien à la question scientifique du notebook : déterminer si l'évolution de l'état transactionnel apporte une information absente du GraphSAGE statique.

TGN possède également un avantage important dans notre contexte : l'état d'un compte peut évoluer au fil des transactions sans nécessiter un embedding d'identité appris à partir de `node_id`.

Cependant, l'audit de faisabilité a montré que l'implémentation PyG standard n'est pas adaptée telle quelle aux **9 073 900 nœuds** de PaySim.

Elle utilise notamment :

- une mémoire dense indexée par tous les nœuds ;
- des timestamps d'état denses ;
- des structures d'association denses ;
- deux message stores Python contenant une entrée pour chaque nœud ;
- un `LastNeighborLoader` utilisant également des matrices indexées par tous les nœuds.

L'utilisation directe du pipeline TGN PyG complet entraînerait donc un coût mémoire important pour des millions de comptes qui disposent en pratique de très peu d'historique.

TGN reste ainsi pertinent en tant que principe de modélisation, mais son implémentation standard n'est pas retenue.

## 3.3 TGAT

TGAT construit des représentations temporelles en agrégeant les voisins historiques à l'aide d'un mécanisme d'attention dépendant du temps.

Cette architecture possède plusieurs qualités :

- fonctionnement inductif ;
- prise en compte explicite du temps ;
- possibilité de représenter des nœuds nouveaux ;
- exploitation directe des voisinages temporels.

Cependant, plusieurs propriétés de PaySim réduisent son intérêt dans notre expérimentation.

La grande majorité des sources apparaissent une seule fois ou presque. Leurs représentations temporelles disposent donc de très peu d'historique.

Les composantes du graphe sont également très peu profondes et majoritairement constituées de paires et d'étoiles convergentes.

Dans ce contexte, pour une transaction de type :

\[
New \rightarrow Known
\]

le voisinage historique réellement informatif est essentiellement constitué des transactions antérieures reçues par la destination connue.

Une attention temporelle complète traiterait donc principalement une succession de sources peu récurrentes autour d'une même destination.

Le notebook 06 a par ailleurs montré qu'une augmentation de la profondeur du message passing spatial de un à deux hops dégrade fortement les performances :

- GraphSAGE 1 couche - Validation PR-AUC : **0.240136**
- GraphSAGE 2 couches - Validation PR-AUC : **0.001883**

Ce résultat ne démontre pas que l'attention temporelle serait inefficace, mais il réduit la justification scientifique d'une architecture de voisinage plus complexe sur cette topologie.

TGAT ajouterait également une complexité d'implémentation et de sampling temporel importante alors que l'information historique utile semble fortement concentrée autour des destinations récurrentes.

TGAT n'est donc pas retenu comme architecture principale.

## 3.4 Où se trouve réellement l'information temporelle dans PaySim ?

Les statistiques de cold-start fournissent une indication essentielle.

Sur le Test :

- Known → Known ≈ **0.1640 %**
- New → Known ≈ **56.3939 %**
- Known → New ≈ **0.1253 %**
- New → New ≈ **43.3168 %**

Ainsi, dans l'immense majorité des transactions :

- la source est nouvelle ;
- la destination est soit connue, soit nouvelle.

Pour les transactions **New → New**, aucun des deux comptes ne possède d'historique exploitable.

Un modèle temporel doit donc naturellement retomber sur les features de la transaction et sur un état initial partagé.

En revanche, pour les transactions **New → Known**, qui représentent environ **56.4 % du Test**, la destination possède un historique causal.

Ce cas constitue le principal espace dans lequel une mémoire temporelle peut apporter une information supplémentaire.

La structure du problème suggère donc une représentation asymétrique :

\[
transaction_t =
(source_t,\ destination_t,\ edge_t)
\]

avec :

- un état initial partagé pour une source sans historique ;
- un état temporel persistant pour une destination déjà observée ;
- les features de la transaction courante ;
- un encodage du temps écoulé depuis la dernière mise à jour.

Cette représentation correspond mieux à la structure réelle de PaySim qu'une mémoire dense allouée indistinctement à tous les comptes.

## 3.5 Architecture retenue

L'architecture principale retenue sera une :

> **architecture temporelle inspirée de TGN à mémoire parcimonieuse et causale**

Il ne s'agit pas d'une reproduction exacte de l'implémentation `TGNMemory` de PyTorch Geometric.

L'objectif est de conserver les mécanismes temporels utiles de TGN tout en supprimant les structures denses qui ne sont pas adaptées à PaySim.

Le principe général sera :

\[
State_{<t}
+
EdgeFeatures_t
+
TimeEncoding_t
\rightarrow
FraudPrediction_t
\]

puis, uniquement après toutes les prédictions du step :

\[
Messages(E_t)
\rightarrow
Aggregation
\rightarrow
GRU
\rightarrow
State_{\leq t}
\]

La mémoire sera créée uniquement lorsqu'un historique doit réellement être conservé.

Aucun embedding d'identité appris à partir de `node_id` ne sera utilisé.

## 3.6 Représentation de l'état temporel

Pour une transaction :

\[
u \rightarrow v
\]

le modèle récupérera :

\[
m_u^{<t}
\]

et :

\[
m_v^{<t}
\]

lorsque ces états existent.

Pour un nœud jamais observé dans l'historique pertinent :

\[
m^{<t} = m_0
\]

où \(m_0\) représente un état initial partagé et non un embedding spécifique au `node_id`.

Compte tenu du cold-start massif des sources, la persistance de millions d'états associés à des comptes sources utilisés une seule fois apporterait peu de valeur pour un coût mémoire important.

La priorité sera donc donnée à la conservation de l'état des nœuds disposant effectivement d'un historique réutilisable, en particulier les destinations récurrentes.

L'état d'une source pourra néanmoins être récupéré si ce compte possède déjà un état historique pertinent.

## 3.7 Message temporel

Après la prédiction du step \(t\), chaque transaction pourra produire un message à partir de :

- l'état source avant le step ;
- l'état destination avant le step ;
- les features Graph Core Edge ;
- un encodage du temps écoulé.

Schématiquement :

\[
msg_t =
f_{msg}
(
m_u^{<t},
m_v^{<t},
x_{uv}^{t},
\Delta t
)
\]

Ces messages servent uniquement à construire l'état utilisé pour les steps futurs.

Le label `isFraud` n'intervient jamais dans la construction du message ou de la mémoire.

## 3.8 Agrégation des événements d'un même step

Plusieurs transactions du même step peuvent concerner le même compte.

Elles doivent toutes être prédites à partir du même état \(State_{<t}\).

Il est donc interdit d'appliquer une mise à jour récurrente transaction par transaction à l'intérieur du step.

Les messages produits par les événements d'un même step et associés à un même nœud doivent d'abord être regroupés :

\[
\{msg_1, msg_2, ..., msg_k\}
\rightarrow
Aggregate
\]

avant une seule mise à jour de l'état.

L'agrégation doit être indépendante de l'ordre arbitraire des transactions partageant le même timestamp.

Une stratégie fondée sur le « dernier événement » n'est donc pas appropriée lorsque plusieurs événements possèdent exactement le même step.

Une agrégation permutation-invariante, telle qu'une moyenne ou une somme, sera testée lors du micro-pilote.

## 3.9 Mise à jour de la mémoire

L'état agrégé sera mis à jour à l'aide d'une cellule récurrente légère, par exemple une `GRUCell` :

\[
m_v^{\leq t}
=
GRU(
AggregatedMessage_v^t,
m_v^{<t}
)
\]

Cette approche permet :

- de conserver un historique compact ;
- d'apprendre quelles informations passées doivent être conservées ;
- d'éviter le stockage explicite de tous les voisins historiques ;
- de maintenir une complexité indépendante de la longueur complète de l'historique du compte.

La dimension de mémoire ne sera pas fixée définitivement avant le micro-pilote.

Des dimensions faibles seront privilégiées afin de limiter le coût mémoire.

## 3.10 Prédiction de fraude

La prédiction d'une transaction reposera sur la combinaison :

\[
[
m_u^{<t},
m_v^{<t},
x_{uv}^{t},
TimeEncoding
]
\]

suivie d'un MLP léger produisant un logit de fraude.

Les features principales resteront les **Graph Core Edge**.

Ainsi, le gain éventuel par rapport au GraphSAGE du notebook 06 pourra être attribué principalement à l'introduction d'une représentation temporelle dynamique, plutôt qu'à l'ajout de features PaySim-sensitive.

## 3.11 Gestion du cold-start

Le modèle possède un comportement explicite pour chaque catégorie.

### New → New

Aucun historique n'est disponible.

La prédiction repose principalement sur :

- l'état initial partagé ;
- les features de l'arête ;
- l'encodage temporel.

### New → Known

La source utilise l'état initial tandis que la destination fournit un état historique dynamique.

Cette catégorie constitue le principal cas dans lequel un gain temporel est attendu.

### Known → New

L'état historique de la source peut être utilisé si celui-ci existe.

Cette catégorie reste cependant très rare dans PaySim.

### Known → Known

Les deux états temporels peuvent être exploités.

Cette catégorie représente également une très faible fraction des transactions.

Cette décomposition permettra d'interpréter précisément l'origine d'un éventuel gain du modèle temporel.

## 3.12 Pourquoi cette architecture est retenue

Cette architecture constitue le meilleur compromis entre les propriétés suivantes :

- modélisation explicitement temporelle ;
- fonctionnement inductif ;
- respect du protocole causal par step ;
- mémoire persistante pour les comptes disposant réellement d'un historique ;
- absence d'embeddings d'identité ;
- absence de voisinage dense global ;
- absence de message passing multi-hop coûteux ;
- compatibilité avec la structure en étoiles de PaySim ;
- possibilité d'exploiter les destinations récurrentes ;
- comportement explicite pour le cold-start ;
- possibilité d'une implémentation memory-efficient.

Elle conserve ainsi le mécanisme central qui nous intéresse dans TGN — **l'évolution d'un état de nœud à partir d'événements temporels** — sans reproduire les choix de stockage de l'implémentation PyG qui sont mal adaptés à notre dataset.

Le modèle devra être présenté comme une **architecture temporelle inspirée de TGN**, et non comme une reproduction exacte du TGN standard.

## 3.13 Hypothèse expérimentale

L'hypothèse principale devient :

> Une mémoire temporelle compacte associée aux comptes disposant d'un historique, notamment aux destinations récurrentes, peut améliorer la détection des transactions `New → Known` par rapport à un GraphSAGE statique, sans nécessiter un message passing temporel profond sur l'ensemble du graphe.

Cette hypothèse sera testée sans supposer qu'elle doit conduire à des performances supérieures à LightGBM Core.

Une absence de gain constituerait également un résultat scientifique pertinent sur les limites du Graph Learning dynamique dans PaySim.

In [8]:
import numpy as np
import psutil

# Charger sans copier les gros arrays
dst = np.load("../data/graph/paysim_dst.npy", mmap_mode="r")
offsets = np.load("../data/graph/paysim_step_offsets.npy", mmap_mode="r")

# Fin du Train
train_end = int(offsets[334])

print("Transactions Train :", f"{train_end:,}")

# Destinations du Train
train_dst = dst[:train_end]

# Comptage exact des destinations uniques
unique_train_dst = np.unique(train_dst)

print("Destinations uniques Train :", f"{len(unique_train_dst):,}")
print(
    "Part des 9,073,900 nœuds :",
    f"{100 * len(unique_train_dst) / 9_073_900:.2f} %"
)

# Coût d'une mémoire destination-only
for dim in [8, 16, 32]:
    gib = len(unique_train_dst) * dim * 4 / 1024**3
    print(f"memory_dim={dim:<2} : {gib:.3f} GiB")

del unique_train_dst

print()
print(
    "RAM disponible après calcul :",
    round(psutil.virtual_memory().available / 1024**3, 2),
    "GB"
)

Transactions Train : 4,771,387
Destinations uniques Train : 2,035,605
Part des 9,073,900 nœuds : 22.43 %
memory_dim=8  : 0.061 GiB
memory_dim=16 : 0.121 GiB
memory_dim=32 : 0.243 GiB

RAM disponible après calcul : 4.14 GB


### 3.14 Validation empirique du choix architectural

Afin de vérifier que la stratégie de mémoire parcimonieuse est réellement compatible avec PaySim, le nombre de destinations distinctes présentes dans le Train a été mesuré directement à partir des artefacts temporels existants.

Le Train contient :

- **4 771 387 transactions**
- **2 035 605 destinations uniques**

Ces destinations représentent seulement :

[\
22.43%\
]

des **9 073 900 nœuds** du graphe complet.

Cette différence est importante.

Une architecture TGN standard réserve des structures pour tous les nœuds, alors que notre analyse montre qu'une grande partie des comptes PaySim n'a pas besoin d'un état temporel persistant.

Pour une mémoire `float32` associée uniquement aux destinations observées dans le Train, le coût théorique est :

| Dimension mémoire |      Coût |
| ----------------: | --------: |
|                 8 | 0.061 GiB |
|                16 | 0.121 GiB |
|                32 | 0.243 GiB |

Ces valeurs restent très faibles par rapport à la RAM disponible et montrent qu'une mémoire temporelle compacte est techniquement réaliste.

Même une dimension de 32 permettrait de maintenir l'état des destinations du Train avec moins de 0.25 GiB pour le tenseur de mémoire lui-même.

### Conséquence architecturale

Ce résultat confirme que le principal problème du TGN standard n'est pas la notion de mémoire temporelle, mais son **mode d'allocation global**.

Dans PaySim :

- la majorité des comptes sources disposent de très peu d'historique réutilisable ;
- les destinations constituent la principale population de nœuds pour laquelle un état temporel peut être exploité plusieurs fois ;
- environ 56 % des transactions Test sont de type `New → Known`, ce qui rend précisément l'historique de destination particulièrement intéressant.

Il n'est donc pas nécessaire de maintenir une structure temporelle dense pour les 9.07 millions de comptes.

### Architecture définitivement retenue

L'architecture principale du notebook sera une :

> **Dynamic GNN inspirée de TGN avec mémoire temporelle parcimonieuse et mise à jour causale par step.**

Ses principes sont désormais fixés :

1. les features principales restent les **Graph Core Edge** ;
2. aucun embedding appris directement à partir de `node_id` n'est utilisé ;
3. un état initial partagé est utilisé pour les nœuds sans historique ;
4. la mémoire persistante est créée uniquement pour les comptes dont l'historique doit être conservé ;
5. la priorité est donnée aux destinations récurrentes ;
6. une source déjà connue pourra également exploiter un état historique si celui-ci existe ;
7. l'état temporel est mis à jour à l'aide d'un mécanisme récurrent léger ;
8. plusieurs messages concernant un même nœud dans un même step sont agrégés avant la mise à jour ;
9. toutes les transactions du step (t) sont prédites à partir de l'état (State\_{\<t}) ;
10. aucune mise à jour mémoire ou de voisinage ne peut intervenir avant la fin des prédictions de (E\_t).

Le schéma causal principal devient :

[\
State\_{\<t}\
+\
EdgeFeatures\_t\
+\
TimeEncoding\_t\
\rightarrow\
Prediction(E\_t)\
]

puis :

[\
Messages(E\_t)\
\rightarrow\
Aggregation\
\rightarrow\
GRU\
\rightarrow\
State\_{\leq t}\
]

Cette architecture conserve ainsi l'idée centrale des Temporal Graph Networks - faire évoluer la représentation des nœuds à partir d'une séquence d'événements - tout en supprimant les structures globales inutiles pour la topologie extrêmement sparse de PaySim.

### Hypothèse finale

L'hypothèse expérimentale retenue est donc :

> **Une mémoire temporelle compacte des comptes récurrents, principalement des destinations, peut fournir une information dynamique utile pour la détection de fraude, en particulier dans les transactions&#x20;****`New → Known`****, sans nécessiter un message passing temporel profond ni une mémoire dense sur l'ensemble du graphe.**

Le gain éventuel sera évalué par rapport au GraphSAGE Core Edge et au LightGBM Core déjà validés.

Une performance inférieure à LightGBM restera considérée comme un résultat valide si elle permet de mieux comprendre la valeur réelle de la dynamique temporelle dans PaySim.


# 4. Flux événementiel et entrées temporelles

## 4.1 Objectif

L'architecture retenue doit traiter PaySim comme une séquence ordonnée d'événements transactionnels.

Pour chaque step \(t\), le modèle reçoit :

\[
E_t =
\{
(src_i,\ dst_i,\ step_i,\ x_i,\ y_i)
\}
\]

où :

- `src_i` représente le compte source ;
- `dst_i` représente le compte destination ;
- `step_i` représente le step temporel ;
- `x_i` contient les features de la transaction ;
- `y_i` contient uniquement le label utilisé pour la fonction de coût.

Le label n'intervient jamais dans la construction de l'état temporel, du message ou des représentations de nœuds.

Le flux reste strictement ordonné selon les offsets temporels construits dans le notebook 05.

## 4.2 Unité temporelle

PaySim regroupe plusieurs transactions dans un même `step`.

Le step constitue donc l'unité causale fondamentale du notebook.

Pour un step \(t\) :

\[
start_t = offsets[t-1]
\]

\[
end_t = offsets[t]
\]

et :

\[
E_t = events[start_t:end_t]
\]

Toutes les transactions comprises dans cette tranche doivent être prédites à partir du même état historique :

\[
State_{<t}
\]

indépendamment de leur position physique dans les fichiers.

L'ordre des lignes à l'intérieur d'un même step ne doit donc jamais créer un ordre causal artificiel.

## 4.3 Features transactionnelles principales

Le modèle dynamique principal utilisera les mêmes **Graph Core Edge** que le GraphSAGE du notebook 06 :

- `amount`
- `log_amount`
- `type`
- `origin_balance_change`
- `destination_balance_change`
- `origin_zero_before`
- `origin_zero_after`
- `destination_zero_before`
- `destination_zero_after`

Après preprocessing :

- `type` est représenté par cinq variables one-hot ;
- les variables numériques utilisent la normalisation ajustée uniquement sur le Train ;
- les tenseurs utilisés par PyTorch sont en `float32`.

La dimension finale attendue est donc :

\[
13
\]

Cette représentation doit autant que possible réutiliser exactement le preprocessing du notebook 06 afin que la comparaison avec GraphSAGE Core Edge reste équitable.

## 4.4 Séparation entre features statiques et état temporel

Les features Graph Node History ne sont pas utilisées dans l'expérience principale.

Les variables suivantes sont donc exclues dans un premier temps :

- `orig_prev_transactions`
- `dest_prev_transactions`
- `orig_prev_volume`
- `dest_prev_volume`
- `orig_time_since_last`
- `dest_time_since_last`

Ces variables représentent déjà explicitement l'historique transactionnel.

Les inclure simultanément à la mémoire dynamique compliquerait l'interprétation scientifique d'un éventuel gain.

Dans l'expérience principale, l'historique devra donc provenir du **state dynamique appris** plutôt que de résumés historiques déjà construits manuellement.

Une expérience secondaire avec Graph Core pourra éventuellement être étudiée plus tard uniquement si elle répond à une question scientifique distincte.

## 4.5 Entrées temporelles par transaction

Pour une transaction :

\[
u \rightarrow v
\]

au step \(t\), la prédiction utilisera conceptuellement :

\[
[
x_{uv}^{t},
m_u^{<t},
m_v^{<t},
\Delta t_u,
\Delta t_v,
known_u,
known_v
]
\]

où :

- \(x_{uv}^{t}\) représente les Graph Core Edge ;
- \(m_u^{<t}\) représente l'état temporel disponible pour la source ;
- \(m_v^{<t}\) représente l'état temporel disponible pour la destination ;
- \(\Delta t_u\) représente le temps depuis la dernière mise à jour connue de la source ;
- \(\Delta t_v\) représente le temps depuis la dernière mise à jour connue de la destination ;
- `known_u` indique si un état historique existe pour la source ;
- `known_v` indique si un état historique existe pour la destination.

Pour un nœud sans historique, un état initial partagé sera utilisé.

La valeur temporelle associée à un nœud nouveau ne devra pas être confondue avec un véritable \(\Delta t = 0\). Le masque `known/new` permettra de distinguer explicitement ces situations.

## 4.6 Temps écoulé

Le temps écoulé sera calculé à partir de l'état dynamique lui-même :

\[
\Delta t_v = t - last\_update_v
\]

uniquement lorsque le nœud possède un historique antérieur.

Cette stratégie évite de dépendre directement des features historiques préparées dans le notebook 03.

Elle garantit également que :

\[
last\_update_v < t
\]

pour chaque état utilisé lors de la prédiction.

Les transactions appartenant au même step ne peuvent donc jamais modifier le `last_update` utilisé par les autres transactions du même step.

## 4.7 Encodage temporel

Les valeurs brutes de \(\Delta t\) peuvent avoir des amplitudes différentes.

Le modèle utilisera donc un encodage temporel compact avant leur combinaison avec les états de mémoire.

L'encodage exact ne sera pas fixé avant le micro-pilote, mais il devra respecter les contraintes suivantes :

- faible dimension ;
- absence de paramètres dépendant du `node_id` ;
- même transformation en Train, Validation et Test ;
- comportement défini pour les nœuds sans historique ;
- coût négligeable par rapport au mécanisme de mémoire.

Une transformation simple de type :

\[
\log(1+\Delta t)
\]

constitue une première option robuste.

Un encodage temporel appris léger pourra être retenu uniquement si le micro-pilote montre qu'il est nécessaire.

## 4.8 Gestion des nouveaux nœuds

Pour un nœud nouveau :

\[
known = 0
\]

et l'état temporel utilisé pour la prédiction correspond à l'état initial partagé.

Pour un nœud connu :

\[
known = 1
\]

et son état dynamique historique est récupéré.

La distinction `Known/New` est calculée uniquement à partir des événements strictement antérieurs au step courant.

Un nœud qui apparaît pour la première fois plusieurs fois dans le même step reste donc considéré comme **New pour toutes les transactions de ce step**.

Il ne devient Known qu'à partir du step suivant.

Cette propriété est indispensable pour préserver la causalité intra-step.

## 4.9 Représentation mémoire parcimonieuse

L'expérience précédente a montré que seulement **2 035 605 destinations uniques** apparaissent dans le Train, soit **22.43 %** des nœuds du graphe complet.

Il n'est donc pas nécessaire de réserver un vecteur mémoire pour chaque `node_id`.

La représentation temporelle utilisera une correspondance entre :

\[
node\_id
\rightarrow
memory\_slot
\]

où seuls les nœuds nécessitant effectivement un état persistant reçoivent un slot de mémoire.

Cette stratégie permet de séparer :

- l'identifiant global du graphe ;
- la position compacte utilisée dans la mémoire dynamique.

Le `node_id` sert uniquement à retrouver l'état et ne constitue jamais une feature apprise.

La structure exacte de cette correspondance sera définie dans la section consacrée à la mémoire.

## 4.10 Pipeline événementiel attendu

Pour chaque step \(t\), le pipeline logique sera :

1. récupérer `src`, `dst`, `step` et les Graph Core Edge de \(E_t\) ;
2. récupérer les états source et destination existant avant \(t\) ;
3. calculer les indicateurs Known/New ;
4. calculer les \(\Delta t\) à partir des derniers updates strictement antérieurs ;
5. produire toutes les prédictions de \(E_t\) ;
6. calculer la loss lorsque les labels sont disponibles pour l'apprentissage ;
7. produire les messages temporels du step ;
8. agréger les messages appartenant au même nœud ;
9. mettre à jour la mémoire ;
10. enregistrer `last_update = t` pour les nœuds mis à jour ;
11. passer au step suivant.

Cette séquence doit rester identique en Train, Validation et Test, à l'exception de l'optimisation des poids qui est désactivée après le Train.

In [9]:
import pyarrow.parquet as pq

# Inspecter le fichier sans charger les données
graph_ready_path = "../data/processed/paysim_graph_ready.parquet"

pf = pq.ParquetFile(graph_ready_path)

print("Nombre de lignes :", f"{pf.metadata.num_rows:,}")
print("Nombre de row groups :", pf.metadata.num_row_groups)

print("\nColonnes :")
for i, name in enumerate(pf.schema_arrow.names, start=1):
    print(f"{i:>2}. {name}")

Nombre de lignes : 6,362,620
Nombre de row groups : 26

Colonnes :
 1. nameOrig
 2. nameDest
 3. step
 4. split
 5. amount
 6. log_amount
 7. type
 8. origin_balance_change
 9. destination_balance_change
10. origin_zero_before
11. origin_zero_after
12. destination_zero_before
13. destination_zero_after
14. orig_prev_transactions
15. dest_prev_transactions
16. orig_prev_volume
17. dest_prev_volume
18. orig_time_since_last
19. dest_time_since_last
20. amount_to_oldbalance_ratio
21. origin_balance_residual
22. destination_balance_residual
23. origin_emptied
24. amount_equals_oldbalance
25. cycle_sin
26. cycle_cos
27. simulated_day
28. dest_active_last_24
29. isFraud


In [10]:
import numpy as np

# Charger les arrays en mmap
src = np.load("../data/graph/paysim_src.npy", mmap_mode="r")
dst = np.load("../data/graph/paysim_dst.npy", mmap_mode="r")
step = np.load("../data/graph/paysim_step.npy", mmap_mode="r")
label = np.load("../data/graph/paysim_label.npy", mmap_mode="r")
offsets = np.load("../data/graph/paysim_step_offsets.npy", mmap_mode="r")

arrays = {
    "src": src,
    "dst": dst,
    "step": step,
    "label": label,
    "offsets": offsets,
}

for name, arr in arrays.items():
    print(
        f"{name:<8} | "
        f"shape={arr.shape} | "
        f"dtype={arr.dtype}"
    )

src      | shape=(6362620,) | dtype=int32
dst      | shape=(6362620,) | dtype=int32
step     | shape=(6362620,) | dtype=int16
label    | shape=(6362620,) | dtype=int8
offsets  | shape=(744,) | dtype=int64


In [11]:
import json
from pprint import pprint

config_path = "../artifacts/gnn/graphsage_core_edge_1layer_config.json"

with open(config_path, "r", encoding="utf-8") as f:
    gnn_config = json.load(f)

print("Clés principales :")
print(list(gnn_config.keys()))

print("\nConfiguration complète :")
pprint(gnn_config, sort_dicts=False)

Clés principales :
['feature_set', 'input_dim', 'hidden_dim', 'num_layers', 'num_hops', 'aggregation', 'dropout', 'batch_size', 'learning_rate', 'weight_decay', 'pos_weight', 'train_start_step', 'train_end_step', 'random_state', 'epochs', 'shuffle', 'device']

Configuration complète :
{'feature_set': 'graph_core_edge',
 'input_dim': 13,
 'hidden_dim': 32,
 'num_layers': 1,
 'num_hops': 1,
 'aggregation': 'sum',
 'dropout': 0.1,
 'batch_size': 4096,
 'learning_rate': 0.001,
 'weight_decay': 1e-05,
 'pos_weight': 1273.0686248331108,
 'train_start_step': 1,
 'train_end_step': 334,
 'random_state': 42,
 'epochs': 1,
 'shuffle': False,
 'device': 'cpu'}


In [12]:
# Colonnes Core Edge
core_edge_cols = [
    "amount",
    "log_amount",
    "type",
    "origin_balance_change",
    "destination_balance_change",
    "origin_zero_before",
    "origin_zero_after",
    "destination_zero_before",
    "destination_zero_after",
]

print("Types physiques Graph Core Edge")
print()

schema = pf.schema_arrow

for col in core_edge_cols:
    print(f"{col:<30} : {schema.field(col).type}")

Types physiques Graph Core Edge

amount                         : double
log_amount                     : float
type                           : dictionary<values=string, indices=int8, ordered=0>
origin_balance_change          : float
destination_balance_change     : float
origin_zero_before             : int8
origin_zero_after              : int8
destination_zero_before        : int8
destination_zero_after         : int8


In [13]:
from pathlib import Path
import json

gnn_dir = Path("../artifacts/gnn")

# Lister les artefacts
print("Artefacts GNN")
print()

for path in sorted(gnn_dir.iterdir()):
    size_mb = path.stat().st_size / 1024**2
    print(f"{path.name:<60} {size_mb:>8.2f} MB")

Artefacts GNN

gnn_baseline_summary.json                                        0.00 MB
graphsage_core_edge_1layer_checkpoint.pt                         0.07 MB
graphsage_core_edge_1layer_config.json                           0.00 MB
graphsage_core_edge_1layer_test_metrics.json                     0.00 MB
graphsage_core_edge_1layer_test_predictions.npz                  2.65 MB
graphsage_core_edge_1layer_train_final.pt                        0.01 MB
graphsage_core_edge_1layer_train_history.json                    0.09 MB
graphsage_core_edge_1layer_validation_history.json               0.01 MB
graphsage_core_edge_1layer_validation_predictions.npz            2.51 MB
graphsage_core_edge_2layer_train_final.pt                        0.02 MB
graphsage_core_edge_2layer_validation_predictions.npz            2.17 MB
graphsage_core_edge_final_analysis.json                          0.00 MB
graphsage_core_edge_model_selection.json                         0.00 MB


In [14]:
# Mots-clés liés au preprocessing
keywords = [
    "scaler",
    "scale",
    "mean",
    "std",
    "normal",
    "preprocess",
    "feature",
    "type",
    "categor",
]

print("Clés JSON potentiellement liées au preprocessing")
print()

for path in sorted(gnn_dir.glob("*.json")):
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        matches = []

        def search_keys(obj, prefix=""):
            if isinstance(obj, dict):
                for key, value in obj.items():
                    current = f"{prefix}.{key}" if prefix else key

                    if any(word in key.lower() for word in keywords):
                        matches.append((current, value))

                    search_keys(value, current)

            elif isinstance(obj, list):
                for i, value in enumerate(obj):
                    search_keys(value, f"{prefix}[{i}]")

        search_keys(data)

        if matches:
            print(f"\n{path.name}")

            for key, value in matches:
                text = repr(value)

                if len(text) > 300:
                    text = text[:300] + "..."

                print(f"  {key} = {text}")

    except Exception as e:
        print(f"\n{path.name} : lecture impossible ({e})")

Clés JSON potentiellement liées au preprocessing


gnn_baseline_summary.json
  model.feature_set = 'Graph Core Edge'

graphsage_core_edge_1layer_config.json
  feature_set = 'graph_core_edge'

graphsage_core_edge_1layer_train_history.json
  [0].mean_loss = 5.462092209425776
  [0].mean_local_nodes = 4341.0
  [0].mean_context_edges = 0.0
  [1].mean_loss = 6.207937989244329
  [1].mean_local_nodes = 2674.0
  [1].mean_context_edges = 906.0
  [2].mean_loss = 6.153413634369339
  [2].mean_local_nodes = 1974.0
  [2].mean_context_edges = 972.0
  [3].mean_loss = 10.097114387444691
  [3].mean_local_nodes = 2087.0
  [3].mean_context_edges = 1074.0
  [4].mean_loss = 7.208817698543233
  [4].mean_local_nodes = 2448.0
  [4].mean_context_edges = 1217.0
  [5].mean_loss = 9.797303863893072
  [5].mean_local_nodes = 4461.0
  [5].mean_context_edges = 1463.0
  [6].mean_loss = 2.2383207438615256
  [6].mean_local_nodes = 7879.5
  [6].mean_context_edges = 1755.0
  [7].mean_loss = 1.395996652113045
  [7].mean_loca

In [15]:
import torch
from pathlib import Path

checkpoint_path = Path(
    "../artifacts/gnn/graphsage_core_edge_1layer_checkpoint.pt"
)

# Charger le checkpoint
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

print("Type checkpoint :", type(checkpoint))

if isinstance(checkpoint, dict):
    print("\nClés principales :")
    for key in checkpoint.keys():
        value = checkpoint[key]

        if isinstance(value, torch.Tensor):
            print(
                f"{key:<30} | "
                f"Tensor {tuple(value.shape)} {value.dtype}"
            )
        elif isinstance(value, dict):
            print(
                f"{key:<30} | "
                f"dict ({len(value)} clés)"
            )
        else:
            text = repr(value)
            if len(text) > 200:
                text = text[:200] + "..."
            print(f"{key:<30} | {type(value).__name__} | {text}")

Type checkpoint : <class 'dict'>

Clés principales :
completed_step                 | int | 334
integrated_end                 | int | 4771387
model_state_dict               | dict (7 clés)
optimizer_state_dict           | dict (2 clés)
config                         | dict (17 clés)
history                        | list | [{'step': 1, 'transactions': 2708, 'frauds': 16, 'mean_loss': 5.462092209425776, 'gradient_norm': 7.671388982656311, 'runtime_seconds': 0.29941730000427924, 'num_batches': 1, 'mean_local_nodes': 4341....


In [16]:
keywords = [
    "scaler",
    "scale",
    "mean",
    "std",
    "normal",
    "preprocess",
    "feature",
    "category",
    "categories",
    "type",
]

matches = []

def inspect_checkpoint(obj, prefix=""):
    if isinstance(obj, dict):
        for key, value in obj.items():
            current = f"{prefix}.{key}" if prefix else str(key)

            if any(word in str(key).lower() for word in keywords):
                if isinstance(value, torch.Tensor):
                    desc = (
                        f"Tensor shape={tuple(value.shape)}, "
                        f"dtype={value.dtype}"
                    )
                else:
                    desc = repr(value)
                    if len(desc) > 300:
                        desc = desc[:300] + "..."

                matches.append((current, desc))

            inspect_checkpoint(value, current)

print("Éléments liés au preprocessing :")
print()

inspect_checkpoint(checkpoint)

if matches:
    for key, value in matches:
        print(f"{key} = {value}")
else:
    print("Aucun élément trouvé.")

Éléments liés au preprocessing :

config.feature_set = 'graph_core_edge'


### 4.11 Preprocessing des Graph Core Edge

L'inspection des artefacts du notebook 06 confirme que le modèle GraphSAGE utilisait bien le feature set `graph_core_edge` avec une dimension finale de **13**, mais que les paramètres exacts de normalisation n'ont pas été persistés.

Le checkpoint contient :

- les poids du modèle ;
- l'état de l'optimiseur ;
- la configuration d'entraînement ;
- l'historique d'apprentissage ;

mais aucune moyenne, aucun écart-type et aucun objet de preprocessing réutilisable.

Afin d'éviter toute reconstruction arbitraire, le notebook 07 définit donc explicitement son preprocessing et le sauvegarde comme un nouvel artefact reproductible.

Les huit variables non catégorielles sont :

- `amount`
- `log_amount`
- `origin_balance_change`
- `destination_balance_change`
- `origin_zero_before`
- `origin_zero_after`
- `destination_zero_before`
- `destination_zero_after`

Leurs statistiques de normalisation sont ajustées **uniquement sur les steps Train 1 à 334**.

La variable `type` utilise un one-hot encoding avec l'ordre fixe :

1. `CASH_IN`
2. `CASH_OUT`
3. `DEBIT`
4. `PAYMENT`
5. `TRANSFER`

La représentation finale conserve ainsi une dimension de **13**.

Les paramètres calculés sur le Train seront sauvegardés et réutilisés sans modification pour :

- le reste du Train ;
- la Validation ;
- le Test.

Aucun ajustement de preprocessing ne sera effectué à partir de la Validation ou du Test.

Les statistiques sont calculées en streaming par row group afin de ne jamais charger les 4.77 millions de transactions Train simultanément en mémoire.

In [17]:
import json
import numpy as np
import pyarrow.parquet as pq
import psutil

from pathlib import Path

# Définir les chemins
graph_ready_path = Path("../data/processed/paysim_graph_ready.parquet")
artifact_dir = Path("../artifacts/dynamic_gnn")
artifact_dir.mkdir(parents=True, exist_ok=True)

preprocessing_path = artifact_dir / "dynamic_gnn_preprocessing.json"

# Features numériques
numeric_cols = [
    "amount",
    "log_amount",
    "origin_balance_change",
    "destination_balance_change",
    "origin_zero_before",
    "origin_zero_after",
    "destination_zero_before",
    "destination_zero_after",
]

# Ordre fixe des types
type_categories = [
    "CASH_IN",
    "CASH_OUT",
    "DEBIT",
    "PAYMENT",
    "TRANSFER",
]

# Ordre final des 13 features
output_feature_order = [
    "amount",
    "log_amount",
    "type_CASH_IN",
    "type_CASH_OUT",
    "type_DEBIT",
    "type_PAYMENT",
    "type_TRANSFER",
    "origin_balance_change",
    "destination_balance_change",
    "origin_zero_before",
    "origin_zero_after",
    "destination_zero_before",
    "destination_zero_after",
]

# Ouvrir le Parquet
pf = pq.ParquetFile(graph_ready_path)

# Statistiques streaming
count = 0
mean = np.zeros(len(numeric_cols), dtype=np.float64)
m2 = np.zeros(len(numeric_cols), dtype=np.float64)

for rg in range(pf.metadata.num_row_groups):

    # Lire uniquement les colonnes nécessaires
    table = pf.read_row_group(
        rg,
        columns=["step"] + numeric_cols,
    )

    steps_rg = table["step"].to_numpy(zero_copy_only=False)

    # Garder uniquement le Train
    mask = steps_rg <= 334

    if not mask.any():
        if steps_rg.min() > 334:
            break
        continue

    # Construire le petit bloc numérique
    x = np.column_stack([
        table[col].to_numpy(zero_copy_only=False)[mask]
        for col in numeric_cols
    ]).astype(np.float64, copy=False)

    n = x.shape[0]

    # Statistiques du bloc
    block_mean = x.mean(axis=0)
    block_m2 = ((x - block_mean) ** 2).sum(axis=0)

    # Fusionner les statistiques
    if count == 0:
        mean = block_mean
        m2 = block_m2
        count = n
    else:
        delta = block_mean - mean
        total = count + n

        mean = mean + delta * n / total
        m2 = (
            m2
            + block_m2
            + delta**2 * count * n / total
        )

        count = total

    del table, steps_rg, mask, x

# Vérifier le nombre de lignes Train
expected_train_rows = 4_771_387

assert count == expected_train_rows, (
    f"Train attendu={expected_train_rows:,}, obtenu={count:,}"
)

# Écart-type population
std = np.sqrt(m2 / count)

# Éviter une division par zéro
scale = np.where(std > 0, std, 1.0)

print("Transactions utilisées :", f"{count:,}")
print()

print("Statistiques Train :")
for col, mu, sigma in zip(numeric_cols, mean, scale):
    print(
        f"{col:<30} "
        f"mean={mu:>14.6f} | "
        f"std={sigma:>14.6f}"
    )

print()
print("Ordre des types :", type_categories)
print("Dimension finale :", len(output_feature_order))
print("Ordre final :", output_feature_order)

Transactions utilisées : 4,771,387

Statistiques Train :
amount                         mean= 179748.531939 | std= 648694.425604
log_amount                     mean=     10.833058 | std=      1.820169
origin_balance_change          mean= -21731.548861 | std= 128793.638313
destination_balance_change     mean= 130154.927112 | std= 900249.394962
origin_zero_before             mean=      0.332624 | std=      0.471153
origin_zero_after              mean=      0.566853 | std=      0.495510
destination_zero_before        mean=      0.422584 | std=      0.493970
destination_zero_after         mean=      0.382497 | std=      0.485997

Ordre des types : ['CASH_IN', 'CASH_OUT', 'DEBIT', 'PAYMENT', 'TRANSFER']
Dimension finale : 13
Ordre final : ['amount', 'log_amount', 'type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER', 'origin_balance_change', 'destination_balance_change', 'origin_zero_before', 'origin_zero_after', 'destination_zero_before', 'destination_zero_after']


In [18]:
# Préparer les métadonnées
preprocessing = {
    "feature_set": "graph_core_edge",
    "train_start_step": 1,
    "train_end_step": 334,
    "train_transactions": count,
    "numeric_features": numeric_cols,
    "numeric_mean": {
        col: float(value)
        for col, value in zip(numeric_cols, mean)
    },
    "numeric_std": {
        col: float(value)
        for col, value in zip(numeric_cols, scale)
    },
    "type_categories": type_categories,
    "type_encoding": "one_hot",
    "output_feature_order": output_feature_order,
    "output_dim": len(output_feature_order),
    "output_dtype": "float32",
    "fit_split": "train_only",
}

# Sauvegarder
with open(preprocessing_path, "w", encoding="utf-8") as f:
    json.dump(
        preprocessing,
        f,
        indent=2,
        ensure_ascii=False,
    )

print("Artefact sauvegardé :", preprocessing_path.exists())
print("Chemin :", preprocessing_path)
print(
    "Taille :",
    round(preprocessing_path.stat().st_size / 1024, 2),
    "KB"
)

print(
    "RAM disponible :",
    round(psutil.virtual_memory().available / 1024**3, 2),
    "GB"
)

Artefact sauvegardé : True
Chemin : ..\artifacts\dynamic_gnn\dynamic_gnn_preprocessing.json
Taille : 1.7 KB
RAM disponible : 3.92 GB


### 4.12 Validation du preprocessing

Le preprocessing des Graph Core Edge a été ajusté exclusivement sur les **4 771 387 transactions du Train**, correspondant aux steps 1 à 334.

Les huit variables non catégorielles possèdent toutes un écart-type strictement positif. Aucune correction artificielle n'a donc été nécessaire pour éviter une division par zéro.

La variable `type` utilise les cinq modalités suivantes dans un ordre fixe :

- `CASH_IN`
- `CASH_OUT`
- `DEBIT`
- `PAYMENT`
- `TRANSFER`

La représentation finale contient bien **13 dimensions**.

Les paramètres de preprocessing ont été sauvegardés dans :

`../artifacts/dynamic_gnn/dynamic_gnn_preprocessing.json`

Cet artefact devient la référence unique du notebook 07. Il sera réutilisé sans nouvel ajustement pour la Validation et le Test.

Le preprocessing exact du GraphSAGE du notebook 06 n'ayant pas été sauvegardé, il n'est pas possible de garantir une transformation numérique strictement identique entre les deux modèles.

La comparaison reste néanmoins scientifiquement cohérente car :

- le même feature set **Graph Core Edge** est utilisé ;
- les mêmes neuf variables transactionnelles sources sont utilisées ;
- la représentation finale possède la même dimension de 13 ;
- toutes les statistiques du modèle dynamique sont ajustées uniquement sur le Train ;
- aucune information de Validation ou de Test n'intervient dans le preprocessing.

Cette différence de preprocessing devra être mentionnée comme une limitation mineure lors de la comparaison finale.

In [19]:
import json
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

from pathlib import Path

# Définir les chemins
graph_ready_path = Path("../data/processed/paysim_graph_ready.parquet")
preprocessing_path = Path(
    "../artifacts/dynamic_gnn/dynamic_gnn_preprocessing.json"
)

# Charger le preprocessing
with open(preprocessing_path, "r", encoding="utf-8") as f:
    preprocessing = json.load(f)

numeric_cols = preprocessing["numeric_features"]
type_categories = preprocessing["type_categories"]
output_feature_order = preprocessing["output_feature_order"]

numeric_mean = np.array(
    [preprocessing["numeric_mean"][col] for col in numeric_cols],
    dtype=np.float64,
)

numeric_std = np.array(
    [preprocessing["numeric_std"][col] for col in numeric_cols],
    dtype=np.float64,
)

# Charger les événements en mmap
src = np.load("../data/graph/paysim_src.npy", mmap_mode="r")
dst = np.load("../data/graph/paysim_dst.npy", mmap_mode="r")
step = np.load("../data/graph/paysim_step.npy", mmap_mode="r")
label = np.load("../data/graph/paysim_label.npy", mmap_mode="r")
offsets = np.load("../data/graph/paysim_step_offsets.npy", mmap_mode="r")


class StepEventReader:
    def __init__(
        self,
        parquet_path,
        src,
        dst,
        step,
        label,
        offsets,
        numeric_cols,
        numeric_mean,
        numeric_std,
        type_categories,
    ):
        self.pf = pq.ParquetFile(parquet_path)

        self.src = src
        self.dst = dst
        self.step = step
        self.label = label
        self.offsets = offsets

        self.numeric_cols = numeric_cols
        self.numeric_mean = numeric_mean
        self.numeric_std = numeric_std
        self.type_categories = type_categories

        self.columns = numeric_cols + ["type"]

        # Bornes des row groups
        rg_sizes = [
            self.pf.metadata.row_group(i).num_rows
            for i in range(self.pf.metadata.num_row_groups)
        ]

        self.rg_bounds = np.concatenate([
            np.array([0], dtype=np.int64),
            np.cumsum(rg_sizes, dtype=np.int64),
        ])

        # Cache d'un row group
        self.cached_rg = None
        self.cached_table = None

    def _get_row_group(self, rg):
        # Réutiliser le row group courant
        if self.cached_rg != rg:
            self.cached_table = self.pf.read_row_group(
                rg,
                columns=self.columns,
            )
            self.cached_rg = rg

        return self.cached_table

    def _transform_table(self, table):
        # Construire les variables numériques
        numeric = np.column_stack([
            table[col].to_numpy(zero_copy_only=False)
            for col in self.numeric_cols
        ]).astype(np.float64, copy=False)

        numeric = (
            (numeric - self.numeric_mean)
            / self.numeric_std
        ).astype(np.float32)

        # Encoder le type
        types = (
            table["type"]
            .cast(pa.string())
            .to_numpy(zero_copy_only=False)
        )

        one_hot = np.zeros(
            (len(types), len(self.type_categories)),
            dtype=np.float32,
        )

        for j, category in enumerate(self.type_categories):
            one_hot[:, j] = types == category

        # Respecter l'ordre final des 13 features
        x = np.column_stack([
            numeric[:, 0],
            numeric[:, 1],
            one_hot,
            numeric[:, 2],
            numeric[:, 3],
            numeric[:, 4],
            numeric[:, 5],
            numeric[:, 6],
            numeric[:, 7],
        ]).astype(np.float32, copy=False)

        return x

    def _read_feature_slice(self, start, end):
        # Identifier les row groups concernés
        first_rg = (
            np.searchsorted(
                self.rg_bounds,
                start,
                side="right",
            )
            - 1
        )

        last_rg = (
            np.searchsorted(
                self.rg_bounds,
                end - 1,
                side="right",
            )
            - 1
        )

        blocks = []

        for rg in range(first_rg, last_rg + 1):
            rg_start = int(self.rg_bounds[rg])
            rg_end = int(self.rg_bounds[rg + 1])

            local_start = max(start, rg_start) - rg_start
            local_end = min(end, rg_end) - rg_start

            table = self._get_row_group(rg)

            sliced = table.slice(
                local_start,
                local_end - local_start,
            )

            blocks.append(
                self._transform_table(sliced)
            )

        if len(blocks) == 1:
            return blocks[0]

        return np.concatenate(blocks, axis=0)

    def read_step(self, t):
        # Vérifier le step
        if t < 1 or t >= len(self.offsets):
            raise ValueError(f"Step invalide : {t}")

        start = int(self.offsets[t - 1])
        end = int(self.offsets[t])

        x = self._read_feature_slice(start, end)

        return {
            "t": t,
            "start": start,
            "end": end,
            "src": np.asarray(self.src[start:end]),
            "dst": np.asarray(self.dst[start:end]),
            "step": np.asarray(self.step[start:end]),
            "label": np.asarray(self.label[start:end]),
            "x": x,
        }


# Initialiser le reader
reader = StepEventReader(
    parquet_path=graph_ready_path,
    src=src,
    dst=dst,
    step=step,
    label=label,
    offsets=offsets,
    numeric_cols=numeric_cols,
    numeric_mean=numeric_mean,
    numeric_std=numeric_std,
    type_categories=type_categories,
)

print("Reader initialisé : True")
print("Dimension attendue :", preprocessing["output_dim"])
print("Row groups :", reader.pf.metadata.num_row_groups)

Reader initialisé : True
Dimension attendue : 13
Row groups : 26


In [20]:
import psutil

# Steps de contrôle
check_steps = [1, 212, 334, 335]

for t in check_steps:
    batch = reader.read_step(t)

    x = batch["x"]

    # Vérifications
    correct_step = bool(np.all(batch["step"] == t))
    correct_rows = len(batch["src"]) == (
        batch["end"] - batch["start"]
    )

    correct_dim = (
        x.ndim == 2
        and x.shape[1] == 13
    )

    correct_dtype = x.dtype == np.float32
    finite = bool(np.isfinite(x).all())

    # Les colonnes 2 à 6 sont le one-hot
    one_hot_sum = x[:, 2:7].sum(axis=1)

    valid_one_hot = bool(
        np.all(one_hot_sum == 1.0)
    )

    aligned = (
        len(batch["src"])
        == len(batch["dst"])
        == len(batch["label"])
        == len(x)
    )

    print(f"Step {t}")
    print(f"  transactions      : {len(x):,}")
    print(f"  plage             : [{batch['start']:,}, {batch['end']:,})")
    print(f"  shape X           : {x.shape}")
    print(f"  dtype X           : {x.dtype}")
    print(f"  step correct      : {correct_step}")
    print(f"  lignes correctes  : {correct_rows}")
    print(f"  dimension correcte: {correct_dim}")
    print(f"  finite            : {finite}")
    print(f"  one-hot valide    : {valid_one_hot}")
    print(f"  arrays alignés    : {aligned}")
    print()

print(
    "RAM disponible :",
    round(
        psutil.virtual_memory().available / 1024**3,
        2,
    ),
    "GB"
)

Step 1
  transactions      : 2,708
  plage             : [0, 2,708)
  shape X           : (2708, 13)
  dtype X           : float32
  step correct      : True
  lignes correctes  : True
  dimension correcte: True
  finite            : True
  one-hot valide    : True
  arrays alignés    : True

Step 212
  transactions      : 34,047
  plage             : [2,726,063, 2,760,110)
  shape X           : (34047, 13)
  dtype X           : float32
  step correct      : True
  lignes correctes  : True
  dimension correcte: True
  finite            : True
  one-hot valide    : True
  arrays alignés    : True

Step 334
  transactions      : 12,016
  plage             : [4,759,371, 4,771,387)
  shape X           : (12016, 13)
  dtype X           : float32
  step correct      : True
  lignes correctes  : True
  dimension correcte: True
  finite            : True
  one-hot valide    : True
  arrays alignés    : True

Step 335
  transactions      : 10,456
  plage             : [4,771,387, 4,781,843)
  s

### 4.13 Validation du reader événementiel

Le reader événementiel a été testé sur plusieurs positions représentatives du flux temporel :

- le premier step du Train ;
- le step 212, contenant un volume important de transactions ;
- le dernier step du Train ;
- le premier step de Validation.

Pour chacun de ces steps :

- le nombre de transactions correspond exactement aux offsets temporels ;
- toutes les lignes appartiennent au step demandé ;
- les arrays `src`, `dst` et `label` sont alignés avec les features ;
- la matrice de features possède exactement **13 dimensions** ;
- les features sont représentées en `float32` ;
- aucune valeur non finie n'est présente ;
- chaque transaction possède exactement une modalité active dans le one-hot de `type`.

La frontière entre Train et Validation est également respectée :

\[
end_{334} = start_{335} = 4\,771\,387
\]

Le reader peut donc parcourir le flux PaySim step par step sans charger l'ensemble du dataset en mémoire et sans modifier l'ordre temporel validé précédemment.

La RAM disponible reste proche de **3.89 GB** après les contrôles, ce qui confirme que cette stratégie de lecture est compatible avec les contraintes de la machine locale.

In [21]:
# Vérifier l'alignement Parquet / arrays
alignment_steps = [1, 212, 334, 335]

all_checks = True

for t in alignment_steps:
    batch = reader.read_step(t)

    start = batch["start"]
    end = batch["end"]

    # Lire les mêmes lignes directement dans le Parquet
    first_rg = (
        np.searchsorted(reader.rg_bounds, start, side="right") - 1
    )

    last_rg = (
        np.searchsorted(reader.rg_bounds, end - 1, side="right") - 1
    )

    parquet_blocks = []

    for rg in range(first_rg, last_rg + 1):
        rg_start = int(reader.rg_bounds[rg])
        rg_end = int(reader.rg_bounds[rg + 1])

        local_start = max(start, rg_start) - rg_start
        local_end = min(end, rg_end) - rg_start

        table = reader.pf.read_row_group(
            rg,
            columns=["step", "isFraud", "type"],
        )

        parquet_blocks.append(
            table.slice(
                local_start,
                local_end - local_start,
            )
        )

    table_step = pa.concat_tables(parquet_blocks)

    parquet_step = table_step["step"].to_numpy(
        zero_copy_only=False
    )

    parquet_label = table_step["isFraud"].to_numpy(
        zero_copy_only=False
    )

    parquet_type = (
        table_step["type"]
        .cast(pa.string())
        .to_numpy(zero_copy_only=False)
    )

    # Reconstituer le type depuis le one-hot
    type_idx = batch["x"][:, 2:7].argmax(axis=1)

    decoded_type = np.array(
        [type_categories[i] for i in type_idx]
    )

    step_match = np.array_equal(
        parquet_step,
        batch["step"],
    )

    label_match = np.array_equal(
        parquet_label,
        batch["label"],
    )

    type_match = np.array_equal(
        parquet_type,
        decoded_type,
    )

    length_match = (
        len(parquet_step)
        == len(batch["src"])
        == len(batch["x"])
    )

    step_ok = (
        step_match
        and label_match
        and type_match
        and length_match
    )

    all_checks &= step_ok

    print(f"Step {t}")
    print(f"  step aligné   : {step_match}")
    print(f"  label aligné  : {label_match}")
    print(f"  type aligné   : {type_match}")
    print(f"  longueur      : {length_match}")
    print(f"  validation    : {step_ok}")
    print()

print("Alignement global :", all_checks)

Step 1
  step aligné   : True
  label aligné  : True
  type aligné   : True
  longueur      : True
  validation    : True

Step 212
  step aligné   : True
  label aligné  : True
  type aligné   : True
  longueur      : True
  validation    : True

Step 334
  step aligné   : True
  label aligné  : True
  type aligné   : True
  longueur      : True
  validation    : True

Step 335
  step aligné   : True
  label aligné  : True
  type aligné   : True
  longueur      : True
  validation    : True

Alignement global : True


# 5. Conception de la mémoire temporelle parcimonieuse

## 5.1 Objectif

Le modèle dynamique nécessite un état temporel persistant, mais cet état ne doit pas être alloué indistinctement aux plus de neuf millions de comptes PaySim.

Les analyses précédentes montrent que l'information temporelle réutilisable est principalement concentrée autour des destinations.

La mémoire adoptera donc une stratégie **destination-centric**.

Un nœud obtient un état persistant lorsqu'il a été observé comme destination dans un step antérieur.

Cette règle permet également à une source actuelle d'utiliser une mémoire si ce même compte a déjà été observé comme destination dans le passé.

En revanche, un compte apparaissant uniquement comme source ne reçoit pas automatiquement de mémoire persistante.

## 5.2 Séparation entre `node_id` et `memory_slot`

Le `node_id` global reste l'identifiant du compte dans le graphe.

Il ne constitue jamais une feature du modèle.

La mémoire utilise un second identifiant interne :

[\node\_id \rightarrow memory\_slot\]

Le `memory_slot` indique uniquement où se trouve l'état temporel du compte.

Pour un nœud sans état :

[\memory\_slot = -1\]

Cette correspondance n'est pas un embedding d'identité appris.

Elle constitue uniquement une structure d'indexation.

## 5.3 Indexation efficace

L'utilisation d'un dictionnaire Python contenant plusieurs millions d'entrées pourrait entraîner un coût mémoire important.

La correspondance `node_id → memory_slot` sera donc représentée par un tableau `int32` de taille égale au nombre total de nœuds.

Pour environ 9.07 millions de nœuds, ce tableau nécessite seulement environ 35 MiB.

Cette petite structure dense est acceptable car elle contient uniquement un index entier.

Les vecteurs temporels coûteux restent, eux, réellement parcimonieux et ne sont créés que pour les comptes possédant un historique utile.

## 5.4 Allocation causale

Pour le step (t), la mémoire disponible avant prédiction correspond exactement à :

[\Memory\_{\<t}\]

Lorsqu'une nouvelle destination apparaît dans (E\_t), aucun slot ne doit être créé avant la production des prédictions du step.

Toutes les transactions du step utilisant ce compte le considèrent donc comme New.

Ce n'est qu'après la prédiction de l'ensemble de (E\_t) que les nouvelles destinations peuvent être enregistrées dans la mémoire.

La séquence est :

[\Lookup(State\_{\<t})\]

[\\rightarrow Prediction(E\_t)\]

[\\rightarrow Allocation\ des\ nouvelles\ destinations\]

[\\rightarrow Update(State\_t)\]

Cette règle garantit qu'une répétition du même compte à l'intérieur d'un step ne crée jamais artificiellement un historique intra-step.

## 5.5 Gestion de `last_update`

Chaque slot persistant possède un `last_update`.

Pour un nœud connu au step (t) :

[\\Delta t = t - last\_update\]

avec nécessairement :

[\last\_update < t\]

Pour un nœud nouveau :

- `known = 0` ;
- aucun véritable (\Delta t) historique n'existe ;
- une valeur neutre peut être retournée par le stockage ;
- le masque `known` permet au modèle de distinguer cette valeur d'un véritable temps écoulé.

Les steps PaySim étant compris entre 1 et 743, `last_update` peut être stocké dans un entier compact.

## 5.6 État initial

Un nœud sans mémoire persistante utilisera un état initial partagé :

[\m\_0\]

Aucun état initial spécifique au `node_id` ne sera appris.

Le même principe s'applique donc à tous les comptes cold-start.

La valeur exacte de (m\_0), la dimension mémoire et le mécanisme GRU seront définis après validation du stockage et du protocole causal.

## 5.7 Cycle de vie d'un compte

Un compte peut suivre le cycle suivant :

**Avant toute apparition comme destination**

[\slot=-1,\ known=0\]

**Première apparition comme destination au step (t)**

Il reste New pendant toutes les prédictions de (E\_t).

Après le step :

[\slot \geq 0\]

[last\_update=t]

**Nouvelle interaction au step (t+k)**

Le compte est désormais Known et :

[\\Delta t=k\]

Cette règle est identique en Train, Validation et Test.

## 5.8 Gestion de l'autograd

La mémoire persistante et sa logique d'indexation ne seront pas assimilées à des paramètres appris du modèle.

La gestion des gradients à travers les mises à jour GRU sera traitée séparément dans le protocole d'entraînement temporel.

Cette séparation est nécessaire afin d'éviter de conserver involontairement un graphe d'autograd sur plusieurs centaines de steps.

La présente section valide donc uniquement :

- l'allocation ;
- l'indexation ;
- Known/New ;
- `last_update` ;
- la causalité du stockage.


In [22]:
# Paramètres validés
num_nodes = 9_073_900
train_unique_dest = 2_035_605

print("Estimation mémoire destination-centric")
print()

for dim in [8, 16, 32]:
    slot_lookup = num_nodes * 4
    node_of_slot = train_unique_dest * 4
    last_update = train_unique_dest * 2
    temporal_memory = train_unique_dest * dim * 4

    total = (
        slot_lookup
        + node_of_slot
        + last_update
        + temporal_memory
    )

    print(
        f"memory_dim={dim:<2} | "
        f"lookup={slot_lookup / 1024**2:.1f} MiB | "
        f"memory={temporal_memory / 1024**2:.1f} MiB | "
        f"total minimum={total / 1024**2:.1f} MiB"
    )

Estimation mémoire destination-centric

memory_dim=8  | lookup=34.6 MiB | memory=62.1 MiB | total minimum=108.4 MiB
memory_dim=16 | lookup=34.6 MiB | memory=124.2 MiB | total minimum=170.5 MiB
memory_dim=32 | lookup=34.6 MiB | memory=248.5 MiB | total minimum=294.7 MiB


In [23]:
import numpy as np


class SparseTemporalIndex:
    def __init__(
        self,
        num_nodes,
        initial_capacity=100_000,
        growth_size=100_000,
    ):
        self.num_nodes = num_nodes
        self.growth_size = growth_size

        # Mapping global vers slot
        self.slot_of_node = np.full(
            num_nodes,
            -1,
            dtype=np.int32,
        )

        # Informations des slots actifs
        self.node_of_slot = np.empty(
            initial_capacity,
            dtype=np.int32,
        )

        self.last_update = np.full(
            initial_capacity,
            -1,
            dtype=np.int16,
        )

        self.size = 0
        self.capacity = initial_capacity

    def _ensure_capacity(self, required):
        # Agrandir seulement si nécessaire
        if required <= self.capacity:
            return

        new_capacity = max(
            required,
            self.capacity + self.growth_size,
        )

        new_nodes = np.empty(
            new_capacity,
            dtype=np.int32,
        )

        new_last_update = np.full(
            new_capacity,
            -1,
            dtype=np.int16,
        )

        new_nodes[:self.size] = self.node_of_slot[:self.size]
        new_last_update[:self.size] = self.last_update[:self.size]

        self.node_of_slot = new_nodes
        self.last_update = new_last_update
        self.capacity = new_capacity

    def lookup(self, node_ids, t):
        # Récupérer les slots
        node_ids = np.asarray(
            node_ids,
            dtype=np.int32,
        )

        slots = self.slot_of_node[node_ids]
        known = slots >= 0

        # Calculer delta t
        delta_t = np.zeros(
            len(node_ids),
            dtype=np.int16,
        )

        if known.any():
            known_slots = slots[known]

            delta_t[known] = (
                t
                - self.last_update[known_slots]
            )

        return slots, known, delta_t

    def integrate_destinations(self, dst_ids, t):
        # Regrouper les destinations du step
        unique_dst = np.unique(
            np.asarray(dst_ids, dtype=np.int32)
        )

        current_slots = self.slot_of_node[unique_dst]

        # Identifier les nouveaux comptes
        new_dst = unique_dst[current_slots < 0]

        n_new = len(new_dst)

        if n_new > 0:
            required = self.size + n_new

            self._ensure_capacity(required)

            new_slots = np.arange(
                self.size,
                required,
                dtype=np.int32,
            )

            self.slot_of_node[new_dst] = new_slots
            self.node_of_slot[new_slots] = new_dst

            self.size = required

        # Mettre à jour le dernier step observé
        all_slots = self.slot_of_node[unique_dst]

        self.last_update[all_slots] = t

        return {
            "unique_destinations": len(unique_dst),
            "new_destinations": n_new,
            "active_slots": self.size,
        }

In [24]:
# Initialiser une mémoire vide
temporal_index = SparseTemporalIndex(
    num_nodes=9_073_900,
    initial_capacity=100_000,
)

# Lire le step 1
batch_1 = reader.read_step(1)

# Lookup avant toute intégration
_, src_known_1, src_dt_1 = temporal_index.lookup(
    batch_1["src"],
    t=1,
)

_, dst_known_1, dst_dt_1 = temporal_index.lookup(
    batch_1["dst"],
    t=1,
)

print("Step 1 avant intégration")
print("  sources connues      :", int(src_known_1.sum()))
print("  destinations connues :", int(dst_known_1.sum()))
print(
    "  tout est New         :",
    bool(
        not src_known_1.any()
        and not dst_known_1.any()
    ),
)

# Intégrer seulement après le step 1
integration_1 = temporal_index.integrate_destinations(
    batch_1["dst"],
    t=1,
)

print("\nAprès intégration du step 1")
print(
    "  destinations uniques :",
    f"{integration_1['unique_destinations']:,}",
)
print(
    "  nouvelles destinations:",
    f"{integration_1['new_destinations']:,}",
)
print(
    "  slots actifs         :",
    f"{integration_1['active_slots']:,}",
)

# Lire le step 2
batch_2 = reader.read_step(2)

_, src_known_2, src_dt_2 = temporal_index.lookup(
    batch_2["src"],
    t=2,
)

_, dst_known_2, dst_dt_2 = temporal_index.lookup(
    batch_2["dst"],
    t=2,
)

# Référence exacte
history_dst = np.unique(batch_1["dst"])

src_reference = np.isin(
    batch_2["src"],
    history_dst,
)

dst_reference = np.isin(
    batch_2["dst"],
    history_dst,
)

print("\nStep 2 avant intégration")
print(
    "  sources connues      :",
    f"{int(src_known_2.sum()):,}",
)
print(
    "  destinations connues :",
    f"{int(dst_known_2.sum()):,}",
)

print(
    "  sources conformes    :",
    bool(np.array_equal(src_known_2, src_reference)),
)

print(
    "  destinations conformes:",
    bool(np.array_equal(dst_known_2, dst_reference)),
)

# Vérifier delta t
known_dt = np.concatenate([
    src_dt_2[src_known_2],
    dst_dt_2[dst_known_2],
])

delta_correct = (
    len(known_dt) == 0
    or np.all(known_dt == 1)
)

print(
    "  delta_t causal       :",
    bool(delta_correct),
)

print(
    "\nRAM index lookup :",
    round(
        temporal_index.slot_of_node.nbytes / 1024**2,
        2,
    ),
    "MiB",
)

Step 1 avant intégration
  sources connues      : 0
  destinations connues : 0
  tout est New         : True

Après intégration du step 1
  destinations uniques : 1,633
  nouvelles destinations: 1,633
  slots actifs         : 1,633

Step 2 avant intégration
  sources connues      : 0
  destinations connues : 348
  sources conformes    : True
  destinations conformes: True
  delta_t causal       : True

RAM index lookup : 34.61 MiB


### 5.9 Validation de la mémoire temporelle parcimonieuse

La stratégie d'indexation destination-centric a été validée sur les premiers steps du flux transactionnel.

Le coût mémoire théorique de la représentation complète reste modéré :

| Dimension mémoire | Coût minimum estimé |
| ---: | ---: |
| 8 | 108.4 MiB |
| 16 | 170.5 MiB |
| 32 | 294.7 MiB |

Cette estimation comprend :

- le tableau global `node_id → memory_slot` ;
- la correspondance inverse `memory_slot → node_id` ;
- `last_update` ;
- les vecteurs de mémoire temporelle.

Le tableau global d'indexation représente seulement **34.61 MiB**, ce qui confirme qu'une petite structure dense d'indices est préférable à plusieurs millions d'entrées dans un dictionnaire Python.

Les vecteurs temporels eux-mêmes restent alloués uniquement aux destinations disposant effectivement d'un historique.

### Validation causale sur les premiers steps

Avant l'intégration du step 1 :

- sources connues : **0**
- destinations connues : **0**

L'ensemble du premier step est donc correctement traité en cold-start.

Après la prédiction conceptuelle du step 1, **1 633 destinations uniques** sont intégrées dans l'index temporel.

Lors de la lecture du step 2, avant toute nouvelle intégration :

- sources connues : **0**
- destinations connues : **348**

Une comparaison indépendante avec les destinations réellement présentes au step 1 confirme que :

- les indicateurs Known/New des sources sont exacts ;
- les indicateurs Known/New des destinations sont exacts ;
- les valeurs de `delta_t` sont strictement causales.

Pour tous les nœuds connus au step 2 :

\[
\Delta t = 2 - 1 = 1
\]

Aucune transaction du step 2 n'est donc utilisée pour déterminer l'état historique d'une autre transaction appartenant au même step.

### Conséquence

Le mécanisme suivant est désormais validé :

\[
State_{<t}
\rightarrow
Lookup(E_t)
\rightarrow
Prediction(E_t)
\rightarrow
Integration(E_t)
\rightarrow
State_{\leq t}
\]

La création des nouveaux slots intervient uniquement après les prédictions du step.

Un compte apparaissant plusieurs fois pour la première fois au même step reste ainsi `New` pour toutes les transactions de ce step.

Une source peut utiliser un état temporel uniquement si ce compte a déjà acquis un état persistant lors d'une apparition antérieure comme destination.

Cette stratégie respecte à la fois :

- le caractère fortement inductif de PaySim ;
- la causalité intra-step ;
- les contraintes mémoire de la machine ;
- l'interdiction des embeddings d'identité basés sur `node_id`.

La logique d'allocation, d'indexation, de Known/New et de `last_update` est donc validée indépendamment du modèle neuronal.



# 6. Protocole causal intra-step et mise à jour de la mémoire

## 6.1 Objectif

Le modèle temporel doit pouvoir traiter plusieurs transactions appartenant au même step sans créer d'ordre causal artificiel entre elles.

Cette contrainte est particulièrement importante lorsqu'une même destination reçoit plusieurs transactions dans un même step.

Une implémentation séquentielle naïve pourrait appliquer :

\[
Prediction(e_{t,1})
\rightarrow
Update
\rightarrow
Prediction(e_{t,2})
\]

Dans ce cas, la deuxième transaction utiliserait déjà une information provenant de la première alors que les deux événements possèdent le même timestamp PaySim.

Cette procédure est interdite.

## 6.2 Protocole retenu

Pour chaque step \(t\), l'état disponible est figé pendant toute la phase de prédiction :

\[
State_{<t}
\]

Toutes les transactions de \(E_t\) récupèrent leurs états source et destination à partir de cette même mémoire.

La séquence obligatoire est :

\[
State_{<t}
\rightarrow
Lookup(E_t)
\rightarrow
Prediction(E_t)
\rightarrow
Messages(E_t)
\rightarrow
Aggregation(E_t)
\rightarrow
Update
\rightarrow
State_{\leq t}
\]

Aucune modification de la mémoire ne peut intervenir entre deux prédictions appartenant au même step.

## 6.3 Construction des messages

Après la production des prédictions, chaque événement peut générer un message temporel à partir de :

- ses Graph Core Edge ;
- l'état source disponible avant le step ;
- l'état destination disponible avant le step ;
- les indicateurs Known/New ;
- les temps écoulés depuis les dernières mises à jour.

Le label de fraude est totalement exclu de cette opération.

Conceptuellement :

\[
msg_i =
f_{msg}
(
x_i,
m_{src_i}^{<t},
m_{dst_i}^{<t},
\Delta t_{src_i},
\Delta t_{dst_i},
known_{src_i},
known_{dst_i}
)
\]

Les messages servent uniquement à construire l'état utilisé pour les steps futurs.

## 6.4 Agrégation des événements simultanés

Une même destination peut recevoir plusieurs messages pendant un même step.

Ces messages doivent être agrégés avant toute mise à jour récurrente :

\[
\{msg_1,\ldots,msg_k\}
\rightarrow
Aggregate
\rightarrow
GRU
\]

L'agrégation doit être permutation-invariante.

Ainsi, inverser l'ordre physique des transactions appartenant au même step ne doit pas modifier le nouvel état obtenu.

Pour le test mécanique de cette section, une **moyenne des messages** sera utilisée.

Ce choix sert uniquement à valider le protocole causal et ne constitue pas encore une sélection définitive de l'agrégateur du modèle.

## 6.5 Mise à jour récurrente

Après agrégation, chaque destination concernée reçoit au maximum une mise à jour pour le step courant :

\[
m_v^{\leq t}
=
GRU(
Aggregate(messages_v^t),
m_v^{<t}
)
\]

Pour une destination nouvelle, l'ancien état correspond à l'état initial partagé.

Pour une destination connue, l'ancien état correspond à sa mémoire avant le step.

Le `last_update` n'est modifié qu'après cette opération.

## 6.6 Objectif du micro-test

Le micro-test doit vérifier sur de vraies transactions PaySim que :

1. plusieurs transactions peuvent viser la même destination dans un même step ;
2. elles utilisent toutes le même état destination antérieur ;
3. toutes les prédictions sont produites avant la mise à jour ;
4. l'agrégation ne dépend pas de l'ordre des transactions ;
5. la mise à jour GRU ne dépend pas de cet ordre ;
6. l'état n'est modifié qu'après l'agrégation.

Aucun entraînement et aucune évaluation de performance ne sont réalisés dans cette section.

In [25]:
# Steps représentatifs
collision_steps = [1, 212, 334, 335]

for t in collision_steps:
    batch = reader.read_step(t)

    _, counts = np.unique(
        batch["dst"],
        return_counts=True,
    )

    repeated = counts[counts > 1]

    print(f"Step {t}")
    print(
        "  transactions                :",
        f"{len(batch['dst']):,}",
    )
    print(
        "  destinations uniques        :",
        f"{len(counts):,}",
    )
    print(
        "  destinations répétées       :",
        f"{len(repeated):,}",
    )
    print(
        "  transactions vers répétées  :",
        f"{int(repeated.sum()):,}",
    )
    print(
        "  multiplicité maximale       :",
        int(counts.max()),
    )
    print()

Step 1
  transactions                : 2,708
  destinations uniques        : 1,633
  destinations répétées       : 181
  transactions vers répétées  : 1,256
  multiplicité maximale       : 32

Step 212
  transactions                : 34,047
  destinations uniques        : 33,137
  destinations répétées       : 878
  transactions vers répétées  : 1,788
  multiplicité maximale       : 4

Step 334
  transactions                : 12,016
  destinations uniques        : 11,977
  destinations répétées       : 39
  transactions vers répétées  : 78
  multiplicité maximale       : 2

Step 335
  transactions                : 10,456
  destinations uniques        : 10,431
  destinations répétées       : 25
  transactions vers répétées  : 50
  multiplicité maximale       : 2



In [26]:
import torch
import torch.nn as nn

# Reproductibilité du sanity check
torch.manual_seed(42)

# Dimension temporaire uniquement pour le test
protocol_memory_dim = 8
edge_dim = 13

# Etat initial partagé
initial_state = torch.zeros(
    protocol_memory_dim,
    dtype=torch.float32,
)

# Entrée :
# edge + src_memory + dst_memory + 2 delta_t + 2 known
protocol_input_dim = (
    edge_dim
    + 2 * protocol_memory_dim
    + 4
)

# Petit module de message
message_module = nn.Sequential(
    nn.Linear(protocol_input_dim, protocol_memory_dim),
    nn.Tanh(),
)

# Petit prédicteur
prediction_module = nn.Sequential(
    nn.Linear(protocol_input_dim, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
)

# Mise à jour temporelle
gru_update = nn.GRUCell(
    input_size=protocol_memory_dim,
    hidden_size=protocol_memory_dim,
)

message_module.eval()
prediction_module.eval()
gru_update.eval()

print("Dimension mémoire test :", protocol_memory_dim)
print("Dimension entrée :", protocol_input_dim)
print("Modules initialisés : True")

Dimension mémoire test : 8
Dimension entrée : 33
Modules initialisés : True


In [27]:
# Charger le step 1
batch = reader.read_step(1)

x = torch.from_numpy(batch["x"])

n = len(x)

# Etat historique avant le premier step
src_state = initial_state.repeat(n, 1)
dst_state = initial_state.repeat(n, 1)

src_known = torch.zeros(n, 1)
dst_known = torch.zeros(n, 1)

src_dt = torch.zeros(n, 1)
dst_dt = torch.zeros(n, 1)

# Construire l'entrée avant toute mise à jour
model_input = torch.cat(
    [
        x,
        src_state,
        dst_state,
        src_dt,
        dst_dt,
        src_known,
        dst_known,
    ],
    dim=1,
)

with torch.no_grad():
    # Toutes les prédictions sont produites d'abord
    logits_before_update = prediction_module(
        model_input
    ).squeeze(1)

    # Tous les messages sont également produits
    messages = message_module(model_input)

# Trouver une destination répétée
unique_dst, inverse, counts = np.unique(
    batch["dst"],
    return_inverse=True,
    return_counts=True,
)

repeated_mask = counts > 1

target_dst = unique_dst[
    np.flatnonzero(repeated_mask)[0]
]

target_event_mask = batch["dst"] == target_dst
target_positions = np.flatnonzero(target_event_mask)

print("Step testé :", 1)
print("Transactions :", f"{n:,}")
print("Prédictions produites :", f"{len(logits_before_update):,}")
print("Destination test :", int(target_dst))
print(
    "Occurrences dans le step :",
    len(target_positions),
)

# Vérifier que toutes utilisent le même état pré-step
target_states = dst_state[
    torch.from_numpy(target_positions)
]

same_pre_state = torch.allclose(
    target_states,
    target_states[0].repeat(
        len(target_positions),
        1,
    ),
)

print(
    "Même état avant prédiction :",
    bool(same_pre_state),
)

Step testé : 1
Transactions : 2,708
Prédictions produites : 2,708
Destination test : 1637600
Occurrences dans le step : 5
Même état avant prédiction : True


In [28]:
# Indices torch des destinations uniques
inverse_t = torch.from_numpy(
    inverse.astype(np.int64)
)

num_unique = len(unique_dst)

# Somme des messages
message_sum = torch.zeros(
    num_unique,
    protocol_memory_dim,
)

message_sum.index_add_(
    0,
    inverse_t,
    messages,
)

# Nombre de messages
message_count = torch.zeros(
    num_unique,
    1,
)

message_count.index_add_(
    0,
    inverse_t,
    torch.ones(n, 1),
)

# Moyenne par destination
aggregated = message_sum / message_count

# Refaire exactement le calcul en ordre inverse
reverse_idx = torch.arange(
    n - 1,
    -1,
    -1,
)

messages_rev = messages[reverse_idx]
inverse_rev = inverse_t[reverse_idx]

message_sum_rev = torch.zeros_like(
    message_sum
)

message_count_rev = torch.zeros_like(
    message_count
)

message_sum_rev.index_add_(
    0,
    inverse_rev,
    messages_rev,
)

message_count_rev.index_add_(
    0,
    inverse_rev,
    torch.ones(n, 1),
)

aggregated_rev = (
    message_sum_rev
    / message_count_rev
)

aggregation_invariant = torch.allclose(
    aggregated,
    aggregated_rev,
    atol=1e-6,
)

# Slot de la destination cible dans unique_dst
target_unique_idx = int(
    np.where(unique_dst == target_dst)[0][0]
)

old_state = initial_state.unsqueeze(0)

with torch.no_grad():
    new_state = gru_update(
        aggregated[
            target_unique_idx
        ].unsqueeze(0),
        old_state,
    )

    new_state_rev = gru_update(
        aggregated_rev[
            target_unique_idx
        ].unsqueeze(0),
        old_state,
    )

update_invariant = torch.allclose(
    new_state,
    new_state_rev,
    atol=1e-6,
)

state_changed = not torch.allclose(
    old_state,
    new_state,
)

print(
    "Agrégation indépendante de l'ordre :",
    bool(aggregation_invariant),
)

print(
    "Update GRU indépendante de l'ordre :",
    bool(update_invariant),
)

print(
    "Etat modifié seulement après agrégation :",
    bool(state_changed),
)

print(
    "Une mise à jour pour la destination :",
    True,
)

Agrégation indépendante de l'ordre : True
Update GRU indépendante de l'ordre : True
Etat modifié seulement après agrégation : True
Une mise à jour pour la destination : True


### 6.7 Validation du protocole causal intra-step

Le protocole de mise à jour différée a été testé sur plusieurs steps réels de PaySim.

L'audit confirme que plusieurs transactions peuvent concerner une même destination à l'intérieur d'un seul step.

Les résultats observés sont notamment :

| Step | Transactions | Destinations uniques | Destinations répétées | Transactions vers destinations répétées | Multiplicité maximale |
| ---: | -----------: | -------------------: | --------------------: | --------------------------------------: | --------------------: |
|    1 |        2 708 |                1 633 |                   181 |                                   1 256 |                    32 |
|  212 |       34 047 |               33 137 |                   878 |                                   1 788 |                     4 |
|  334 |       12 016 |               11 977 |                    39 |                                      78 |                     2 |
|  335 |       10 456 |               10 431 |                    25 |                                      50 |                     2 |

Ce résultat démontre que la gestion des événements simultanés est nécessaire.

Une mise à jour transaction par transaction pourrait en effet créer un historique artificiel à l'intérieur d'un même step, notamment lorsqu'une destination reçoit plusieurs interactions simultanées.

### Validation sur le step 1

Le micro-test a utilisé une mémoire temporaire de dimension 8 uniquement pour vérifier le protocole mécanique.

Les **2 708 transactions** du step 1 ont toutes été prédites avant toute opération de mise à jour.

Une destination présente cinq fois dans ce step a été utilisée comme cas de contrôle.

Les cinq événements ont récupéré exactement le même état destination antérieur.

Le contrôle :

`Même état avant prédiction : True`

confirme donc qu'aucune transaction du step ne modifie l'état utilisé par une autre transaction du même step.

### Agrégation des messages

Les messages associés à une même destination ont ensuite été regroupés à l'aide d'une moyenne.

Le calcul a été répété après inversion complète de l'ordre physique des transactions.

Le résultat obtenu est :

`Agrégation indépendante de l'ordre : True`

L'agrégation est donc permutation-invariante dans le test réalisé.

L'ordre arbitraire des lignes appartenant au même timestamp n'introduit pas d'information supplémentaire.

### Mise à jour récurrente

La sortie agrégée a ensuite été transmise à une `GRUCell`.

Les contrôles donnent :

- `Update GRU indépendante de l'ordre : True`
- `Etat modifié seulement après agrégation : True`
- `Une mise à jour pour la destination : True`

Une destination reçoit donc au maximum une mise à jour récurrente par step, même lorsqu'elle reçoit plusieurs transactions simultanées.

Le nouveau state dépend de l'ensemble agrégé des événements du step et non de leur ordre de stockage.

### Protocole définitivement retenu

Le traitement causal devient :

[State\_{\<t}]

[\downarrow]

[Lookup(E\_t)]

[\downarrow]

[Prediction(E\_t)]

[\downarrow]

[Messages(E\_t)]

[\downarrow]

[Aggregation\ par\ nœud]

[\downarrow]

[GRU\ Update]

[\downarrow]

[State\_{\leq t}]

Aucune opération modifiant la mémoire ne peut intervenir entre deux transactions appartenant au même step.

Ce protocole s'appliquera de manière identique au Train, à la Validation et au Test.

La différence entre les splits concernera uniquement l'optimisation des paramètres :

- Train : poids du modèle appris ;
- Validation : poids figés, état temporel mis à jour causalement ;
- Test : poids et décisions expérimentales entièrement figés, état temporel mis à jour causalement.

Le protocole intra-step est donc validé indépendamment des performances du futur modèle.



# 7. Micro-pilote temporel

## 7.1 Objectif

Les sections précédentes ont validé indépendamment :

- le flux événementiel causal ;
- les Graph Core Edge ;
- l'indexation Known/New ;
- le calcul de `delta_t` ;
- la mémoire parcimonieuse destination-centric ;
- l'agrégation intra-step ;
- la mise à jour différée de la mémoire.

L'objectif est maintenant d'assembler ces composants dans un premier modèle temporel entraînable.

Ce micro-pilote ne cherche pas à mesurer les performances finales.

Il doit uniquement vérifier que :

1. la loss peut être calculée sur de vraies transactions ;
2. les gradients atteignent le prédicteur ;
3. les gradients atteignent également le module de message et la GRU ;
4. les états temporels évoluent entre les steps ;
5. la causalité intra-step reste respectée ;
6. la mémoire reste compatible avec les ressources disponibles ;
7. le runtime est suffisamment raisonnable pour envisager un entraînement plus long.

## 7.2 Problème du gradient temporel

Une particularité importante apparaît avec une mémoire récurrente.

La prédiction du step (t) utilise :

[\
State\_{\<t}\
]

alors que la mise à jour produite à partir de (E\_t) sera utilisée pour les prédictions futures.

Par conséquent, si la mémoire était détachée du graphe d'autograd après chaque step, le module de message et la GRU recevraient très peu de signal d'apprentissage provenant des conséquences futures de leurs mises à jour.

Le micro-pilote utilisera donc une forme courte de **Truncated Backpropagation Through Time**.

Plusieurs steps consécutifs seront parcourus avant l'appel à `backward()`.

Les états calculés à l'intérieur de cette petite fenêtre restent différentiables.

À la fin de la fenêtre :

1. les losses sont agrégées ;
2. `backward()` est appliqué ;
3. les paramètres sont mis à jour ;
4. les états temporels sont détachés ;
5. la fenêtre suivante repart de cette mémoire détachée.

Cette stratégie évite de conserver un graphe d'autograd sur les centaines de steps du Train.

## 7.3 Architecture du micro-pilote

Pour ce test mécanique, une configuration volontairement compacte est utilisée :

- Graph Core Edge : 13 dimensions ;
- mémoire temporelle : 8 dimensions ;
- état initial partagé ;
- transformation temporelle : `log1p(delta_t)` normalisée ;
- MLP de prédiction léger ;
- MLP de message léger ;
- `GRUCell` pour la mise à jour temporelle ;
- agrégation moyenne des messages simultanés ;
- mémoire destination-centric.

La dimension mémoire de 8 est utilisée uniquement pour le micro-pilote.

Elle ne constitue pas encore un hyperparamètre sélectionné.

## 7.4 Fenêtre temporelle pilote

Le premier entraînement est limité aux steps 1 à 6 du Train.

Une fenêtre TBPTT de deux steps est utilisée :

- steps 1–2 ;
- steps 3–4 ;
- steps 5–6.

Cette taille très courte permet de vérifier que les gradients traversent au moins une transition temporelle tout en limitant fortement la consommation mémoire.

Aucune Validation et aucun Test ne sont utilisés dans ce micro-pilote.

## 7.5 Fonction de coût

Le dataset étant extrêmement déséquilibré, la loss utilise `BCEWithLogitsLoss` avec le `pos_weight` calculé exclusivement sur le Train :

[pos\_weight \approx 1273.07]

Cette pondération est déjà issue du split Train validé et ne dépend ni de la Validation ni du Test.

La loss du micro-pilote sert uniquement à vérifier le fonctionnement de l'optimisation.

Aucune conclusion de performance ne sera tirée de sa valeur.


In [29]:
# Auditer les steps du pilote
pilot_steps = list(range(1, 7))

total_tx = 0
total_fraud = 0

for t in pilot_steps:
    batch = reader.read_step(t)

    n_tx = len(batch["label"])
    n_fraud = int(batch["label"].sum())

    total_tx += n_tx
    total_fraud += n_fraud

    print(
        f"Step {t:<2} | "
        f"tx={n_tx:,} | "
        f"fraudes={n_fraud}"
    )

print()
print("Transactions pilote :", f"{total_tx:,}")
print("Fraudes pilote :", f"{total_fraud:,}")

Step 1  | tx=2,708 | fraudes=16
Step 2  | tx=1,014 | fraudes=8
Step 3  | tx=552 | fraudes=4
Step 4  | tx=565 | fraudes=10
Step 5  | tx=665 | fraudes=6
Step 6  | tx=1,660 | fraudes=22

Transactions pilote : 7,164
Fraudes pilote : 66


In [30]:
# Auditer les steps du pilote
pilot_steps = list(range(1, 7))

total_tx = 0
total_fraud = 0

for t in pilot_steps:
    batch = reader.read_step(t)

    n_tx = len(batch["label"])
    n_fraud = int(batch["label"].sum())

    total_tx += n_tx
    total_fraud += n_fraud

    print(
        f"Step {t:<2} | "
        f"tx={n_tx:,} | "
        f"fraudes={n_fraud}"
    )

print()
print("Transactions pilote :", f"{total_tx:,}")
print("Fraudes pilote :", f"{total_fraud:,}")

Step 1  | tx=2,708 | fraudes=16
Step 2  | tx=1,014 | fraudes=8
Step 3  | tx=552 | fraudes=4
Step 4  | tx=565 | fraudes=10
Step 5  | tx=665 | fraudes=6
Step 6  | tx=1,660 | fraudes=22

Transactions pilote : 7,164
Fraudes pilote : 66


In [31]:
import math
import torch
import torch.nn as nn

# Reproductibilité
torch.manual_seed(42)

# Dimensions du pilote
edge_dim = 13
memory_dim = 8
hidden_dim = 32

input_dim = edge_dim + 2 * memory_dim + 4

# Transformation temporelle
max_log_dt = math.log1p(743)


class PilotDynamicModel(nn.Module):
    def __init__(
        self,
        edge_dim=13,
        memory_dim=8,
        hidden_dim=32,
    ):
        super().__init__()

        self.memory_dim = memory_dim

        # Etat initial partagé
        self.initial_state = nn.Parameter(
            torch.zeros(memory_dim)
        )

        # Prédicteur
        self.predictor = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(hidden_dim, 1),
        )

        # Message temporel
        self.message_module = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, memory_dim),
            nn.Tanh(),
        )

        # Mise à jour mémoire
        self.gru = nn.GRUCell(
            input_size=memory_dim,
            hidden_size=memory_dim,
        )

    def build_input(
        self,
        x,
        src_state,
        dst_state,
        src_dt,
        dst_dt,
        src_known,
        dst_known,
    ):
        # Encoder delta t
        src_time = torch.log1p(src_dt) / max_log_dt
        dst_time = torch.log1p(dst_dt) / max_log_dt

        return torch.cat(
            [
                x,
                src_state,
                dst_state,
                src_time,
                dst_time,
                src_known,
                dst_known,
            ],
            dim=1,
        )


model = PilotDynamicModel(
    edge_dim=edge_dim,
    memory_dim=memory_dim,
    hidden_dim=hidden_dim,
)

num_params = sum(
    p.numel()
    for p in model.parameters()
)

print("Input dim :", input_dim)
print("Memory dim :", memory_dim)
print("Hidden dim :", hidden_dim)
print("Paramètres :", f"{num_params:,}")

Input dim : 33
Memory dim : 8
Hidden dim : 32
Paramètres : 2,913


In [32]:
class PilotTemporalMemory:
    def __init__(
        self,
        temporal_index,
        memory_dim,
        initial_capacity=100_000,
        growth_size=100_000,
    ):
        self.index = temporal_index
        self.memory_dim = memory_dim
        self.growth_size = growth_size

        self.capacity = initial_capacity

        # Mémoire persistante détachée
        self.memory = torch.zeros(
            initial_capacity,
            memory_dim,
            dtype=torch.float32,
        )

        # Etats différentiables de la fenêtre
        self.overlay = {}

    def ensure_capacity(self):
        # Suivre la capacité de l'index
        required = self.index.capacity

        if required <= self.capacity:
            return

        new_capacity = max(
            required,
            self.capacity + self.growth_size,
        )

        new_memory = torch.zeros(
            new_capacity,
            self.memory_dim,
            dtype=torch.float32,
        )

        new_memory[:self.capacity] = self.memory

        self.memory = new_memory
        self.capacity = new_capacity

    def get_states(
        self,
        slots,
        model,
    ):
        n = len(slots)

        # Etat partagé pour les nouveaux nœuds
        states = (
            model.initial_state
            .unsqueeze(0)
            .expand(n, -1)
            .clone()
        )

        known_pos = np.flatnonzero(slots >= 0)

        if len(known_pos) == 0:
            return states

        known_slots = slots[known_pos]

        # Etat persistant
        base_states = self.memory[
            torch.from_numpy(
                known_slots.astype(np.int64)
            )
        ]

        states[
            torch.from_numpy(
                known_pos.astype(np.int64)
            )
        ] = base_states

        # Remplacer par les états différentiables récents
        for pos, slot in zip(
            known_pos,
            known_slots,
        ):
            slot = int(slot)

            if slot in self.overlay:
                states[pos] = self.overlay[slot]

        return states

    def set_overlay(
        self,
        slots,
        states,
    ):
        for i, slot in enumerate(slots):
            self.overlay[int(slot)] = states[i]

    def commit(self):
        # Sauvegarder puis détacher la fenêtre
        with torch.no_grad():
            for slot, state in self.overlay.items():
                self.memory[slot] = state.detach()

        self.overlay.clear()

In [33]:
def process_pilot_step(
    model,
    temporal_memory,
    batch,
):
    t = batch["t"]

    src_np = batch["src"]
    dst_np = batch["dst"]

    # Lookup strictement avant le step
    src_slots, src_known, src_dt = (
        temporal_memory.index.lookup(
            src_np,
            t=t,
        )
    )

    dst_slots, dst_known, dst_dt = (
        temporal_memory.index.lookup(
            dst_np,
            t=t,
        )
    )

    # Etats avant le step
    src_state = temporal_memory.get_states(
        src_slots,
        model,
    )

    dst_state = temporal_memory.get_states(
        dst_slots,
        model,
    )

    # Construire les tenseurs
    x = torch.from_numpy(batch["x"])

    y = torch.from_numpy(
        batch["label"].astype(np.float32)
    )

    src_dt_t = torch.from_numpy(
        src_dt.astype(np.float32)
    ).unsqueeze(1)

    dst_dt_t = torch.from_numpy(
        dst_dt.astype(np.float32)
    ).unsqueeze(1)

    src_known_t = torch.from_numpy(
        src_known.astype(np.float32)
    ).unsqueeze(1)

    dst_known_t = torch.from_numpy(
        dst_known.astype(np.float32)
    ).unsqueeze(1)

    # Entrée causale
    model_input = model.build_input(
        x=x,
        src_state=src_state,
        dst_state=dst_state,
        src_dt=src_dt_t,
        dst_dt=dst_dt_t,
        src_known=src_known_t,
        dst_known=dst_known_t,
    )

    # Prédictions avant update
    logits = model.predictor(
        model_input
    ).squeeze(1)

    # Messages produits après la prédiction
    messages = model.message_module(
        model_input
    )

    # Destinations uniques du step
    unique_dst, first_idx, inverse = np.unique(
        dst_np,
        return_index=True,
        return_inverse=True,
    )

    inverse_t = torch.from_numpy(
        inverse.astype(np.int64)
    )

    num_unique = len(unique_dst)

    # Agrégation moyenne
    msg_sum = torch.zeros(
        num_unique,
        memory_dim,
    )

    msg_sum.index_add_(
        0,
        inverse_t,
        messages,
    )

    msg_count = torch.zeros(
        num_unique,
        1,
    )

    msg_count.index_add_(
        0,
        inverse_t,
        torch.ones(len(messages), 1),
    )

    aggregated = msg_sum / msg_count

    # Etat destination avant le step
    first_idx_t = torch.from_numpy(
        first_idx.astype(np.int64)
    )

    old_dst_state = dst_state[first_idx_t]

    # Une seule update par destination
    new_dst_state = model.gru(
        aggregated,
        old_dst_state,
    )

    # Allocation seulement après prédictions
    temporal_memory.index.integrate_destinations(
        dst_np,
        t=t,
    )

    temporal_memory.ensure_capacity()

    unique_slots = (
        temporal_memory.index
        .slot_of_node[unique_dst]
    )

    # Garder le graphe dans l'overlay
    temporal_memory.set_overlay(
        unique_slots,
        new_dst_state,
    )

    return logits, y, {
        "transactions": len(y),
        "frauds": int(y.sum().item()),
        "known_src": int(src_known.sum()),
        "known_dst": int(dst_known.sum()),
        "unique_dst": len(unique_dst),
        "active_slots": temporal_memory.index.size,
    }

In [34]:
import time
import psutil

# Nouvelle mémoire vide
pilot_index = SparseTemporalIndex(
    num_nodes=9_073_900,
    initial_capacity=100_000,
)

pilot_memory = PilotTemporalMemory(
    temporal_index=pilot_index,
    memory_dim=memory_dim,
    initial_capacity=100_000,
)

# Loss Train-only
pos_weight = torch.tensor(
    1273.0686248331108,
    dtype=torch.float32,
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight,
    reduction="sum",
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5,
)

tbptt_steps = 2

pilot_history = []

start_total = time.perf_counter()

for window_start in range(1, 7, tbptt_steps):
    optimizer.zero_grad()

    loss_sum = 0.0
    window_tx = 0
    window_fraud = 0

    window_start_time = time.perf_counter()

    for t in range(
        window_start,
        min(window_start + tbptt_steps, 7),
    ):
        batch = reader.read_step(t)

        logits, y, info = process_pilot_step(
            model=model,
            temporal_memory=pilot_memory,
            batch=batch,
        )

        step_loss = criterion(
            logits,
            y,
        )

        loss_sum = loss_sum + step_loss

        window_tx += info["transactions"]
        window_fraud += info["frauds"]

    # Moyenne par transaction
    loss = loss_sum / window_tx

    # Backpropagation temporelle tronquée
    loss.backward()

    # Normes de gradient
    predictor_grad = torch.sqrt(
        sum(
            (
                p.grad.detach().norm() ** 2
                if p.grad is not None
                else torch.tensor(0.0)
            )
            for p in model.predictor.parameters()
        )
    ).item()

    message_grad = torch.sqrt(
        sum(
            (
                p.grad.detach().norm() ** 2
                if p.grad is not None
                else torch.tensor(0.0)
            )
            for p in model.message_module.parameters()
        )
    ).item()

    gru_grad = torch.sqrt(
        sum(
            (
                p.grad.detach().norm() ** 2
                if p.grad is not None
                else torch.tensor(0.0)
            )
            for p in model.gru.parameters()
        )
    ).item()

    # Stabiliser le pilote
    total_grad = torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=5.0,
    )

    optimizer.step()

    # Détacher l'état temporel
    pilot_memory.commit()

    runtime = (
        time.perf_counter()
        - window_start_time
    )

    ram_available = (
        psutil.virtual_memory().available
        / 1024**3
    )

    result = {
        "steps": (
            window_start,
            min(
                window_start + tbptt_steps - 1,
                6,
            ),
        ),
        "transactions": window_tx,
        "frauds": window_fraud,
        "loss": float(loss.detach()),
        "predictor_grad": predictor_grad,
        "message_grad": message_grad,
        "gru_grad": gru_grad,
        "total_grad": float(total_grad),
        "active_slots": pilot_index.size,
        "runtime": runtime,
        "ram_available": ram_available,
    }

    pilot_history.append(result)

    print(
        f"Steps {result['steps'][0]}-{result['steps'][1]} | "
        f"tx={window_tx:,} | "
        f"fraudes={window_fraud} | "
        f"loss={result['loss']:.6f}"
    )

    print(
        f"  grad predictor={predictor_grad:.6f} | "
        f"message={message_grad:.6f} | "
        f"GRU={gru_grad:.6f} | "
        f"total={float(total_grad):.6f}"
    )

    print(
        f"  slots={pilot_index.size:,} | "
        f"runtime={runtime:.2f}s | "
        f"RAM libre={ram_available:.2f} GB"
    )

    print()

print(
    "Runtime total :",
    round(
        time.perf_counter() - start_total,
        2,
    ),
    "s",
)

Steps 1-2 | tx=3,722 | fraudes=24 | loss=6.853791
  grad predictor=9.582780 | message=0.016231 | GRU=0.024524 | total=9.591762
  slots=2,259 | runtime=0.67s | RAM libre=3.71 GB

Steps 3-4 | tx=1,117 | fraudes=14 | loss=15.405386
  grad predictor=107.090927 | message=0.046690 | GRU=0.044833 | total=107.093994
  slots=2,938 | runtime=0.05s | RAM libre=3.71 GB

Steps 5-6 | tx=2,325 | fraudes=28 | loss=11.744672
  grad predictor=10.555797 | message=0.012025 | GRU=0.017372 | total=10.593807
  slots=4,583 | runtime=0.10s | RAM libre=3.71 GB

Runtime total : 0.86 s


In [35]:
# Vérifications mécaniques
loss_finite = all(
    np.isfinite(item["loss"])
    for item in pilot_history
)

predictor_has_grad = any(
    item["predictor_grad"] > 0
    for item in pilot_history
)

message_has_grad = any(
    item["message_grad"] > 0
    for item in pilot_history
)

gru_has_grad = any(
    item["gru_grad"] > 0
    for item in pilot_history
)

slots_grow = all(
    pilot_history[i]["active_slots"]
    <= pilot_history[i + 1]["active_slots"]
    for i in range(len(pilot_history) - 1)
)

memory_finite = bool(
    torch.isfinite(
        pilot_memory.memory[
            :pilot_index.size
        ]
    ).all()
)

memory_nonzero = bool(
    (
        pilot_memory.memory[
            :pilot_index.size
        ].abs().sum()
        > 0
    ).item()
)

overlay_empty = (
    len(pilot_memory.overlay) == 0
)

print("Loss finie :", loss_finite)
print(
    "Gradient prédicteur :",
    predictor_has_grad,
)
print(
    "Gradient message module :",
    message_has_grad,
)
print(
    "Gradient GRU :",
    gru_has_grad,
)
print(
    "Slots monotones :",
    slots_grow,
)
print(
    "Mémoire finie :",
    memory_finite,
)
print(
    "Mémoire mise à jour :",
    memory_nonzero,
)
print(
    "Overlay vidé après TBPTT :",
    overlay_empty,
)
print(
    "Slots actifs finaux :",
    f"{pilot_index.size:,}",
)

Loss finie : True
Gradient prédicteur : True
Gradient message module : True
Gradient GRU : True
Slots monotones : True
Mémoire finie : True
Mémoire mise à jour : True
Overlay vidé après TBPTT : True
Slots actifs finaux : 4,583


### 7.6 Résultats du micro-pilote temporel

Le premier entraînement du modèle dynamique a été réalisé sur les steps 1 à 6 du Train.

Cette fenêtre représente :

- **7 164 transactions**
- **66 fraudes**

Le micro-pilote utilise une mémoire de dimension 8 et une fenêtre TBPTT de deux steps.

Ces valeurs servent uniquement à vérifier le fonctionnement mécanique de l'apprentissage temporel et ne constituent pas encore des hyperparamètres sélectionnés.

### Stabilité de la loss

Les trois fenêtres temporelles produisent des losses finies :

| Fenêtre | Transactions | Fraudes | Loss |
| ---: | ---: | ---: | ---: |
| 1-2 | 3 722 | 24 | 6.853791 |
| 3-4 | 1 117 | 14 | 15.405386 |
| 5-6 | 2 325 | 28 | 11.744672 |

Ces valeurs ne sont pas interprétées comme des indicateurs de performance.

La fenêtre est trop petite et la distribution locale des fraudes varie fortement.

L'objectif est uniquement de vérifier que le calcul forward/backward reste numérique et que l'optimisation peut être exécutée.

### Propagation des gradients

Les gradients observés sont :

| Fenêtre | Prédicteur | Module de message | GRU |
| ---: | ---: | ---: | ---: |
| 1-2 | 9.582780 | 0.016231 | 0.024524 |
| 3-4 | 107.090927 | 0.046690 | 0.044833 |
| 5-6 | 10.555797 | 0.012025 | 0.017372 |

Le résultat principal est que les gradients du module de message et de la GRU sont **strictement non nuls dans les trois fenêtres**.

La prédiction du step suivant dépend donc effectivement des états temporels produits par les événements précédents.

Le TBPTT permet ainsi au signal d'apprentissage de traverser les transitions temporelles sans conserver le graphe d'autograd sur l'ensemble de l'historique.

Les gradients du prédicteur peuvent présenter des amplitudes importantes, notamment en raison du fort `pos_weight` utilisé pour compenser le déséquilibre de classes.

Le gradient clipping est donc conservé comme mesure de stabilité pour les expériences suivantes.

### Evolution de la mémoire

Le nombre de slots actifs augmente causalement au cours du pilote :

- après les steps 1–2 : **2 259**
- après les steps 3–4 : **2 938**
- après les steps 5–6 : **4 583**

La mémoire persistante :

- ne contient aucune valeur non finie ;
- est effectivement modifiée par les événements ;
- reste détachée entre les fenêtres TBPTT.

L'overlay différentiable est correctement vidé après chaque optimisation.

Le mécanisme distingue donc bien :

1. la mémoire persistante détachée ;
2. les états temporaires différentiables nécessaires à la fenêtre TBPTT.

### Runtime et mémoire

Le micro-pilote complet s'exécute en environ **0.86 seconde** sur CPU.

La RAM disponible reste proche de **3.71 GB** pendant les trois fenêtres.

Ce résultat confirme la faisabilité du prototype, mais ne permet pas encore d'extrapoler le coût d'un entraînement complet.

Les premiers steps de PaySim sont relativement petits et la mémoire dynamique ne contient encore que quelques milliers de comptes.

Un pilote plus représentatif est nécessaire sur une portion du Train contenant :

- davantage de transactions par step ;
- une mémoire temporelle déjà plus développée ;
- des destinations plus fréquemment connues.

### Validation technique

Les contrôles finaux donnent :

- loss finie : **True**
- gradient du prédicteur : **True**
- gradient du module de message : **True**
- gradient de la GRU : **True**
- croissance monotone des slots : **True**
- mémoire finie : **True**
- mémoire effectivement mise à jour : **True**
- overlay vidé après TBPTT : **True**

Le pipeline suivant est donc désormais entraînable :

\[Graph\ Core\ Edge + State_{<t} + Known/New + \Delta t\]

\[\downarrow\]

\[Prediction(E_t)\]

\[\downarrow\]

\[Messages(E_t)
\rightarrow
Aggregation
\rightarrow
GRU\]

\[\downarrow\]

\[State_{\leq t}\]

La dynamique temporelle reçoit réellement un signal d'apprentissage provenant des steps futurs de la fenêtre TBPTT.



# 8. Pilote mémoire et runtime représentatif

## 8.1 Objectif

Le micro-pilote précédent a démontré que l'architecture temporelle est entraînable et que les gradients atteignent correctement le prédicteur, le module de message et la GRU.

Cependant, les steps 1 à 6 contiennent relativement peu de transactions et la mémoire temporelle ne contient encore que quelques milliers de comptes.

Ces résultats ne permettent donc pas d'estimer directement le coût d'un entraînement complet sur les **334 steps du Train**.

L'objectif de cette section est de construire un pilote plus représentatif permettant de mesurer :

- le runtime par step ;
- le runtime par fenêtre TBPTT ;
- la RAM disponible ;
- le nombre de slots temporels actifs ;
- la taille de l'overlay différentiable ;
- la stabilité des gradients ;
- le comportement du modèle lorsque les steps contiennent plusieurs dizaines de milliers de transactions.

Le benchmark doit rester limité au **Train**.

Aucune donnée de Validation ou de Test ne sera utilisée pour prendre une décision concernant le matériel ou l'implémentation.

## 8.2 Choix de la fenêtre représentative

La fenêtre pilote ne doit pas être choisie arbitrairement.

Le nombre de transactions de chaque step peut être calculé directement à partir des offsets :

$$
N_t = offsets[t] - offsets[t-1]
$$

Cette opération n'exige aucun chargement du Parquet ni reconstruction du graphe.

Nous allons d'abord examiner :

- la distribution des volumes par step ;
- les steps les plus lourds du Train ;
- plusieurs quantiles de cette distribution.

Une fenêtre représentative sera ensuite choisie à partir de ces résultats.

Le pilote devra idéalement se situer suffisamment loin dans le Train pour que l'état temporel contienne déjà un nombre important de destinations observées.

In [36]:
# Nombre de transactions par step Train
train_step_counts = np.diff(
    np.asarray(offsets[:335], dtype=np.int64)
)

train_steps = np.arange(
    1,
    335,
    dtype=np.int16,
)

print("Steps Train :", len(train_steps))
print(
    "Transactions Train :",
    f"{train_step_counts.sum():,}"
)

print()
print("Distribution des transactions par step")

for q in [0, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.00]:
    value = np.quantile(
        train_step_counts,
        q,
    )

    print(
        f"q={q:>4.2f} : "
        f"{value:,.0f}"
    )

Steps Train : 334
Transactions Train : 4,771,387

Distribution des transactions par step
q=0.00 : 2
q=0.25 : 12
q=0.50 : 6,594
q=0.75 : 29,522
q=0.90 : 36,563
q=0.95 : 40,623
q=0.99 : 47,318
q=1.00 : 51,352


In [37]:
# Trier les steps par volume décroissant
top_idx = np.argsort(
    train_step_counts
)[::-1][:15]

print("15 steps Train les plus volumineux")
print()

for idx in top_idx:
    t = int(train_steps[idx])
    n = int(train_step_counts[idx])

    print(
        f"Step {t:<3} | "
        f"transactions={n:,}"
    )

15 steps Train les plus volumineux

Step 19  | transactions=51,352
Step 18  | transactions=49,579
Step 187 | transactions=49,083
Step 235 | transactions=47,491
Step 307 | transactions=46,968
Step 163 | transactions=46,352
Step 139 | transactions=46,054
Step 43  | transactions=45,060
Step 15  | transactions=44,609
Step 186 | transactions=43,747
Step 306 | transactions=43,615
Step 17  | transactions=43,361
Step 259 | transactions=43,328
Step 16  | transactions=42,471
Step 14  | transactions=41,485


In [38]:
# Volume sur fenêtres contiguës de 4 steps
window_size = 4

window_counts = np.convolve(
    train_step_counts,
    np.ones(window_size, dtype=np.int64),
    mode="valid",
)

top_windows = np.argsort(
    window_counts
)[::-1][:10]

print(
    f"10 fenêtres de {window_size} steps "
    "les plus volumineuses"
)
print()

for idx in top_windows:
    start_step = int(idx + 1)
    end_step = start_step + window_size - 1

    print(
        f"Steps {start_step:>3}-{end_step:<3} | "
        f"transactions={int(window_counts[idx]):,}"
    )

10 fenêtres de 4 steps les plus volumineuses

Steps  16-19  | transactions=186,763
Steps  17-20  | transactions=184,917
Steps  15-18  | transactions=180,020
Steps  14-17  | transactions=171,926
Steps 185-188 | transactions=169,232
Steps  13-16  | transactions=166,080
Steps 184-187 | transactions=162,328
Steps 305-308 | transactions=162,161
Steps  41-44  | transactions=161,235
Steps  18-21  | transactions=160,708


## 8.3 Sélection de la fenêtre de benchmark

La distribution des volumes transactionnels du Train montre une forte hétérogénéité entre les steps.

Les principaux quantiles observés sont :

| Quantile | Transactions par step |
| ---: | ---: |
| 50 % | 6 594 |
| 75 % | 29 522 |
| 90 % | 36 563 |
| 95 % | 40 623 |
| 99 % | 47 318 |
| Maximum | 51 352 |

Le benchmark ne doit pas utiliser uniquement un step médian, car cela sous-estimerait le coût des périodes transactionnelles denses.

À l'inverse, sélectionner uniquement le maximum constituerait un stress test plutôt qu'une mesure représentative.

La fenêtre retenue est :

> **Steps 185 à 188**

Elle contient au total :

$$
169\,232 \text{ transactions}
$$

soit environ :

$$
\frac{169\,232}{4}
=
42\,308
$$

transactions par step en moyenne.

Cette charge se situe légèrement au-dessus du 95e percentile du volume par step et représente donc un scénario exigeant mais réaliste.

La fenêtre présente également un autre avantage important : elle se situe suffisamment tard dans le Train pour qu'une partie significative des destinations possède déjà un historique temporel.

Le benchmark sera effectué avec des fenêtres TBPTT de deux steps :

- steps 185–186 ;
- steps 187–188.

Aucune conclusion de performance prédictive ne sera tirée de cette expérience.

L'objectif est exclusivement de mesurer :

- le coût de la mémoire temporelle ;
- le nombre de comptes connus ;
- le runtime du forward et du backward ;
- la consommation RAM ;
- la stabilité numérique ;
- les gradients du prédicteur, du module de message et de la GRU.

## 8.4 Warm-up causal de la mémoire

Pour représenter correctement l'état disponible avant le step 185, les steps 1 à 184 doivent être intégrés dans l'index temporel.

Cependant, il n'est pas nécessaire de rejouer le modèle neuronal complet pour cette mesure de faisabilité.

Le benchmark utilisera donc un **warm-up structurel** :

1. les destinations des steps 1 à 184 sont parcourues dans l'ordre ;
2. les slots temporels sont créés causalement ;
3. `last_update` est mis à jour ;
4. aucun forward neuronal n'est exécuté ;
5. les vecteurs de mémoire sont initialisés à zéro avant la fenêtre mesurée.

Cette procédure reproduit exactement :

- la population de nœuds Known/New ;
- les slots actifs ;
- les valeurs de `last_update` ;
- les besoins de stockage de la mémoire.

Elle ne reproduit pas les valeurs apprises des états temporels.

Le pilote ne pourra donc pas être utilisé pour évaluer la qualité prédictive du modèle.

Cette simplification est volontaire : la présente section mesure la **faisabilité computationnelle**, et non les performances.

Un entraînement causal depuis le step 1 sera réalisé uniquement après validation de cette faisabilité.

In [39]:
import time
import psutil

# Préparer un index suffisamment grand
benchmark_index = SparseTemporalIndex(
    num_nodes=9_073_900,
    initial_capacity=2_100_000,
    growth_size=100_000,
)

warmup_start = time.perf_counter()

# Intégrer causalement les steps 1 à 184
for t in range(1, 185):
    start = int(offsets[t - 1])
    end = int(offsets[t])

    benchmark_index.integrate_destinations(
        dst[start:end],
        t=t,
    )

warmup_runtime = time.perf_counter() - warmup_start

print("Warm-up terminé : True")
print("Dernier step intégré : 184")
print(
    "Slots actifs :",
    f"{benchmark_index.size:,}",
)
print(
    "Capacité index :",
    f"{benchmark_index.capacity:,}",
)
print(
    "Runtime warm-up :",
    f"{warmup_runtime:.2f} s",
)
print(
    "RAM disponible :",
    f"{psutil.virtual_memory().available / 1024**3:.2f} GB",
)

Warm-up terminé : True
Dernier step intégré : 184
Slots actifs : 918,981
Capacité index : 2,100,000
Runtime warm-up : 0.21 s
RAM disponible : 4.83 GB


In [40]:
# Vérifier la causalité de l'état warm-up
active_last_update = benchmark_index.last_update[
    :benchmark_index.size
]

valid_last_update = bool(
    np.all(active_last_update < 185)
)

valid_positive = bool(
    np.all(active_last_update >= 1)
)

print(
    "Tous les last_update < 185 :",
    valid_last_update,
)
print(
    "Tous les last_update >= 1 :",
    valid_positive,
)
print(
    "last_update min :",
    int(active_last_update.min()),
)
print(
    "last_update max :",
    int(active_last_update.max()),
)

# Known/New au début du benchmark
batch_185 = reader.read_step(185)

_, src_known_185, src_dt_185 = (
    benchmark_index.lookup(
        batch_185["src"],
        t=185,
    )
)

_, dst_known_185, dst_dt_185 = (
    benchmark_index.lookup(
        batch_185["dst"],
        t=185,
    )
)

print()
print("Step 185")
print(
    "Transactions :",
    f"{len(batch_185['src']):,}",
)
print(
    "Sources connues :",
    f"{int(src_known_185.sum()):,}",
    f"({100 * src_known_185.mean():.2f} %)",
)
print(
    "Destinations connues :",
    f"{int(dst_known_185.sum()):,}",
    f"({100 * dst_known_185.mean():.2f} %)",
)

if dst_known_185.any():
    print(
        "Delta t destination connu min/max :",
        int(dst_dt_185[dst_known_185].min()),
        "/",
        int(dst_dt_185[dst_known_185].max()),
    )

Tous les last_update < 185 : True
Tous les last_update >= 1 : True
last_update min : 1
last_update max : 184

Step 185
Transactions : 37,323
Sources connues : 7 (0.02 %)
Destinations connues : 21,999 (58.94 %)
Delta t destination connu min/max : 1 / 174


In [41]:
# Reproductibilité
torch.manual_seed(42)

# Nouveau modèle pour le benchmark
benchmark_model = PilotDynamicModel(
    edge_dim=13,
    memory_dim=8,
    hidden_dim=32,
)

# Mémoire alignée sur l'index warm-up
benchmark_memory = PilotTemporalMemory(
    temporal_index=benchmark_index,
    memory_dim=8,
    initial_capacity=benchmark_index.capacity,
    growth_size=100_000,
)

# Loss
benchmark_pos_weight = torch.tensor(
    1273.0686248331108,
    dtype=torch.float32,
)

benchmark_criterion = nn.BCEWithLogitsLoss(
    pos_weight=benchmark_pos_weight,
    reduction="sum",
)

benchmark_optimizer = torch.optim.AdamW(
    benchmark_model.parameters(),
    lr=1e-3,
    weight_decay=1e-5,
)

memory_bytes = (
    benchmark_memory.memory.numel()
    * benchmark_memory.memory.element_size()
)

print(
    "Slots actifs avant benchmark :",
    f"{benchmark_index.size:,}",
)
print(
    "Capacité mémoire :",
    f"{benchmark_memory.capacity:,}",
)
print(
    "Tensor mémoire :",
    f"{memory_bytes / 1024**2:.2f} MiB",
)
print(
    "RAM disponible :",
    f"{psutil.virtual_memory().available / 1024**3:.2f} GB",
)

Slots actifs avant benchmark : 918,981
Capacité mémoire : 2,100,000
Tensor mémoire : 64.09 MiB
RAM disponible : 4.63 GB


In [42]:
benchmark_history = []

process = psutil.Process()

benchmark_start = time.perf_counter()

for window_start in [185, 187]:

    benchmark_optimizer.zero_grad()

    loss_sum = 0.0
    window_tx = 0
    window_fraud = 0

    step_records = []

    window_start_time = time.perf_counter()

    for t in range(
        window_start,
        window_start + 2,
    ):
        step_start_time = time.perf_counter()

        batch = reader.read_step(t)

        logits, y, info = process_pilot_step(
            model=benchmark_model,
            temporal_memory=benchmark_memory,
            batch=batch,
        )

        step_loss = benchmark_criterion(
            logits,
            y,
        )

        loss_sum = loss_sum + step_loss

        window_tx += info["transactions"]
        window_fraud += info["frauds"]

        step_runtime = (
            time.perf_counter()
            - step_start_time
        )

        rss_gib = (
            process.memory_info().rss
            / 1024**3
        )

        ram_available = (
            psutil.virtual_memory().available
            / 1024**3
        )

        step_records.append({
            "step": t,
            "transactions": info["transactions"],
            "frauds": info["frauds"],
            "known_src": info["known_src"],
            "known_dst": info["known_dst"],
            "unique_dst": info["unique_dst"],
            "active_slots": info["active_slots"],
            "runtime": step_runtime,
            "rss_gib": rss_gib,
            "ram_available": ram_available,
        })

        print(
            f"Step {t} | "
            f"tx={info['transactions']:,} | "
            f"fraudes={info['frauds']} | "
            f"known_src={info['known_src']:,} | "
            f"known_dst={info['known_dst']:,} | "
            f"runtime={step_runtime:.2f}s | "
            f"RAM libre={ram_available:.2f} GB"
        )

    # Loss moyenne par transaction
    loss = loss_sum / window_tx

    backward_start = time.perf_counter()

    loss.backward()

    backward_runtime = (
        time.perf_counter()
        - backward_start
    )

    # Gradients
    predictor_grad = torch.sqrt(
        sum(
            (
                p.grad.detach().norm() ** 2
                if p.grad is not None
                else torch.tensor(0.0)
            )
            for p in benchmark_model.predictor.parameters()
        )
    ).item()

    message_grad = torch.sqrt(
        sum(
            (
                p.grad.detach().norm() ** 2
                if p.grad is not None
                else torch.tensor(0.0)
            )
            for p in benchmark_model.message_module.parameters()
        )
    ).item()

    gru_grad = torch.sqrt(
        sum(
            (
                p.grad.detach().norm() ** 2
                if p.grad is not None
                else torch.tensor(0.0)
            )
            for p in benchmark_model.gru.parameters()
        )
    ).item()

    total_grad = torch.nn.utils.clip_grad_norm_(
        benchmark_model.parameters(),
        max_norm=5.0,
    )

    benchmark_optimizer.step()

    # Détacher la mémoire
    benchmark_memory.commit()

    window_runtime = (
        time.perf_counter()
        - window_start_time
    )

    result = {
        "steps": (
            window_start,
            window_start + 1,
        ),
        "transactions": window_tx,
        "frauds": window_fraud,
        "loss": float(loss.detach()),
        "predictor_grad": predictor_grad,
        "message_grad": message_grad,
        "gru_grad": gru_grad,
        "total_grad": float(total_grad),
        "backward_runtime": backward_runtime,
        "window_runtime": window_runtime,
        "active_slots": benchmark_index.size,
        "step_records": step_records,
    }

    benchmark_history.append(result)

    print(
        f"\nFenêtre {window_start}-{window_start + 1}"
    )
    print(
        f"  tx={window_tx:,} | "
        f"fraudes={window_fraud} | "
        f"loss={float(loss.detach()):.6f}"
    )
    print(
        f"  grad predictor={predictor_grad:.6f} | "
        f"message={message_grad:.6f} | "
        f"GRU={gru_grad:.6f}"
    )
    print(
        f"  backward={backward_runtime:.2f}s | "
        f"runtime total={window_runtime:.2f}s"
    )
    print(
        f"  slots actifs={benchmark_index.size:,} | "
        f"RAM libre="
        f"{psutil.virtual_memory().available / 1024**3:.2f} GB"
    )
    print()

print(
    "Runtime benchmark total :",
    f"{time.perf_counter() - benchmark_start:.2f} s",
)

Step 185 | tx=37,323 | fraudes=16 | known_src=7 | known_dst=21,999 | runtime=1.27s | RAM libre=4.42 GB
Step 186 | tx=43,747 | fraudes=17 | known_src=5 | known_dst=26,151 | runtime=2.63s | RAM libre=4.07 GB

Fenêtre 185-186
  tx=81,070 | fraudes=33 | loss=1.170635
  grad predictor=2.546268 | message=0.000604 | GRU=0.000991
  backward=0.95s | runtime total=5.90s
  slots actifs=951,473 | RAM libre=4.23 GB

Step 187 | tx=49,083 | fraudes=10 | known_src=2 | known_dst=28,977 | runtime=0.63s | RAM libre=4.16 GB
Step 188 | tx=39,079 | fraudes=10 | known_src=3 | known_dst=22,163 | runtime=0.50s | RAM libre=4.07 GB

Fenêtre 187-188
  tx=88,162 | fraudes=20 | loss=0.947272
  grad predictor=1.907590 | message=0.000592 | GRU=0.000936
  backward=1.09s | runtime total=3.29s
  slots actifs=988,068 | RAM libre=4.22 GB

Runtime benchmark total : 9.21 s


In [43]:
benchmark_loss_finite = all(
    np.isfinite(item["loss"])
    for item in benchmark_history
)

benchmark_predictor_grad = all(
    item["predictor_grad"] > 0
    for item in benchmark_history
)

benchmark_message_grad = all(
    item["message_grad"] > 0
    for item in benchmark_history
)

benchmark_gru_grad = all(
    item["gru_grad"] > 0
    for item in benchmark_history
)

benchmark_memory_finite = bool(
    torch.isfinite(
        benchmark_memory.memory[
            :benchmark_index.size
        ]
    ).all()
)

benchmark_overlay_empty = (
    len(benchmark_memory.overlay) == 0
)

print(
    "Loss finie :",
    benchmark_loss_finite,
)
print(
    "Gradient prédicteur :",
    benchmark_predictor_grad,
)
print(
    "Gradient message :",
    benchmark_message_grad,
)
print(
    "Gradient GRU :",
    benchmark_gru_grad,
)
print(
    "Mémoire finie :",
    benchmark_memory_finite,
)
print(
    "Overlay vidé :",
    benchmark_overlay_empty,
)
print(
    "Slots actifs finaux :",
    f"{benchmark_index.size:,}",
)
print(
    "RSS processus final :",
    f"{process.memory_info().rss / 1024**3:.2f} GB",
)
print(
    "RAM disponible finale :",
    f"{psutil.virtual_memory().available / 1024**3:.2f} GB",
)

Loss finie : True
Gradient prédicteur : True
Gradient message : True
Gradient GRU : True
Mémoire finie : True
Overlay vidé : True
Slots actifs finaux : 988,068
RSS processus final : 0.66 GB
RAM disponible finale : 4.28 GB


### 8.5 Résultats du benchmark représentatif

Le benchmark a été réalisé sur les steps 185 à 188 du Train.

Cette fenêtre contient au total :

$$
169\,232 \text{ transactions}
$$

et se situe dans une zone de forte activité transactionnelle, avec environ 42 000 transactions par step en moyenne.

Avant le benchmark, un warm-up structurel causal a été effectué sur les steps 1 à 184.

Ce warm-up a produit :

- **918 981 slots temporels actifs** ;
- un `last_update` compris entre 1 et 184 ;
- aucune violation de la contrainte $last\_update < 185$.

Le warm-up structurel s'est exécuté en environ **0,21 seconde**.

### Régime Known/New

Au début du step 185 :

- transactions : **37 323** ;
- sources connues : **7**, soit environ **0,02 %** ;
- destinations connues : **21 999**, soit environ **58,94 %**.

Les valeurs de $\Delta t$ observées pour les destinations connues sont comprises entre 1 et 174 steps.

Ces résultats confirment le caractère fortement asymétrique du graphe PaySim :

$$
P(\text{source connue}) \approx 0
$$

alors qu'une proportion importante des destinations possède un historique exploitable.

Cette observation renforce la justification de la mémoire temporelle destination-centric retenue dans les sections précédentes.

### Runtime du modèle temporel

Les quatre steps du benchmark ont produit les résultats suivants :

| Step | Transactions | Fraudes | Sources connues | Destinations connues | Runtime |
| ---: | ---: | ---: | ---: | ---: | ---: |
| 185 | 37 323 | 16 | 7 | 21 999 | 1,27 s |
| 186 | 43 747 | 17 | 5 | 26 151 | 2,63 s |
| 187 | 49 083 | 10 | 2 | 28 977 | 0,63 s |
| 188 | 39 079 | 10 | 3 | 22 163 | 0,50 s |

Les deux fenêtres TBPTT donnent :

| Fenêtre | Transactions | Fraudes | Loss | Backward | Runtime total |
| ---: | ---: | ---: | ---: | ---: | ---: |
| 185–186 | 81 070 | 33 | 1.170635 | 0,95 s | 5,90 s |
| 187–188 | 88 162 | 20 | 0.947272 | 1,09 s | 3,29 s |

Le benchmark complet s'exécute en environ **9,21 secondes sur CPU**.

Ces valeurs servent uniquement à mesurer la faisabilité computationnelle.

Elles ne doivent pas être utilisées pour évaluer les performances prédictives du modèle, car l'état neuronal antérieur au step 185 a été initialisé structurellement et non appris depuis le début du Train.

### Gradients temporels

Les deux fenêtres produisent des gradients strictement non nuls pour les trois composants entraînables.

Pour les steps 185–186 :

- prédicteur : **2.546268** ;
- module de message : **0.000604** ;
- GRU : **0.000991**.

Pour les steps 187–188 :

- prédicteur : **1.907590** ;
- module de message : **0.000592** ;
- GRU : **0.000936**.

Le signal transmis aux composants temporels est beaucoup plus faible que celui du prédicteur, mais il reste présent dans les deux fenêtres.

Ce comportement sera surveillé pendant l'entraînement complet sans modifier prématurément l'architecture sur la seule base de ce pilote.

Le gradient clipping reste conservé pour limiter les risques d'instabilité.

### Consommation mémoire

Avant le benchmark :

- slots actifs : **918 981** ;
- capacité allouée : **2 100 000 slots** ;
- tenseur de mémoire : **64,09 MiB**.

Après les steps 185 à 188 :

- slots actifs : **988 068** ;
- RSS du processus : **0,66 GB** ;
- RAM disponible finale : **4,28 GB**.

La RAM disponible est restée supérieure à environ **4 GB** pendant tout le benchmark.

La capacité actuelle de 2,1 millions de slots est également suffisante pour couvrir les **2 035 605 destinations uniques du Train** identifiées précédemment.

### Validation technique

Les contrôles finaux donnent :

- loss finie : **True** ;
- gradient du prédicteur : **True** ;
- gradient du module de message : **True** ;
- gradient de la GRU : **True** ;
- mémoire finie : **True** ;
- overlay correctement vidé : **True**.

Aucun problème numérique ou mémoire n'a été observé sur une fenêtre représentative à forte charge transactionnelle.

## 8.6 Décision CPU / GPU

Le benchmark ne met en évidence aucune nécessité immédiate d'utiliser un GPU.

L'environnement CPU local est capable de traiter :

- plusieurs dizaines de milliers de transactions par step ;
- une mémoire contenant près d'un million de slots actifs ;
- le forward temporel ;
- le backward TBPTT ;
- la mise à jour GRU ;

tout en maintenant une consommation RAM raisonnable.

La décision retenue est donc :

> **L'entraînement principal du Dynamic GNN sera lancé en priorité sur le CPU local.**

Google Colab T4 reste une solution de repli si l'entraînement complet révèle un coût significativement supérieur à celui observé lors du benchmark, mais aucune migration GPU n'est justifiée à ce stade.

Cette décision repose sur un benchmark réel et non sur une estimation théorique.

**Section 8 validée.**

# 9. Entraînement temporel complet sur le Train

## 9.1 Objectif

Les sections précédentes ont validé :

- la lecture causale des événements ;
- la mémoire temporelle destination-centric ;
- les indicateurs Known/New ;
- le calcul de $\Delta t$ ;
- l'agrégation intra-step ;
- la mise à jour GRU ;
- le TBPTT ;
- la propagation des gradients ;
- la faisabilité CPU et mémoire sur une fenêtre représentative.

Le modèle peut maintenant être entraîné sur l'intégralité du split Train :

$$
t = 1,\ldots,334
$$

soit :

$$
4\,771\,387
$$

transactions et :

$$
3\,745
$$

fraudes.

L'entraînement doit conserver strictement l'ordre temporel.

Pour chaque step $t$ :

$$
State_{<t}
\rightarrow
Prediction(E_t)
\rightarrow
Messages(E_t)
\rightarrow
Aggregation(E_t)
\rightarrow
GRU
\rightarrow
State_{\leq t}
$$

Aucun événement du step courant ne peut modifier la mémoire utilisée pour prédire une autre transaction du même step.

## 9.2 Configuration du premier candidat dynamique

Le premier candidat entraîné reprend la configuration validée lors des micro-pilotes :

| Paramètre | Valeur |
| --- | ---: |
| Graph Core Edge | 13 features |
| `memory_dim` | 8 |
| `hidden_dim` | 32 |
| Agrégation | moyenne |
| TBPTT | 2 steps |
| Learning rate | $10^{-3}$ |
| Weight decay | $10^{-5}$ |
| Gradient clipping | 5.0 |
| `pos_weight` | 1273.0686 |
| Device | CPU |
| Epochs | 1 passage temporel |

Cette configuration constitue le **premier candidat dynamique**, et non encore le modèle final sélectionné.

Les choix d'architecture ne seront comparés qu'à partir de la Validation.

Le Test reste totalement exclu de cette phase.

## 9.3 Un seul passage temporel

Contrairement à un dataset tabulaire classique, le modèle maintient un état temporel qui évolue avec le flux transactionnel.

Un passage complet sur le Train correspond donc à :

$$
State_0
\rightarrow
E_1
\rightarrow
State_1
\rightarrow
E_2
\rightarrow
\cdots
\rightarrow
E_{334}
\rightarrow
State_{334}
$$

Le premier candidat sera entraîné pendant **un passage causal complet** sur le Train.

Aucun mélange des steps et aucun shuffle ne sont autorisés.

## 9.4 Backpropagation temporelle tronquée

La fenêtre TBPTT reste fixée à deux steps pour ce premier candidat.

Pour une fenêtre composée des steps $t$ et $t+1$ :

1. le modèle traite $E_t$ ;
2. l'état différentiable produit est conservé ;
3. le modèle traite $E_{t+1}$ ;
4. les losses sont agrégées ;
5. `backward()` est exécuté ;
6. les gradients sont éventuellement clippés ;
7. l'optimiseur met à jour les paramètres ;
8. la mémoire temporelle est détachée.

Le graphe d'autograd ne traverse donc jamais les 334 steps simultanément.

## 9.5 Checkpointing et reprise

L'entraînement complet doit pouvoir reprendre après une interruption du kernel.

Un checkpoint doit sauvegarder au minimum :

- les paramètres du modèle ;
- l'état de l'optimiseur ;
- le dernier step complètement intégré ;
- la mémoire temporelle persistante ;
- `slot_of_node` ;
- `node_of_slot` ;
- `last_update` ;
- l'historique d'entraînement ;
- l'état des générateurs aléatoires.

Les checkpoints sont créés uniquement lorsque l'overlay TBPTT a été vidé.

Ainsi, chaque checkpoint représente un état causal cohérent :

$$
State_{\leq t}
$$

et jamais un état intermédiaire d'une fenêtre non terminée.

Le fichier `latest` sera remplacé atomiquement afin d'éviter qu'une interruption pendant la sauvegarde ne laisse un checkpoint partiellement écrit.

## 9.6 Politique de sauvegarde

Un checkpoint intermédiaire sera sauvegardé périodiquement pendant le Train.

Le même fichier `latest` sera écrasé afin d'éviter d'accumuler plusieurs gigaoctets de checkpoints.

À la fin du step 334, un artefact final séparé sera conservé.

L'historique d'entraînement sera également sauvegardé en JSON pour permettre l'analyse du runtime, de la loss, des gradients et de la consommation mémoire.

Avant le véritable entraînement, le mécanisme de sauvegarde et de restauration sera testé indépendamment.

In [44]:
import json
import hashlib
from pathlib import Path

# Répertoire du modèle dynamique
dynamic_artifact_dir = Path(
    "../artifacts/dynamic_gnn"
)

dynamic_artifact_dir.mkdir(
    parents=True,
    exist_ok=True,
)

checkpoint_path = (
    dynamic_artifact_dir
    / "dynamic_core_edge_candidate_latest.pt"
)

final_checkpoint_path = (
    dynamic_artifact_dir
    / "dynamic_core_edge_candidate_train_final.pt"
)

history_path = (
    dynamic_artifact_dir
    / "dynamic_core_edge_candidate_train_history.json"
)

config_path = (
    dynamic_artifact_dir
    / "dynamic_core_edge_candidate_config.json"
)

# Empreinte du preprocessing
with open(
    preprocessing_path,
    "rb",
) as f:
    preprocessing_sha256 = hashlib.sha256(
        f.read()
    ).hexdigest()

train_config = {
    "model": "sparse_dynamic_gnn",
    "architecture": "tgn_inspired_sparse_memory",
    "feature_set": "graph_core_edge",
    "input_dim": 33,
    "edge_dim": 13,
    "memory_dim": 8,
    "hidden_dim": 32,
    "aggregation": "mean",
    "tbptt_steps": 2,
    "learning_rate": 1e-3,
    "weight_decay": 1e-5,
    "gradient_clip": 5.0,
    "pos_weight": 1273.0686248331108,
    "train_start_step": 1,
    "train_end_step": 334,
    "train_transactions": 4_771_387,
    "train_frauds": 3_745,
    "epochs": 1,
    "shuffle": False,
    "device": "cpu",
    "destination_centric_memory": True,
    "same_step_update": False,
    "node_id_embedding": False,
    "historical_labels": False,
    "random_seed": 42,
    "preprocessing_sha256": preprocessing_sha256,
}

with open(
    config_path,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        train_config,
        f,
        indent=2,
        ensure_ascii=False,
    )

print("Répertoire :", dynamic_artifact_dir)
print("Configuration sauvegardée :", config_path.exists())
print("Preprocessing SHA-256 :", preprocessing_sha256)
print("Train steps :", "1-334")
print("TBPTT :", train_config["tbptt_steps"])
print("Device :", train_config["device"])

Répertoire : ..\artifacts\dynamic_gnn
Configuration sauvegardée : True
Preprocessing SHA-256 : 48873cb41fee4e948133d751ac46129427f28c331796a007da63629a23a1cf8b
Train steps : 1-334
TBPTT : 2
Device : cpu


In [45]:
import os
import numpy as np
import torch


def save_training_checkpoint(
    path,
    model,
    optimizer,
    temporal_memory,
    completed_step,
    history,
    config,
):
    # Sauvegarder uniquement un état TBPTT propre
    if len(temporal_memory.overlay) != 0:
        raise RuntimeError(
            "Checkpoint interdit : overlay TBPTT non vide."
        )

    index = temporal_memory.index
    active_size = index.size

    payload = {
        "completed_step": int(completed_step),
        "config": config,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),

        # Etat de l'index temporel
        "index_size": int(active_size),
        "index_capacity": int(index.capacity),
        "slot_of_node": index.slot_of_node.copy(),
        "node_of_slot": (
            index.node_of_slot[:active_size].copy()
        ),
        "last_update": (
            index.last_update[:active_size].copy()
        ),

        # Mémoire persistante
        "memory": (
            temporal_memory.memory[:active_size]
            .detach()
            .cpu()
            .clone()
        ),

        # Historique
        "history": history,

        # Etats aléatoires
        "torch_rng_state": torch.get_rng_state(),
        "numpy_rng_state": np.random.get_state(),
    }

    path = Path(path)

    tmp_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    torch.save(
        payload,
        tmp_path,
    )

    # Remplacement atomique
    os.replace(
        tmp_path,
        path,
    )


def load_training_checkpoint(
    path,
    model,
    optimizer,
    memory_dim,
    growth_size=100_000,
):
    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    size = int(
        checkpoint["index_size"]
    )

    saved_capacity = int(
        checkpoint["index_capacity"]
    )

    capacity = max(
        saved_capacity,
        size,
    )

    # Reconstruire l'index
    restored_index = SparseTemporalIndex(
        num_nodes=len(
            checkpoint["slot_of_node"]
        ),
        initial_capacity=capacity,
        growth_size=growth_size,
    )

    restored_index.slot_of_node[:] = (
        checkpoint["slot_of_node"]
    )

    restored_index.node_of_slot[:size] = (
        checkpoint["node_of_slot"]
    )

    restored_index.last_update[:size] = (
        checkpoint["last_update"]
    )

    restored_index.size = size

    # Reconstruire la mémoire
    restored_memory = PilotTemporalMemory(
        temporal_index=restored_index,
        memory_dim=memory_dim,
        initial_capacity=capacity,
        growth_size=growth_size,
    )

    restored_memory.memory[:size] = (
        checkpoint["memory"]
    )

    # Restaurer les générateurs aléatoires
    torch.set_rng_state(
        checkpoint["torch_rng_state"]
    )

    np.random.set_state(
        checkpoint["numpy_rng_state"]
    )

    return {
        "completed_step": int(
            checkpoint["completed_step"]
        ),
        "history": checkpoint["history"],
        "config": checkpoint["config"],
        "temporal_memory": restored_memory,
    }

In [46]:
# Modèle de test
torch.manual_seed(42)

checkpoint_test_model = PilotDynamicModel(
    edge_dim=13,
    memory_dim=8,
    hidden_dim=32,
)

checkpoint_test_optimizer = torch.optim.AdamW(
    checkpoint_test_model.parameters(),
    lr=1e-3,
    weight_decay=1e-5,
)

# Mémoire de test
checkpoint_test_index = SparseTemporalIndex(
    num_nodes=9_073_900,
    initial_capacity=100_000,
)

checkpoint_test_memory = PilotTemporalMemory(
    temporal_index=checkpoint_test_index,
    memory_dim=8,
    initial_capacity=100_000,
)

# Intégrer deux steps structurellement
for t in [1, 2]:
    batch = reader.read_step(t)

    checkpoint_test_index.integrate_destinations(
        batch["dst"],
        t=t,
    )

checkpoint_test_memory.ensure_capacity()

# Placer une valeur identifiable dans la mémoire
with torch.no_grad():
    checkpoint_test_memory.memory[
        :checkpoint_test_index.size
    ] = 0.123456

test_checkpoint_path = (
    dynamic_artifact_dir
    / "_checkpoint_roundtrip_test.pt"
)

test_history = [
    {
        "completed_step": 2,
        "test": True,
    }
]

save_training_checkpoint(
    path=test_checkpoint_path,
    model=checkpoint_test_model,
    optimizer=checkpoint_test_optimizer,
    temporal_memory=checkpoint_test_memory,
    completed_step=2,
    history=test_history,
    config=train_config,
)

print(
    "Checkpoint test sauvegardé :",
    test_checkpoint_path.exists(),
)

print(
    "Taille checkpoint :",
    f"{test_checkpoint_path.stat().st_size / 1024**2:.2f} MiB",
)

Checkpoint test sauvegardé : True
Taille checkpoint : 69.33 MiB


In [47]:
# Nouveau modèle vide
restored_model = PilotDynamicModel(
    edge_dim=13,
    memory_dim=8,
    hidden_dim=32,
)

restored_optimizer = torch.optim.AdamW(
    restored_model.parameters(),
    lr=1e-3,
    weight_decay=1e-5,
)

restored = load_training_checkpoint(
    path=test_checkpoint_path,
    model=restored_model,
    optimizer=restored_optimizer,
    memory_dim=8,
)

restored_memory = restored["temporal_memory"]

# Vérifier l'index
same_slots = np.array_equal(
    checkpoint_test_index.slot_of_node,
    restored_memory.index.slot_of_node,
)

same_nodes = np.array_equal(
    checkpoint_test_index.node_of_slot[
        :checkpoint_test_index.size
    ],
    restored_memory.index.node_of_slot[
        :restored_memory.index.size
    ],
)

same_last_update = np.array_equal(
    checkpoint_test_index.last_update[
        :checkpoint_test_index.size
    ],
    restored_memory.index.last_update[
        :restored_memory.index.size
    ],
)

# Vérifier la mémoire
same_memory = torch.equal(
    checkpoint_test_memory.memory[
        :checkpoint_test_index.size
    ],
    restored_memory.memory[
        :restored_memory.index.size
    ],
)

# Vérifier les paramètres
same_model = all(
    torch.equal(
        p1.detach(),
        p2.detach(),
    )
    for p1, p2 in zip(
        checkpoint_test_model.parameters(),
        restored_model.parameters(),
    )
)

print(
    "Completed step :",
    restored["completed_step"],
)
print(
    "Taille index restaurée :",
    f"{restored_memory.index.size:,}",
)
print(
    "slot_of_node identique :",
    same_slots,
)
print(
    "node_of_slot identique :",
    same_nodes,
)
print(
    "last_update identique :",
    same_last_update,
)
print(
    "Mémoire identique :",
    same_memory,
)
print(
    "Modèle identique :",
    same_model,
)
print(
    "Historique identique :",
    restored["history"] == test_history,
)
print(
    "Overlay vide :",
    len(restored_memory.overlay) == 0,
)

Completed step : 2
Taille index restaurée : 2,259
slot_of_node identique : True
node_of_slot identique : True
last_update identique : True
Mémoire identique : True
Modèle identique : True
Historique identique : True
Overlay vide : True


In [48]:
# Supprimer uniquement le checkpoint temporaire de test
if test_checkpoint_path.exists():
    test_checkpoint_path.unlink()

print(
    "Checkpoint temporaire supprimé :",
    not test_checkpoint_path.exists(),
)

print(
    "Configuration conservée :",
    config_path.exists(),
)

Checkpoint temporaire supprimé : True
Configuration conservée : True


## 9.7 Validation du mécanisme de checkpoint

Avant de lancer l'entraînement complet, une sauvegarde suivie d'une restauration a été testée sur un état temporel construit à partir des steps 1 et 2.

Le checkpoint de test contenait :

- les paramètres du modèle ;
- l'état de l'optimiseur ;
- `slot_of_node` ;
- `node_of_slot` ;
- `last_update` ;
- la mémoire temporelle persistante ;
- l'historique d'entraînement ;
- les états des générateurs aléatoires.

Après restauration, les contrôles suivants sont tous validés :

- `slot_of_node` identique : **True** ;
- `node_of_slot` identique : **True** ;
- `last_update` identique : **True** ;
- mémoire temporelle identique : **True** ;
- paramètres du modèle identiques : **True** ;
- historique identique : **True** ;
- overlay TBPTT vide : **True**.

Le checkpoint temporaire utilisé pour ce test a ensuite été supprimé.

Le mécanisme de sauvegarde permet donc de reconstruire exactement un état causal :

$$
State_{\leq t}
$$

sans devoir rejouer les événements précédents.

L'entraînement complet peut désormais utiliser ce mécanisme pour reprendre automatiquement après une interruption.

**Checkpointing validé.**

In [49]:
import gc
import time
import psutil
import json
import numpy as np
import torch
import torch.nn as nn

# Sauvegarde atomique de l'historique
def save_history_json(path, history):
    path = Path(path)
    tmp_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    with open(
        tmp_path,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            history,
            f,
            indent=2,
            ensure_ascii=False,
        )

    os.replace(
        tmp_path,
        path,
    )


# Norme des gradients d'un module
def module_grad_norm(module):
    squared_norm = 0.0

    for parameter in module.parameters():
        if parameter.grad is not None:
            squared_norm += (
                parameter.grad
                .detach()
                .norm()
                .item() ** 2
            )

    return squared_norm ** 0.5


# Reproductibilité initiale
torch.manual_seed(
    train_config["random_seed"]
)
np.random.seed(
    train_config["random_seed"]
)

# Modèle complet
train_model = PilotDynamicModel(
    edge_dim=train_config["edge_dim"],
    memory_dim=train_config["memory_dim"],
    hidden_dim=train_config["hidden_dim"],
)

# Optimiseur
train_optimizer = torch.optim.AdamW(
    train_model.parameters(),
    lr=train_config["learning_rate"],
    weight_decay=train_config["weight_decay"],
)

# Fonction de coût
train_pos_weight = torch.tensor(
    train_config["pos_weight"],
    dtype=torch.float32,
)

train_criterion = nn.BCEWithLogitsLoss(
    pos_weight=train_pos_weight,
    reduction="sum",
)

# Reprise ou initialisation
if checkpoint_path.exists():

    restored = load_training_checkpoint(
        path=checkpoint_path,
        model=train_model,
        optimizer=train_optimizer,
        memory_dim=train_config["memory_dim"],
    )

    # Vérifier la compatibilité
    saved_config = restored["config"]

    critical_keys = [
        "feature_set",
        "edge_dim",
        "memory_dim",
        "hidden_dim",
        "aggregation",
        "tbptt_steps",
        "learning_rate",
        "weight_decay",
        "pos_weight",
        "preprocessing_sha256",
    ]

    config_match = all(
        saved_config[key] == train_config[key]
        for key in critical_keys
    )

    if not config_match:
        raise RuntimeError(
            "Le checkpoint existant n'est pas "
            "compatible avec la configuration."
        )

    train_memory = restored[
        "temporal_memory"
    ]

    train_history = restored[
        "history"
    ]

    completed_step = restored[
        "completed_step"
    ]

    start_step = completed_step + 1

    print("Mode : reprise")
    print(
        "Dernier step validé :",
        completed_step,
    )

else:

    train_index = SparseTemporalIndex(
        num_nodes=9_073_900,
        initial_capacity=2_100_000,
        growth_size=100_000,
    )

    train_memory = PilotTemporalMemory(
        temporal_index=train_index,
        memory_dim=train_config["memory_dim"],
        initial_capacity=2_100_000,
        growth_size=100_000,
    )

    train_history = []
    completed_step = 0
    start_step = 1

    print("Mode : nouvel entraînement")


print("Prochain step :", start_step)
print(
    "Fenêtres déjà terminées :",
    len(train_history),
)
print(
    "Slots actifs :",
    f"{train_memory.index.size:,}",
)
print(
    "Overlay vide :",
    len(train_memory.overlay) == 0,
)
print(
    "RAM disponible :",
    f"{psutil.virtual_memory().available / 1024**3:.2f} GB",
)


Mode : nouvel entraînement
Prochain step : 1
Fenêtres déjà terminées : 0
Slots actifs : 0
Overlay vide : True
RAM disponible : 3.00 GB


In [50]:
tbptt_steps = train_config[
    "tbptt_steps"
]

checkpoint_every = 20
log_every = 10

training_start_time = time.perf_counter()

# Cas où le Train est déjà terminé
if start_step > 334:
    print(
        "Train déjà terminé au step :",
        completed_step,
    )

else:

    for window_start in range(
        start_step,
        335,
        tbptt_steps,
    ):

        window_end = min(
            window_start
            + tbptt_steps
            - 1,
            334,
        )

        train_optimizer.zero_grad(
            set_to_none=True
        )

        loss_sum = None
        window_tx = 0
        window_fraud = 0
        window_known_src = 0
        window_known_dst = 0

        window_start_time = (
            time.perf_counter()
        )

        # Traiter les steps de la fenêtre TBPTT
        for t in range(
            window_start,
            window_end + 1,
        ):

            batch = reader.read_step(t)

            logits, y, info = (
                process_pilot_step(
                    model=train_model,
                    temporal_memory=train_memory,
                    batch=batch,
                )
            )

            step_loss = train_criterion(
                logits,
                y,
            )

            if loss_sum is None:
                loss_sum = step_loss
            else:
                loss_sum = (
                    loss_sum + step_loss
                )

            window_tx += info[
                "transactions"
            ]

            window_fraud += info[
                "frauds"
            ]

            window_known_src += info[
                "known_src"
            ]

            window_known_dst += info[
                "known_dst"
            ]

        # Loss moyenne par transaction
        loss = loss_sum / window_tx

        # Contrôle numérique avant backward
        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Loss non finie aux steps "
                f"{window_start}-{window_end}"
            )

        # Backpropagation
        loss.backward()

        predictor_grad = module_grad_norm(
            train_model.predictor
        )

        message_grad = module_grad_norm(
            train_model.message_module
        )

        gru_grad = module_grad_norm(
            train_model.gru
        )

        # Norme avant clipping
        total_grad = (
            torch.nn.utils.clip_grad_norm_(
                train_model.parameters(),
                max_norm=train_config[
                    "gradient_clip"
                ],
            )
        )

        total_grad_value = float(
            total_grad.detach()
        )

        # Vérifier les gradients
        gradient_values = [
            predictor_grad,
            message_grad,
            gru_grad,
            total_grad_value,
        ]

        if not all(
            np.isfinite(value)
            for value in gradient_values
        ):
            raise RuntimeError(
                f"Gradient non fini aux steps "
                f"{window_start}-{window_end}"
            )

        # Mise à jour des poids
        train_optimizer.step()

        # Détacher la mémoire temporelle
        train_memory.commit()

        completed_step = window_end

        window_runtime = (
            time.perf_counter()
            - window_start_time
        )

        ram_available = (
            psutil.virtual_memory().available
            / 1024**3
        )

        process_rss = (
            psutil.Process()
            .memory_info()
            .rss
            / 1024**3
        )

        history_item = {
            "start_step": int(
                window_start
            ),
            "end_step": int(
                window_end
            ),
            "transactions": int(
                window_tx
            ),
            "frauds": int(
                window_fraud
            ),
            "loss": float(
                loss.detach()
            ),
            "predictor_grad": float(
                predictor_grad
            ),
            "message_grad": float(
                message_grad
            ),
            "gru_grad": float(
                gru_grad
            ),
            "total_grad_preclip": float(
                total_grad_value
            ),
            "known_src": int(
                window_known_src
            ),
            "known_dst": int(
                window_known_dst
            ),
            "active_slots": int(
                train_memory.index.size
            ),
            "runtime_seconds": float(
                window_runtime
            ),
            "rss_gib": float(
                process_rss
            ),
            "ram_available_gib": float(
                ram_available
            ),
        }

        train_history.append(
            history_item
        )

        # Affichage périodique
        if (
            completed_step % log_every == 0
            or completed_step == 334
        ):
            print(
                f"Step {completed_step:>3}/334 | "
                f"tx={window_tx:,} | "
                f"fraudes={window_fraud} | "
                f"loss={float(loss.detach()):.6f}"
            )

            print(
                f"  grad predictor="
                f"{predictor_grad:.6f} | "
                f"message={message_grad:.6f} | "
                f"GRU={gru_grad:.6f} | "
                f"total={total_grad_value:.6f}"
            )

            print(
                f"  slots="
                f"{train_memory.index.size:,} | "
                f"window={window_runtime:.2f}s | "
                f"RAM libre="
                f"{ram_available:.2f} GB"
            )

        # Checkpoint périodique
        should_checkpoint = (
            completed_step
            % checkpoint_every == 0
            or completed_step == 334
        )

        if should_checkpoint:

            save_training_checkpoint(
                path=checkpoint_path,
                model=train_model,
                optimizer=train_optimizer,
                temporal_memory=train_memory,
                completed_step=completed_step,
                history=train_history,
                config=train_config,
            )

            save_history_json(
                history_path,
                train_history,
            )

            print(
                f"  checkpoint sauvegardé "
                f"au step {completed_step}"
            )

        # Libérer les références temporaires
        del batch
        del logits
        del y
        del loss
        del loss_sum

        if completed_step % 20 == 0:
            gc.collect()

print()
print(
    "Dernier step terminé :",
    completed_step,
)
print(
    "Fenêtres TBPTT :",
    len(train_history),
)
print(
    "Runtime session :",
    f"{time.perf_counter() - training_start_time:.2f} s",
)
print(
    "RAM disponible :",
    f"{psutil.virtual_memory().available / 1024**3:.2f} GB",
)

Step  10/334 | tx=73,619 | fraudes=30 | loss=1.091021
  grad predictor=0.796957 | message=0.003237 | GRU=0.005055 | total=0.797710
  slots=55,434 | window=3.38s | RAM libre=2.99 GB
Step  20/334 | tx=91,977 | fraudes=15 | loss=0.881550
  grad predictor=1.330084 | message=0.002085 | GRU=0.003060 | total=1.330811
  slots=228,348 | window=5.48s | RAM libre=3.10 GB
  checkpoint sauvegardé au step 20
Step  30/334 | tx=12 | fraudes=12 | loss=864.872742
  grad predictor=1095.768742 | message=0.000000 | GRU=0.000000 | total=1099.022095
  slots=250,727 | window=0.01s | RAM libre=3.10 GB
Step  40/334 | tx=57,661 | fraudes=26 | loss=1.024242
  grad predictor=0.763460 | message=0.001159 | GRU=0.001582 | total=0.764057
  slots=350,532 | window=2.17s | RAM libre=3.09 GB
  checkpoint sauvegardé au step 40
Step  50/334 | tx=770 | fraudes=10 | loss=10.602913
  grad predictor=22.268636 | message=0.000000 | GRU=0.000000 | total=22.289814
  slots=439,795 | window=0.03s | RAM libre=3.06 GB
Step  60/334 | tx

In [51]:
# Résumé de l'historique
history_tx = sum(
    item["transactions"]
    for item in train_history
)

history_frauds = sum(
    item["frauds"]
    for item in train_history
)

losses = np.array(
    [
        item["loss"]
        for item in train_history
    ],
    dtype=np.float64,
)

message_grads = np.array(
    [
        item["message_grad"]
        for item in train_history
    ],
    dtype=np.float64,
)

gru_grads = np.array(
    [
        item["gru_grad"]
        for item in train_history
    ],
    dtype=np.float64,
)

active_size = (
    train_memory.index.size
)

active_last_update = (
    train_memory.index.last_update[
        :active_size
    ]
)

# Invariants
completed_ok = (
    completed_step == 334
)

transactions_ok = (
    history_tx == 4_771_387
)

frauds_ok = (
    history_frauds == 3_745
)

slots_ok = (
    active_size == 2_035_605
)

losses_finite = bool(
    np.isfinite(losses).all()
)

message_grads_finite = bool(
    np.isfinite(
        message_grads
    ).all()
)

gru_grads_finite = bool(
    np.isfinite(
        gru_grads
    ).all()
)

memory_finite = bool(
    torch.isfinite(
        train_memory.memory[
            :active_size
        ]
    ).all()
)

last_update_valid = bool(
    np.all(
        active_last_update >= 1
    )
    and np.all(
        active_last_update <= 334
    )
)

overlay_empty = (
    len(train_memory.overlay) == 0
)

print("Step 334 atteint :", completed_ok)
print(
    "Transactions exactes :",
    transactions_ok,
    f"({history_tx:,})",
)
print(
    "Fraudes exactes :",
    frauds_ok,
    f"({history_frauds:,})",
)
print(
    "Slots exacts :",
    slots_ok,
    f"({active_size:,})",
)
print(
    "Losses finies :",
    losses_finite,
)
print(
    "Gradients message finis :",
    message_grads_finite,
)
print(
    "Gradients GRU finis :",
    gru_grads_finite,
)
print(
    "Mémoire finie :",
    memory_finite,
)
print(
    "last_update valide :",
    last_update_valid,
)
print(
    "last_update min/max :",
    int(active_last_update.min()),
    "/",
    int(active_last_update.max()),
)
print(
    "Overlay vide :",
    overlay_empty,
)

print()
print(
    "Loss min/médiane/max :",
    f"{losses.min():.6f}",
    "/",
    f"{np.median(losses):.6f}",
    "/",
    f"{losses.max():.6f}",
)

print(
    "Gradient message min/médiane/max :",
    f"{message_grads.min():.8f}",
    "/",
    f"{np.median(message_grads):.8f}",
    "/",
    f"{message_grads.max():.8f}",
)

print(
    "Gradient GRU min/médiane/max :",
    f"{gru_grads.min():.8f}",
    "/",
    f"{np.median(gru_grads):.8f}",
    "/",
    f"{gru_grads.max():.8f}",
)

print()
print(
    "Checkpoint latest existe :",
    checkpoint_path.exists(),
)
print(
    "Historique existe :",
    history_path.exists(),
)

Step 334 atteint : True
Transactions exactes : True (4,771,387)
Fraudes exactes : True (3,745)
Slots exacts : True (2,035,605)
Losses finies : True
Gradients message finis : True
Gradients GRU finis : True
Mémoire finie : True
last_update valide : True
last_update min/max : 1 / 334
Overlay vide : True

Loss min/médiane/max : 0.824423 / 1.227095 / 864.872742
Gradient message min/médiane/max : 0.00000000 / 0.00025379 / 0.04669026
Gradient GRU min/médiane/max : 0.00000000 / 0.00042155 / 0.04483314

Checkpoint latest existe : True
Historique existe : True


## 9.8 Résultats de l'entraînement complet

Le premier candidat Dynamic GNN a été entraîné sur l'intégralité du split Train, en respectant strictement l'ordre temporel des transactions.

Le parcours couvre :

- **334 steps** ;
- **4 771 387 transactions** ;
- **3 745 fraudes** ;
- **167 fenêtres TBPTT** de deux steps.

Les contrôles finaux confirment que :

- le step 334 est atteint ;
- le nombre total de transactions est exact ;
- le nombre total de fraudes est exact ;
- le nombre final de slots est exactement égal aux **2 035 605 destinations uniques du Train** ;
- toutes les losses sont finies ;
- tous les gradients enregistrés sont finis ;
- la mémoire temporelle ne contient aucune valeur non finie ;
- tous les `last_update` appartiennent à l'intervalle $[1,334]$ ;
- aucun overlay différentiable ne subsiste après l'entraînement.

La mémoire finale représente donc bien un état causal :

$$
State_{\leq 334}
$$

construit exclusivement à partir du Train.

### Runtime

L'entraînement complet s'est exécuté sur CPU en :

$$
202.33 \text{ secondes}
$$

soit environ **3 minutes et 22 secondes**.

La RAM disponible à la fin de l'entraînement reste proche de **3,07 GB**.

Ces résultats confirment la décision de conserver l'environnement CPU local pour ce premier candidat.

### Evolution de la loss

Les losses observées sur les 167 fenêtres TBPTT présentent la distribution suivante :

| Statistique | Loss |
| --- | ---: |
| Minimum | 0.824423 |
| Médiane | 1.227095 |
| Maximum | 864.872742 |

Les valeurs maximales apparaissent sur certaines fenêtres contenant très peu de transactions et une proportion exceptionnellement élevée de fraudes.

Par exemple, certaines fenêtres affichées contiennent seulement quelques dizaines de transactions, toutes ou presque frauduleuses.

Dans ce contexte, le `pos_weight` Train très élevé :

$$
pos\_weight \approx 1273.07
$$

amplifie fortement leur contribution à la fonction de coût.

Ces pics ne doivent donc pas être interprétés isolément comme une divergence numérique.

La loss reste finie sur l'ensemble de l'entraînement et revient à des valeurs proches de 1 sur les fenêtres transactionnelles plus représentatives.

### Gradients temporels

Les gradients du module de message présentent :

| Statistique | Norme |
| --- | ---: |
| Minimum | 0.00000000 |
| Médiane | 0.00025379 |
| Maximum | 0.04669026 |

Pour la GRU :

| Statistique | Norme |
| --- | ---: |
| Minimum | 0.00000000 |
| Médiane | 0.00042155 |
| Maximum | 0.04483314 |

Certaines petites fenêtres produisent un gradient temporel nul.

Ce comportement est compatible avec l'architecture : le module de message et la GRU ne reçoivent un signal provenant de la loss que lorsqu'un état produit dans un step de la fenêtre TBPTT influence effectivement une prédiction ultérieure de cette même fenêtre.

Lorsque cette dépendance temporelle n'existe pas, le prédicteur peut recevoir un gradient alors que les composants responsables de la construction des états futurs n'en reçoivent pas.

La médiane strictement positive des gradients temporels montre néanmoins que le mécanisme dynamique est régulièrement entraîné au cours du parcours temporel.

Une quantification précise de ces fenêtres sera effectuée avant de clôturer définitivement cette section.

### Etat temporel final

À la fin du step 334 :

$$
N_{\text{slots}} = 2\,035\,605
$$

avec :

$$
1 \leq last\_update \leq 334
$$

Le nombre de slots correspond exactement au nombre de destinations uniques observées dans le Train.

La mémoire ne contient aucune valeur non finie et l'overlay TBPTT est vide.

Le modèle dispose donc d'un état temporel cohérent pouvant être utilisé comme point de départ causal de la Validation :

$$
State_{\leq334}
\rightarrow
E_{335}
$$

Aucune information de Validation ou de Test n'a été utilisée pendant l'entraînement.

In [52]:
# Diagnostiquer les gradients temporels nuls
message_grads = np.array(
    [item["message_grad"] for item in train_history],
    dtype=np.float64,
)

gru_grads = np.array(
    [item["gru_grad"] for item in train_history],
    dtype=np.float64,
)

window_tx = np.array(
    [item["transactions"] for item in train_history],
    dtype=np.int64,
)

window_frauds = np.array(
    [item["frauds"] for item in train_history],
    dtype=np.int64,
)

message_zero = message_grads <= 1e-12
gru_zero = gru_grads <= 1e-12
both_zero = message_zero & gru_zero

print("Fenêtres TBPTT :", len(train_history))

print(
    "Gradient message nul :",
    int(message_zero.sum()),
    f"({100 * message_zero.mean():.2f} %)",
)

print(
    "Gradient GRU nul :",
    int(gru_zero.sum()),
    f"({100 * gru_zero.mean():.2f} %)",
)

print(
    "Deux gradients nuls :",
    int(both_zero.sum()),
    f"({100 * both_zero.mean():.2f} %)",
)

if both_zero.any():
    print()
    print(
        "Transactions médianes si gradients nuls :",
        int(np.median(window_tx[both_zero])),
    )
    print(
        "Transactions min/max si gradients nuls :",
        int(window_tx[both_zero].min()),
        "/",
        int(window_tx[both_zero].max()),
    )
    print(
        "Fraudes médianes si gradients nuls :",
        int(np.median(window_frauds[both_zero])),
    )

print()
print(
    "Fenêtres avec gradient temporel non nul :",
    int((~both_zero).sum()),
)

Fenêtres TBPTT : 167
Gradient message nul : 67 (40.12 %)
Gradient GRU nul : 67 (40.12 %)
Deux gradients nuls : 67 (40.12 %)

Transactions médianes si gradients nuls : 24
Transactions min/max si gradients nuls : 8 / 3359
Fraudes médianes si gradients nuls : 22

Fenêtres avec gradient temporel non nul : 100


In [53]:
# Sauvegarder l'état final du Train
save_training_checkpoint(
    path=final_checkpoint_path,
    model=train_model,
    optimizer=train_optimizer,
    temporal_memory=train_memory,
    completed_step=334,
    history=train_history,
    config=train_config,
)

# Calculer le SHA-256 sans charger le fichier en RAM
sha256 = hashlib.sha256()

with open(final_checkpoint_path, "rb") as f:
    for block in iter(
        lambda: f.read(8 * 1024 * 1024),
        b"",
    ):
        sha256.update(block)

final_checkpoint_sha256 = sha256.hexdigest()

print(
    "Checkpoint final existe :",
    final_checkpoint_path.exists(),
)

print(
    "Taille checkpoint final :",
    f"{final_checkpoint_path.stat().st_size / 1024**2:.2f} MiB",
)

print(
    "Completed step final :",
    334,
)

print(
    "Historique sauvegardé :",
    history_path.exists(),
)

print(
    "Configuration sauvegardée :",
    config_path.exists(),
)

print(
    "SHA-256 final :",
    final_checkpoint_sha256,
)

Checkpoint final existe : True
Taille checkpoint final : 140.00 MiB
Completed step final : 334
Historique sauvegardé : True
Configuration sauvegardée : True
SHA-256 final : 5fa3b6d0a5dc0c189b1714df5da9c26753871f4ce34e586d6c868300e9be1852


# 10. Évaluation causale sur la Validation

## 10.1 Objectif

Le premier candidat Dynamic GNN a été entraîné exclusivement sur les steps 1 à 334.

La Validation couvre :

$$
335 \leq t \leq 392
$$

avec :

$$
792\,406
$$

transactions et :

$$
638
$$

fraudes.

Cette phase remplit deux fonctions distinctes :

1. mesurer les performances du candidat dynamique sur des événements futurs ;
2. effectuer les décisions expérimentales autorisées, notamment le choix du seuil de classification et, ultérieurement, la comparaison éventuelle de variantes du modèle.

Le Test reste totalement exclu.

## 10.2 Poids figés et mémoire dynamique

Pendant la Validation, les paramètres appris sont complètement figés.

Aucune opération suivante n'est autorisée :

- `backward()` ;
- `optimizer.step()` ;
- modification des paramètres du prédicteur ;
- modification des paramètres du module de message ;
- modification des paramètres de la GRU.

En revanche, l'état temporel doit continuer à évoluer.

Pour le premier step de Validation :

$$
State_{\leq334}
\rightarrow
Prediction(E_{335})
\rightarrow
Update(E_{335})
\rightarrow
State_{\leq335}
$$

puis :

$$
State_{\leq335}
\rightarrow
Prediction(E_{336})
\rightarrow
\cdots
$$

jusqu'au step 392.

Cette évolution utilise uniquement les transactions déjà observées et jamais leurs labels de fraude.

## 10.3 Causalité intra-step

La règle validée précédemment reste inchangée.

Pour tout step $t$ de Validation :

$$
State_{<t}
\rightarrow
Prediction(E_t)
\rightarrow
Messages(E_t)
\rightarrow
Aggregation(E_t)
\rightarrow
GRU
\rightarrow
State_{\leq t}
$$

Toutes les transactions de $E_t$ sont donc prédites avant l'intégration du step courant.

## 10.4 Etat initial de la Validation

La Validation doit obligatoirement démarrer à partir du checkpoint final du Train :

$$
State_{\leq334}
$$

et non d'une mémoire réinitialisée.

Cette continuité est nécessaire pour représenter correctement les comptes déjà observés avant le step 335.

Avant l'évaluation complète, l'intégrité de cette frontière temporelle et le gel des paramètres seront vérifiés.

In [54]:
import copy
import psutil

# Modèle Validation
val_model = PilotDynamicModel(
    edge_dim=13,
    memory_dim=8,
    hidden_dim=32,
)

# Optimiseur uniquement nécessaire au loader existant
val_loader_optimizer = torch.optim.AdamW(
    val_model.parameters(),
    lr=1e-3,
    weight_decay=1e-5,
)

val_restored = load_training_checkpoint(
    path=final_checkpoint_path,
    model=val_model,
    optimizer=val_loader_optimizer,
    memory_dim=8,
)

val_memory = val_restored[
    "temporal_memory"
]

# L'optimiseur n'est plus utilisé
del val_loader_optimizer

# Mode évaluation
val_model.eval()

for parameter in val_model.parameters():
    parameter.requires_grad_(False)

print(
    "Completed step chargé :",
    val_restored["completed_step"],
)

print(
    "Slots au début Validation :",
    f"{val_memory.index.size:,}",
)

print(
    "Overlay vide :",
    len(val_memory.overlay) == 0,
)

print(
    "Paramètres entraînables :",
    sum(
        p.requires_grad
        for p in val_model.parameters()
    ),
)

print(
    "RAM disponible :",
    f"{psutil.virtual_memory().available / 1024**3:.2f} GB",
)

Completed step chargé : 334
Slots au début Validation : 2,035,605
Overlay vide : True
Paramètres entraînables : 0
RAM disponible : 2.33 GB


In [55]:
# Etat temporel avant le step 335
active_size = val_memory.index.size

active_last_update = (
    val_memory.index.last_update[
        :active_size
    ]
)

boundary_ok = bool(
    active_size == 2_035_605
    and active_last_update.min() >= 1
    and active_last_update.max() == 334
)

# Vérifier les offsets
train_end = int(offsets[334])
val_start = int(offsets[334])
val_end = int(offsets[392])

print(
    "Etat Train final correct :",
    boundary_ok,
)

print(
    "Dernier last_update :",
    int(active_last_update.max()),
)

print(
    "Fin Train offset :",
    f"{train_end:,}",
)

print(
    "Début Validation offset :",
    f"{val_start:,}",
)

print(
    "Fin Validation offset :",
    f"{val_end:,}",
)

print(
    "Transactions Validation :",
    f"{val_end - val_start:,}",
)

print(
    "Frontière Train/Val exacte :",
    train_end == val_start,
)

Etat Train final correct : True
Dernier last_update : 334
Fin Train offset : 4,771,387
Début Validation offset : 4,771,387
Fin Validation offset : 5,563,793
Transactions Validation : 792,406
Frontière Train/Val exacte : True


In [56]:
# Copier les paramètres avant le step 335
params_before = {
    name: parameter.detach().clone()
    for name, parameter
    in val_model.named_parameters()
}

batch_335 = reader.read_step(335)

with torch.no_grad():
    logits_335, y_335, info_335 = (
        process_pilot_step(
            model=val_model,
            temporal_memory=val_memory,
            batch=batch_335,
        )
    )

# Validation : on commit après le step
val_memory.commit()

params_unchanged = all(
    torch.equal(
        params_before[name],
        parameter.detach(),
    )
    for name, parameter
    in val_model.named_parameters()
)

scores_335 = torch.sigmoid(
    logits_335
).cpu().numpy()

print("Step évalué : 335")
print(
    "Transactions :",
    f"{info_335['transactions']:,}",
)
print(
    "Fraudes :",
    info_335["frauds"],
)
print(
    "Sources connues :",
    f"{info_335['known_src']:,}",
)
print(
    "Destinations connues :",
    f"{info_335['known_dst']:,}",
)
print(
    "Scores finis :",
    bool(np.isfinite(scores_335).all()),
)
print(
    "Scores min/max :",
    f"{scores_335.min():.6f}",
    "/",
    f"{scores_335.max():.6f}",
)
print(
    "Paramètres inchangés :",
    params_unchanged,
)
print(
    "Overlay vide après commit :",
    len(val_memory.overlay) == 0,
)
print(
    "Slots après step 335 :",
    f"{val_memory.index.size:,}",
)
print(
    "last_update max :",
    int(
        val_memory.index.last_update[
            :val_memory.index.size
        ].max()
    ),
)

Step évalué : 335
Transactions : 10,456
Fraudes : 4
Sources connues : 2
Destinations connues : 4,386
Scores finis : True
Scores min/max : 0.115165 / 0.999013
Paramètres inchangés : True
Overlay vide après commit : True
Slots après step 335 : 2,041,675
last_update max : 335


## 10.5 Validation de la frontière Train → Validation

Le checkpoint final du Train a été restauré avant toute évaluation.

L'état chargé correspond exactement à :

$$
State_{\leq334}
$$

avec :

- **2 035 605 slots actifs** ;
- `last_update` maximal égal à **334** ;
- aucun overlay TBPTT résiduel ;
- aucun paramètre entraînable.

La frontière physique entre les splits est également exacte :

$$
offset_{334}
=
4\,771\,387
$$

et la Validation commence exactement à cette même position.

La Validation contient :

$$
5\,563\,793 - 4\,771\,387
=
792\,406
$$

transactions.

Le sanity check effectué sur le step 335 confirme que :

- les scores produits sont tous finis ;
- les paramètres du modèle restent strictement inchangés ;
- la mémoire temporelle évolue après la prédiction ;
- `last_update` atteint 335 uniquement après l'intégration du step ;
- l'overlay est vidé après la mise à jour.

Au step 335, seulement **2 sources** disposent d'un état historique, contre **4 386 destinations**.

Cette asymétrie confirme une nouvelle fois que la principale information temporelle réutilisable de PaySim se situe du côté des destinations.

Le protocole d'évaluation causale est donc validé.

La Validation complète peut maintenant être exécutée avec des poids entièrement figés.

In [57]:
# Recharger State_<=334 avant la vraie Validation
val_model = PilotDynamicModel(
    edge_dim=13,
    memory_dim=8,
    hidden_dim=32,
)

val_loader_optimizer = torch.optim.AdamW(
    val_model.parameters(),
    lr=1e-3,
    weight_decay=1e-5,
)

val_restored = load_training_checkpoint(
    path=final_checkpoint_path,
    model=val_model,
    optimizer=val_loader_optimizer,
    memory_dim=8,
)

val_memory = val_restored["temporal_memory"]

del val_loader_optimizer

# Figer définitivement les poids
val_model.eval()

for parameter in val_model.parameters():
    parameter.requires_grad_(False)

print(
    "Completed step :",
    val_restored["completed_step"],
)

print(
    "Slots initiaux :",
    f"{val_memory.index.size:,}",
)

print(
    "last_update max :",
    int(
        val_memory.index.last_update[
            :val_memory.index.size
        ].max()
    ),
)

print(
    "Paramètres entraînables :",
    sum(
        p.requires_grad
        for p in val_model.parameters()
    ),
)

Completed step : 334
Slots initiaux : 2,035,605
last_update max : 334
Paramètres entraînables : 0


In [58]:
import time
import psutil

val_scores_blocks = []
val_labels_blocks = []
val_steps_blocks = []
val_type_blocks = []
val_src_known_blocks = []
val_dst_known_blocks = []

val_runtime_start = time.perf_counter()

val_total_tx = 0
val_total_fraud = 0
val_total_known_src = 0
val_total_known_dst = 0

# Snapshot des poids avant Validation
val_params_before = {
    name: parameter.detach().clone()
    for name, parameter
    in val_model.named_parameters()
}

for t in range(335, 393):

    step_start_time = time.perf_counter()

    batch = reader.read_step(t)

    # Known/New strictement avant E_t
    _, src_known, _ = val_memory.index.lookup(
        batch["src"],
        t=t,
    )

    _, dst_known, _ = val_memory.index.lookup(
        batch["dst"],
        t=t,
    )

    with torch.no_grad():

        logits, y, info = process_pilot_step(
            model=val_model,
            temporal_memory=val_memory,
            batch=batch,
        )

        scores = torch.sigmoid(
            logits
        ).cpu().numpy().astype(
            np.float32,
            copy=False,
        )

    # Commit seulement après toutes les prédictions
    val_memory.commit()

    # Type depuis le one-hot
    type_idx = np.argmax(
        batch["x"][:, 2:7],
        axis=1,
    ).astype(np.int8)

    # Sauvegarder les sorties
    val_scores_blocks.append(scores)

    val_labels_blocks.append(
        batch["label"].astype(
            np.int8,
            copy=True,
        )
    )

    val_steps_blocks.append(
        np.full(
            len(scores),
            t,
            dtype=np.int16,
        )
    )

    val_type_blocks.append(type_idx)

    val_src_known_blocks.append(
        src_known.astype(
            np.bool_,
            copy=True,
        )
    )

    val_dst_known_blocks.append(
        dst_known.astype(
            np.bool_,
            copy=True,
        )
    )

    val_total_tx += len(scores)

    val_total_fraud += int(
        batch["label"].sum()
    )

    val_total_known_src += int(
        src_known.sum()
    )

    val_total_known_dst += int(
        dst_known.sum()
    )

    step_runtime = (
        time.perf_counter()
        - step_start_time
    )

    if (
        t == 335
        or t % 10 == 0
        or t == 392
    ):
        print(
            f"Step {t}/392 | "
            f"tx={len(scores):,} | "
            f"fraudes={int(batch['label'].sum())} | "
            f"known_src={int(src_known.sum()):,} | "
            f"known_dst={int(dst_known.sum()):,} | "
            f"runtime={step_runtime:.2f}s | "
            f"RAM libre="
            f"{psutil.virtual_memory().available / 1024**3:.2f} GB"
        )

print()
print(
    "Transactions Validation :",
    f"{val_total_tx:,}",
)

print(
    "Fraudes Validation :",
    f"{val_total_fraud:,}",
)

print(
    "Sources connues :",
    f"{val_total_known_src:,}",
)

print(
    "Destinations connues :",
    f"{val_total_known_dst:,}",
)

print(
    "Runtime Validation :",
    f"{time.perf_counter() - val_runtime_start:.2f} s",
)

Step 335/392 | tx=10,456 | fraudes=4 | known_src=2 | known_dst=4,386 | runtime=1.47s | RAM libre=2.83 GB
Step 340/392 | tx=10 | fraudes=10 | known_src=0 | known_dst=3 | runtime=0.00s | RAM libre=2.83 GB
Step 350/392 | tx=28,245 | fraudes=14 | known_src=6 | known_dst=16,592 | runtime=1.38s | RAM libre=2.95 GB
Step 360/392 | tx=3,508 | fraudes=8 | known_src=1 | known_dst=1,453 | runtime=0.08s | RAM libre=2.94 GB
Step 370/392 | tx=25,429 | fraudes=18 | known_src=9 | known_dst=14,831 | runtime=0.91s | RAM libre=2.97 GB
Step 380/392 | tx=37,693 | fraudes=14 | known_src=12 | known_dst=21,142 | runtime=1.18s | RAM libre=2.94 GB
Step 390/392 | tx=12 | fraudes=12 | known_src=0 | known_dst=3 | runtime=0.00s | RAM libre=2.94 GB
Step 392/392 | tx=10 | fraudes=10 | known_src=0 | known_dst=4 | runtime=0.00s | RAM libre=2.94 GB

Transactions Validation : 792,406
Fraudes Validation : 638
Sources connues : 184
Destinations connues : 451,215
Runtime Validation : 25.28 s


In [59]:
# Consolider les blocs
val_scores = np.concatenate(
    val_scores_blocks
)

val_labels = np.concatenate(
    val_labels_blocks
)

val_steps_pred = np.concatenate(
    val_steps_blocks
)

val_type_idx = np.concatenate(
    val_type_blocks
)

val_src_known = np.concatenate(
    val_src_known_blocks
)

val_dst_known = np.concatenate(
    val_dst_known_blocks
)

# Vérifier que les poids sont toujours identiques
val_params_unchanged = all(
    torch.equal(
        val_params_before[name],
        parameter.detach(),
    )
    for name, parameter
    in val_model.named_parameters()
)

# Invariants
val_tx_ok = (
    len(val_scores) == 792_406
)

val_fraud_ok = (
    int(val_labels.sum()) == 638
)

val_steps_ok = (
    int(val_steps_pred.min()) == 335
    and int(val_steps_pred.max()) == 392
)

val_scores_finite = bool(
    np.isfinite(
        val_scores
    ).all()
)

val_memory_finite = bool(
    torch.isfinite(
        val_memory.memory[
            :val_memory.index.size
        ]
    ).all()
)

val_overlay_empty = (
    len(val_memory.overlay) == 0
)

val_last_update_ok = (
    int(
        val_memory.index.last_update[
            :val_memory.index.size
        ].max()
    )
    == 392
)

print(
    "Transactions exactes :",
    val_tx_ok,
    f"({len(val_scores):,})",
)

print(
    "Fraudes exactes :",
    val_fraud_ok,
    f"({int(val_labels.sum()):,})",
)

print(
    "Steps exacts :",
    val_steps_ok,
    f"({val_steps_pred.min()}-{val_steps_pred.max()})",
)

print(
    "Scores finis :",
    val_scores_finite,
)

print(
    "Paramètres inchangés :",
    val_params_unchanged,
)

print(
    "Mémoire finie :",
    val_memory_finite,
)

print(
    "Overlay vide :",
    val_overlay_empty,
)

print(
    "last_update max = 392 :",
    val_last_update_ok,
)

print(
    "Slots après Validation :",
    f"{val_memory.index.size:,}",
)

print()
print(
    "Taux source Known :",
    f"{100 * val_src_known.mean():.4f} %",
)

print(
    "Taux destination Known :",
    f"{100 * val_dst_known.mean():.4f} %",
)

print(
    "Scores min/moyenne/max :",
    f"{val_scores.min():.6f}",
    "/",
    f"{val_scores.mean():.6f}",
    "/",
    f"{val_scores.max():.6f}",
)

Transactions exactes : True (792,406)
Fraudes exactes : True (638)
Steps exacts : True (335-392)
Scores finis : True
Paramètres inchangés : True
Mémoire finie : True
Overlay vide : True
last_update max = 392 : True
Slots après Validation : 2,375,723

Taux source Known : 0.0232 %
Taux destination Known : 56.9424 %
Scores min/moyenne/max : 0.040775 / 0.520913 / 1.000000


In [60]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

val_pr_auc = average_precision_score(
    val_labels,
    val_scores,
)

val_roc_auc = roc_auc_score(
    val_labels,
    val_scores,
)

print(
    "Validation PR-AUC :",
    f"{val_pr_auc:.6f}",
)

print(
    "Validation ROC-AUC :",
    f"{val_roc_auc:.6f}",
)

print(
    "Fraud rate Validation :",
    f"{100 * val_labels.mean():.6f} %",
)

Validation PR-AUC : 0.547378
Validation ROC-AUC : 0.980523
Fraud rate Validation : 0.080514 %


In [61]:
val_predictions_path = (
    dynamic_artifact_dir
    / "dynamic_core_edge_validation_predictions.npz"
)

val_metrics_path = (
    dynamic_artifact_dir
    / "dynamic_core_edge_validation_metrics.json"
)

# Sauvegarder les prédictions
np.savez_compressed(
    val_predictions_path,
    scores=val_scores,
    labels=val_labels,
    steps=val_steps_pred,
    type_idx=val_type_idx,
    src_known=val_src_known,
    dst_known=val_dst_known,
)

val_metrics = {
    "split": "validation",
    "start_step": 335,
    "end_step": 392,
    "transactions": int(
        len(val_scores)
    ),
    "frauds": int(
        val_labels.sum()
    ),
    "pr_auc": float(
        val_pr_auc
    ),
    "roc_auc": float(
        val_roc_auc
    ),
    "source_known_rate": float(
        val_src_known.mean()
    ),
    "destination_known_rate": float(
        val_dst_known.mean()
    ),
    "weights_frozen": bool(
        val_params_unchanged
    ),
    "train_checkpoint_sha256":
        final_checkpoint_sha256,
}

with open(
    val_metrics_path,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        val_metrics,
        f,
        indent=2,
        ensure_ascii=False,
    )

print(
    "Prédictions sauvegardées :",
    val_predictions_path.exists(),
)

print(
    "Métriques sauvegardées :",
    val_metrics_path.exists(),
)

print(
    "Taille prédictions :",
    f"{val_predictions_path.stat().st_size / 1024**2:.2f} MiB",
)

Prédictions sauvegardées : True
Métriques sauvegardées : True
Taille prédictions : 2.92 MiB


## 10.6 Résultats globaux sur la Validation

L'évaluation causale complète a été réalisée sur les steps 335 à 392.

Elle couvre exactement :

- **792 406 transactions** ;
- **638 fraudes**.

Tous les invariants du protocole sont satisfaits :

- les scores sont finis ;
- les paramètres du modèle sont restés strictement inchangés ;
- la mémoire temporelle reste finie ;
- l'overlay est vide après chaque mise à jour ;
- `last_update` atteint exactement le step 392 ;
- aucun événement du Test n'a été utilisé.

À la fin de la Validation, la mémoire contient :

$$
2\,375\,723
$$

slots temporels actifs.

### Régime inductif observé

Sur l'ensemble de la Validation :

- taux de sources connues : **0,0232 %** ;
- taux de destinations connues : **56,9424 %**.

La quasi-totalité des sources reste donc en situation de cold-start, tandis qu'une majorité des transactions implique une destination déjà observée.

Cette structure confirme que l'intérêt potentiel du modèle dynamique repose principalement sur l'exploitation de l'historique des destinations.

### Performances de ranking

Le premier candidat Dynamic GNN obtient :

$$
PR\text{-}AUC_{Val} = \mathbf{0.547378}
$$

et :

$$
ROC\text{-}AUC_{Val} = \mathbf{0.980523}
$$

Le taux de fraude de la Validation est seulement :

$$
0.080514\%
$$

La PR-AUC est donc très largement supérieure au niveau de base correspondant à la prévalence de la classe positive.

### Comparaison avec le GraphSAGE statique

Le GraphSAGE Core Edge du notebook 06 obtenait :

$$
PR\text{-}AUC_{Val}^{GraphSAGE}
=
0.240136
$$

Le Dynamic GNN obtient :

$$
PR\text{-}AUC_{Val}^{Dynamic}
=
0.547378
$$

soit un gain absolu de :

$$
0.547378 - 0.240136
=
\mathbf{0.307242}
$$

La qualité du ranking est donc plus que doublée par rapport au GraphSAGE statique utilisant le même socle de features transactionnelles.

Ce résultat constitue une première indication forte que la représentation temporelle explicite est mieux adaptée à la structure de PaySim que le simple message passing spatial testé précédemment.

### Comparaison avec LightGBM

Le LightGBM Core de référence obtenait :

$$
PR\text{-}AUC_{Val}^{LightGBM}
\approx
0.929704
$$

Le Dynamic GNN reste donc nettement en dessous de cette baseline tabulaire.

Cette comparaison doit cependant être interprétée avec prudence : le LightGBM Core utilise un ensemble de features tabulaires plus riche, tandis que le candidat dynamique actuel utilise volontairement uniquement les Graph Core Edge afin d'isoler la valeur ajoutée de la mémoire temporelle apprise.

### Calibration des scores

Les scores Validation sont compris entre :

$$
0.040775
$$

et :

$$
1.000000
$$

avec une moyenne de :

$$
0.520913
$$

Cette moyenne est très élevée au regard du taux réel de fraude.

Le modèle n'est donc pas calibré comme une probabilité de fraude directement interprétable.

Ce comportement est cohérent avec l'utilisation d'une `BCEWithLogitsLoss` fortement pondérée par :

$$
pos\_weight \approx 1273.07
$$

La valeur `0.5` ne sera donc pas utilisée comme seuil par défaut.

Le seuil de classification sera sélectionné exclusivement sur la Validation après l'analyse des performances par type de transaction et par régime Known/New.

In [62]:
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


def compute_ranking_metrics(labels, scores):
    labels = np.asarray(labels)
    scores = np.asarray(scores)

    n = len(labels)
    frauds = int(labels.sum())

    # AP et ROC nécessitent les deux classes
    if frauds == 0 or frauds == n:
        pr_auc = np.nan
        roc_auc = np.nan
    else:
        pr_auc = average_precision_score(
            labels,
            scores,
        )

        roc_auc = roc_auc_score(
            labels,
            scores,
        )

    return {
        "transactions": n,
        "frauds": frauds,
        "fraud_rate_pct": (
            100 * frauds / n
        ),
        "pr_auc": pr_auc,
        "roc_auc": roc_auc,
        "score_mean": float(
            scores.mean()
        ),
    }


# Décoder les types
type_names = np.array(
    type_categories
)

val_type_name = type_names[
    val_type_idx
]

type_rows = []

for tx_type in type_categories:

    mask = (
        val_type_name == tx_type
    )

    metrics = compute_ranking_metrics(
        val_labels[mask],
        val_scores[mask],
    )

    type_rows.append({
        "type": tx_type,
        **metrics,
    })

type_analysis = pd.DataFrame(
    type_rows
)

print(
    type_analysis.to_string(
        index=False,
        formatters={
            "fraud_rate_pct":
                lambda x: f"{x:.6f}",
            "pr_auc":
                lambda x: (
                    "NA"
                    if np.isnan(x)
                    else f"{x:.6f}"
                ),
            "roc_auc":
                lambda x: (
                    "NA"
                    if np.isnan(x)
                    else f"{x:.6f}"
                ),
            "score_mean":
                lambda x: f"{x:.6f}",
        },
    )
)

    type  transactions  frauds fraud_rate_pct   pr_auc  roc_auc score_mean
 CASH_IN        174986       0       0.000000      NaN      NaN   0.328679
CASH_OUT        274426     319       0.116243 0.524411 0.927485   0.632805
   DEBIT          5989       0       0.000000      NaN      NaN   0.423833
 PAYMENT        270349       0       0.000000      NaN      NaN   0.510266
TRANSFER         66656     319       0.478577 0.592375 0.984695   0.616812


In [63]:
# Construire les catégories cold-start
cold_category = np.full(
    len(val_scores),
    "",
    dtype="<U3",
)

cold_category[
    val_src_known
    & val_dst_known
] = "K→K"

cold_category[
    ~val_src_known
    & val_dst_known
] = "N→K"

cold_category[
    val_src_known
    & ~val_dst_known
] = "K→N"

cold_category[
    ~val_src_known
    & ~val_dst_known
] = "N→N"


cold_rows = []

for category in [
    "K→K",
    "N→K",
    "K→N",
    "N→N",
]:

    mask = (
        cold_category == category
    )

    metrics = compute_ranking_metrics(
        val_labels[mask],
        val_scores[mask],
    )

    cold_rows.append({
        "category": category,
        **metrics,
    })


cold_analysis = pd.DataFrame(
    cold_rows
)

print(
    cold_analysis.to_string(
        index=False,
        formatters={
            "fraud_rate_pct":
                lambda x: f"{x:.6f}",
            "pr_auc":
                lambda x: (
                    "NA"
                    if np.isnan(x)
                    else f"{x:.6f}"
                ),
            "roc_auc":
                lambda x: (
                    "NA"
                    if np.isnan(x)
                    else f"{x:.6f}"
                ),
            "score_mean":
                lambda x: f"{x:.6f}",
        },
    )
)

category  transactions  frauds fraud_rate_pct   pr_auc  roc_auc score_mean
     K→K           104       0       0.000000      NaN      NaN   0.358065
     N→K        451111     241       0.053424 0.420695 0.961375   0.497943
     K→N            80       0       0.000000      NaN      NaN   0.412223
     N→N        341111     397       0.116384 0.634929 0.991849   0.551366


## 10.7 Analyse par type de transaction

Les 638 fraudes de la Validation sont réparties exactement entre les deux types frauduleux de PaySim :

- **319 CASH_OUT** ;
- **319 TRANSFER**.

Les performances obtenues sont :

| Type | Transactions | Fraudes | PR-AUC | ROC-AUC |
| --- | ---: | ---: | ---: | ---: |
| CASH_OUT | 274 426 | 319 | 0.524411 | 0.927485 |
| TRANSFER | 66 656 | 319 | 0.592375 | 0.984695 |

Les autres types ne contiennent aucune fraude et ne permettent donc pas de calculer une PR-AUC ou une ROC-AUC interprétable.

Le modèle obtient un meilleur ranking sur les transactions `TRANSFER` que sur les `CASH_OUT`.

Cependant, la PR-AUC de **0.524411 sur CASH_OUT** reste particulièrement intéressante, car ce type constituait la principale source d'erreurs du modèle tabulaire Core et du GraphSAGE statique analysés précédemment.

Le Dynamic GNN ne résout pas entièrement cette difficulté, mais il produit un signal discriminant substantiel sur les deux mécanismes de fraude présents dans PaySim.

## 10.8 Analyse du régime Known/New

La Validation présente une structure fortement inductive :

| Régime | Transactions | Fraudes | PR-AUC | ROC-AUC |
| --- | ---: | ---: | ---: | ---: |
| K→K | 104 | 0 | — | — |
| N→K | 451 111 | 241 | 0.420695 | 0.961375 |
| K→N | 80 | 0 | — | — |
| N→N | 341 111 | 397 | **0.634929** | **0.991849** |

Les catégories impliquant une source connue sont pratiquement inexistantes et ne contiennent aucune fraude sur la Validation.

L'analyse se concentre donc sur :

$$
N \rightarrow K
$$

et :

$$
N \rightarrow N
$$

### Régime N→K

Le régime `N→K` représente :

$$
451\,111
$$

transactions et **241 fraudes**.

Il obtient :

$$
PR\text{-}AUC_{N\rightarrow K}
=
0.420695
$$

La destination dispose ici d'une mémoire historique, tandis que la source est nouvelle.

Ce régime correspond précisément au scénario pour lequel la mémoire destination-centric avait été conçue.

### Régime N→N

Le régime `N→N` contient :

$$
341\,111
$$

transactions et **397 fraudes**.

Il obtient :

$$
PR\text{-}AUC_{N\rightarrow N}
=
\mathbf{0.634929}
$$

Ce résultat est supérieur à celui du régime `N→K`.

Or, dans une transaction `N→N`, ni la source ni la destination ne disposent d'un état historique appris avant le step.

Les deux comptes utilisent donc l'état initial partagé.

La performance élevée de ce régime ne peut pas être attribuée directement à l'exploitation d'un historique de nœud.

### Interprétation

Ce résultat nuance l'hypothèse formulée lors de la conception de l'architecture.

Le gain global du Dynamic GNN par rapport au GraphSAGE statique ne peut pas être attribué uniquement à la mémoire temporelle des destinations récurrentes.

Le premier candidat combine en réalité plusieurs éléments :

- un prédicteur neuronal directement appliqué aux Graph Core Edge ;
- les indicateurs Known/New ;
- l'encodage de $\Delta t$ ;
- la mémoire temporelle lorsqu'elle existe ;
- une dynamique d'entraînement différente du GraphSAGE statique.

Sur les transactions `N→N`, l'information discriminante provient principalement des features transactionnelles et du prédicteur appris, puisque les états historiques spécifiques aux nœuds sont absents.

À l'inverse, la PR-AUC plus faible de **0.420695** sur `N→K` suggère que la mémoire des destinations n'est pas nécessairement exploitée de manière optimale par ce premier candidat.

Ce résultat ne signifie pas que la mémoire dégrade nécessairement les performances : les régimes `N→K` et `N→N` présentent des prévalences de fraude différentes et ne constituent pas deux populations directement interchangeables.

Il indique cependant qu'une **ablation explicite de la mémoire temporelle** sera nécessaire avant d'attribuer scientifiquement le gain observé à la composante dynamique.

Cette ablation sera réalisée ultérieurement sur la Validation, sans utiliser le Test.

Pour le moment, le premier candidat conserve une performance globale de :

$$
PR\text{-}AUC_{Val}
=
0.547378
$$

et peut passer à la sélection du seuil de classification.

## 10.9 Sélection du seuil de classification

Les scores produits par le Dynamic GNN ne sont pas directement calibrés comme des probabilités de fraude.

La valeur 0.5 ne constitue donc pas un seuil de décision justifié.

Le seuil opérationnel du premier candidat sera sélectionné exclusivement sur la Validation.

Afin de rester cohérent avec l'évaluation des modèles précédents, le critère retenu est la maximisation du F1-score :

$$
F_1
=
2
\frac{Precision \times Recall}
{Precision + Recall}
$$

Pour chaque seuil candidat issu de la courbe Precision-Recall, le F1-score est calculé.

Le seuil donnant le F1 maximal est retenu :

$$
\tau^*
=
\underset{\tau}{\operatorname{argmax}}
\;
F_1(\tau)
$$

Ce seuil sera ensuite figé.

Aucune information du Test ne pourra intervenir dans sa sélection ou sa modification.

Un unique seuil global est utilisé pour toutes les transactions, indépendamment :

- du type de transaction ;
- du statut Known/New ;
- du step temporel.

In [64]:
from sklearn.metrics import precision_recall_curve

# Courbe Precision-Recall
precision_curve, recall_curve, thresholds = (
    precision_recall_curve(
        val_labels,
        val_scores,
    )
)

# Les thresholds ont une valeur de moins
precision_thr = precision_curve[:-1]
recall_thr = recall_curve[:-1]

denominator = (
    precision_thr + recall_thr
)

f1_curve = np.divide(
    2 * precision_thr * recall_thr,
    denominator,
    out=np.zeros_like(denominator),
    where=denominator > 0,
)

best_idx = int(
    np.argmax(f1_curve)
)

val_threshold = float(
    thresholds[best_idx]
)

best_precision = float(
    precision_thr[best_idx]
)

best_recall = float(
    recall_thr[best_idx]
)

best_f1 = float(
    f1_curve[best_idx]
)

print(
    "Seuil Validation :",
    f"{val_threshold:.10f}",
)

print(
    "Precision au seuil :",
    f"{best_precision:.6f}",
)

print(
    "Recall au seuil :",
    f"{best_recall:.6f}",
)

print(
    "F1 au seuil :",
    f"{best_f1:.6f}",
)

Seuil Validation : 0.9800333381
Precision au seuil : 0.730964
Recall au seuil : 0.451411
F1 au seuil : 0.558140


In [65]:
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)

val_pred = (
    val_scores >= val_threshold
).astype(np.int8)

tn, fp, fn, tp = confusion_matrix(
    val_labels,
    val_pred,
    labels=[0, 1],
).ravel()

val_precision = precision_score(
    val_labels,
    val_pred,
    zero_division=0,
)

val_recall = recall_score(
    val_labels,
    val_pred,
    zero_division=0,
)

val_f1 = f1_score(
    val_labels,
    val_pred,
    zero_division=0,
)

print("Validation au seuil sélectionné")
print()

print("TP :", f"{tp:,}")
print("FP :", f"{fp:,}")
print("FN :", f"{fn:,}")
print("TN :", f"{tn:,}")

print()

print(
    "Precision :",
    f"{val_precision:.6f}",
)

print(
    "Recall :",
    f"{val_recall:.6f}",
)

print(
    "F1 :",
    f"{val_f1:.6f}",
)

print(
    "PR-AUC :",
    f"{val_pr_auc:.6f}",
)

print(
    "ROC-AUC :",
    f"{val_roc_auc:.6f}",
)

print(
    "Transactions prédites fraude :",
    f"{int(val_pred.sum()):,}",
)

Validation au seuil sélectionné

TP : 288
FP : 106
FN : 350
TN : 791,662

Precision : 0.730964
Recall : 0.451411
F1 : 0.558140
PR-AUC : 0.547378
ROC-AUC : 0.980523
Transactions prédites fraude : 394


In [66]:
pred_05 = (
    val_scores >= 0.5
).astype(np.int8)

tn05, fp05, fn05, tp05 = confusion_matrix(
    val_labels,
    pred_05,
    labels=[0, 1],
).ravel()

print("Seuil 0.5")
print(
    "Precision :",
    f"{precision_score(val_labels, pred_05, zero_division=0):.6f}",
)
print(
    "Recall :",
    f"{recall_score(val_labels, pred_05, zero_division=0):.6f}",
)
print(
    "F1 :",
    f"{f1_score(val_labels, pred_05, zero_division=0):.6f}",
)
print(
    "TP / FP / FN :",
    f"{tp05:,} / {fp05:,} / {fn05:,}",
)
print(
    "Prédites fraude :",
    f"{int(pred_05.sum()):,}",
)

Seuil 0.5
Precision : 0.002005
Recall : 0.998433
F1 : 0.004002
TP / FP / FN : 637 / 317,041 / 1
Prédites fraude : 317,678


## 10.10 Seuil de classification retenu

Le seuil de classification a été sélectionné exclusivement sur la Validation en maximisant le F1-score.

Le seuil retenu est :

$$
\tau^* = 0.9800333381
$$

À ce seuil, le Dynamic GNN obtient :

| Métrique | Validation |
| --- | ---: |
| Precision | 0.730964 |
| Recall | 0.451411 |
| F1 | 0.558140 |
| TP | 288 |
| FP | 106 |
| FN | 350 |
| TN | 791 662 |

Le modèle prédit **394 transactions frauduleuses** sur les 792 406 transactions de Validation.

### Calibration des scores

Une comparaison informative avec le seuil standard de 0.5 montre :

- Precision : **0.002005** ;
- Recall : **0.998433** ;
- F1 : **0.004002** ;
- TP : **637** ;
- FP : **317 041** ;
- FN : **1**.

À ce seuil, **317 678 transactions** seraient classées comme frauduleuses.

Le modèle produit donc des scores fortement décalés vers les grandes valeurs.

Cette situation est cohérente avec l'utilisation d'une fonction de coût fortement pondérée par :

$$
pos\_weight \approx 1273.07
$$

Les scores ne doivent donc pas être interprétés directement comme des probabilités calibrées de fraude.

### Décision expérimentale

Le seuil :

$$
\boxed{\tau^* = 0.9800333381}
$$

est désormais **figé**.

Il pourra être appliqué ultérieurement au Test sans aucune modification.

Aucune information du Test ne sera utilisée pour le recalculer, l'ajuster ou sélectionner un autre seuil.

Les prochaines analyses de Validation utiliseront également ce même seuil afin d'étudier le comportement du modèle selon :

- le type de transaction ;
- le régime Known/New.

In [67]:
# Analyse au seuil par type
threshold_type_rows = []

for tx_type in type_categories:

    mask = (
        val_type_name == tx_type
    )

    y_true = val_labels[mask]
    y_pred = val_pred[mask]

    tn_t, fp_t, fn_t, tp_t = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    precision_t = precision_score(
        y_true,
        y_pred,
        zero_division=0,
    )

    recall_t = recall_score(
        y_true,
        y_pred,
        zero_division=0,
    )

    f1_t = f1_score(
        y_true,
        y_pred,
        zero_division=0,
    )

    threshold_type_rows.append({
        "type": tx_type,
        "transactions": int(mask.sum()),
        "frauds": int(y_true.sum()),
        "predicted_fraud": int(y_pred.sum()),
        "tp": int(tp_t),
        "fp": int(fp_t),
        "fn": int(fn_t),
        "precision": precision_t,
        "recall": recall_t,
        "f1": f1_t,
    })

threshold_type_analysis = pd.DataFrame(
    threshold_type_rows
)

print(
    threshold_type_analysis.to_string(
        index=False,
        formatters={
            "precision": lambda x: f"{x:.6f}",
            "recall": lambda x: f"{x:.6f}",
            "f1": lambda x: f"{x:.6f}",
        },
    )
)

    type  transactions  frauds  predicted_fraud  tp  fp  fn precision   recall       f1
 CASH_IN        174986       0                0   0   0   0  0.000000 0.000000 0.000000
CASH_OUT        274426     319              124 123   1 196  0.991935 0.385580 0.555305
   DEBIT          5989       0                0   0   0   0  0.000000 0.000000 0.000000
 PAYMENT        270349       0                0   0   0   0  0.000000 0.000000 0.000000
TRANSFER         66656     319              270 165 105 154  0.611111 0.517241 0.560272


In [68]:
# Analyse au seuil par régime cold-start
threshold_cold_rows = []

for category in [
    "K→K",
    "N→K",
    "K→N",
    "N→N",
]:

    mask = (
        cold_category == category
    )

    y_true = val_labels[mask]
    y_pred = val_pred[mask]

    tn_c, fp_c, fn_c, tp_c = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    precision_c = precision_score(
        y_true,
        y_pred,
        zero_division=0,
    )

    recall_c = recall_score(
        y_true,
        y_pred,
        zero_division=0,
    )

    f1_c = f1_score(
        y_true,
        y_pred,
        zero_division=0,
    )

    threshold_cold_rows.append({
        "category": category,
        "transactions": int(mask.sum()),
        "frauds": int(y_true.sum()),
        "predicted_fraud": int(y_pred.sum()),
        "tp": int(tp_c),
        "fp": int(fp_c),
        "fn": int(fn_c),
        "precision": precision_c,
        "recall": recall_c,
        "f1": f1_c,
    })

threshold_cold_analysis = pd.DataFrame(
    threshold_cold_rows
)

print(
    threshold_cold_analysis.to_string(
        index=False,
        formatters={
            "precision": lambda x: f"{x:.6f}",
            "recall": lambda x: f"{x:.6f}",
            "f1": lambda x: f"{x:.6f}",
        },
    )
)

category  transactions  frauds  predicted_fraud  tp  fp  fn precision   recall       f1
     K→K           104       0                0   0   0   0  0.000000 0.000000 0.000000
     N→K        451111     241              144  87  57 154  0.604167 0.360996 0.451948
     K→N            80       0                0   0   0   0  0.000000 0.000000 0.000000
     N→N        341111     397              250 201  49 196  0.804000 0.506297 0.621329


In [69]:
threshold_path = (
    dynamic_artifact_dir
    / "dynamic_core_edge_threshold.json"
)

threshold_metadata = {
    "selection_split": "validation",
    "selection_metric": "f1",
    "threshold": float(val_threshold),
    "precision": float(val_precision),
    "recall": float(val_recall),
    "f1": float(val_f1),
    "pr_auc": float(val_pr_auc),
    "roc_auc": float(val_roc_auc),
    "tp": int(tp),
    "fp": int(fp),
    "fn": int(fn),
    "tn": int(tn),
    "test_used_for_selection": False,
}

with open(
    threshold_path,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        threshold_metadata,
        f,
        indent=2,
        ensure_ascii=False,
    )

print(
    "Seuil sauvegardé :",
    threshold_path.exists(),
)

print(
    "Seuil figé :",
    f"{threshold_metadata['threshold']:.10f}",
)

print(
    "Split de sélection :",
    threshold_metadata["selection_split"],
)

print(
    "Test utilisé :",
    threshold_metadata["test_used_for_selection"],
)

Seuil sauvegardé : True
Seuil figé : 0.9800333381
Split de sélection : validation
Test utilisé : False


## 10.11 Analyse au seuil par type de transaction

Le seuil sélectionné sur la Validation est :

$$
\tau^* = 0.9800333381
$$

Son application aux différents types de transactions montre des comportements distincts.

| Type | Fraudes | TP | FP | FN | Precision | Recall | F1 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| CASH_OUT | 319 | 123 | 1 | 196 | 0.991935 | 0.385580 | 0.555305 |
| TRANSFER | 319 | 165 | 105 | 154 | 0.611111 | 0.517241 | 0.560272 |

Aucune transaction `CASH_IN`, `DEBIT` ou `PAYMENT` n'est classée frauduleuse au seuil retenu.

### CASH_OUT

Sur les 319 fraudes `CASH_OUT`, le modèle en détecte 123.

La précision atteint :

$$
Precision_{CASH\_OUT} = 0.991935
$$

pour un rappel de :

$$
Recall_{CASH\_OUT} = 0.385580
$$

Le modèle est donc particulièrement conservateur sur ce type : lorsqu'il signale un `CASH_OUT`, sa décision est presque toujours correcte, mais une part importante des fraudes reste non détectée.

### TRANSFER

Sur les 319 fraudes `TRANSFER`, le modèle en détecte 165.

Les performances sont :

$$
Precision_{TRANSFER} = 0.611111
$$

$$
Recall_{TRANSFER} = 0.517241
$$

Le rappel est supérieur à celui obtenu sur `CASH_OUT`, mais les faux positifs sont également plus nombreux.

Les deux types obtiennent finalement des F1-scores proches :

$$
F1_{CASH\_OUT} = 0.555305
$$

$$
F1_{TRANSFER} = 0.560272
$$

Le modèle produit donc un niveau de performance opérationnelle relativement équilibré entre les deux mécanismes de fraude, malgré des compromis Precision/Recall différents.

## 10.12 Analyse au seuil par régime Known/New

Les performances selon le statut historique des comptes sont :

| Régime | Fraudes | TP | FP | FN | Precision | Recall | F1 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| N→K | 241 | 87 | 57 | 154 | 0.604167 | 0.360996 | 0.451948 |
| N→N | 397 | 201 | 49 | 196 | 0.804000 | 0.506297 | 0.621329 |

Les catégories `K→K` et `K→N` ne contiennent aucune fraude sur la Validation et sont trop rares pour permettre une analyse pertinente.

Le régime `N→N` obtient les meilleures performances :

$$
Precision_{N\rightarrow N} = 0.804000
$$

$$
Recall_{N\rightarrow N} = 0.506297
$$

$$
F1_{N\rightarrow N} = 0.621329
$$

À l'inverse, le régime `N→K`, pour lequel une mémoire destination est disponible, obtient :

$$
F1_{N\rightarrow K} = 0.451948
$$

Cette différence confirme qu'il serait incorrect d'attribuer directement l'amélioration globale du modèle à la seule mémoire temporelle.

Le régime sans historique spécifique aux deux nœuds reste actuellement le plus performant.

## 10.13 Décision expérimentale

Le seuil a été sauvegardé avec les métadonnées suivantes :

- split de sélection : **Validation** ;
- critère : maximisation du F1-score ;
- seuil : **0.9800333381** ;
- Test utilisé pour la sélection : **False**.

Le seuil est désormais figé.

Cependant, avant toute utilisation du Test, une ablation de la mémoire temporelle est nécessaire.

Cette ablation permettra de distinguer :

1. la performance du prédicteur neuronal utilisant les Graph Core Edge ;
2. la valeur réellement ajoutée par les états temporels, `Known/New` et $\Delta t$.

**Section 10 validée.**

# 11. Ablation de la mémoire temporelle

## 11.1 Motivation

Le premier Dynamic GNN atteint sur la Validation :

$$
PR\text{-}AUC = 0.547378
$$

contre :

$$
PR\text{-}AUC = 0.240136
$$

pour le GraphSAGE statique du notebook 06.

Cependant, l'analyse Known/New révèle un résultat important :

$$
PR\text{-}AUC_{N\rightarrow N} = 0.634929
$$

alors que :

$$
PR\text{-}AUC_{N\rightarrow K} = 0.420695
$$

Le régime `N→N` ne dispose d'aucune mémoire spécifique aux comptes.

Il serait donc incorrect d'attribuer automatiquement le gain global du modèle à la dynamique temporelle.

Une ablation contrôlée est nécessaire.

## 11.2 Contrôle sans mémoire

Le contrôle conserve exactement le même socle transactionnel :

- 13 Graph Core Edge ;
- même normalisation Train-only ;
- même ordre temporel ;
- même fonction de coût ;
- même `pos_weight` ;
- même learning rate ;
- même weight decay ;
- même gradient clipping ;
- même seed ;
- même cadence d'optimisation par fenêtres de deux steps.

Le prédicteur conserve également une entrée de dimension 33 afin de garder la même structure que le prédicteur du modèle dynamique.

Les dimensions temporelles sont cependant neutralisées.

Pour chaque transaction :

$$
z_i =
[x_i,\ m_0,\ m_0,\ 0,\ 0,\ 0,\ 0]
$$

où :

- $x_i$ représente les 13 Graph Core Edge ;
- $m_0$ est un état initial partagé et appris ;
- les deux $\Delta t$ sont nuls ;
- les indicateurs Known/New sont nuls.

Aucune mémoire spécifique aux nœuds n'est créée.

Aucun message temporel n'est calculé.

Aucune GRU n'est utilisée.

Le contrôle représente donc un **prédicteur neuronal sans mémoire temporelle**, entraîné dans des conditions aussi proches que possible du candidat dynamique.

## 11.3 Critère d'interprétation

Après entraînement sur les steps 1 à 334, le contrôle sera évalué sur exactement les mêmes steps de Validation :

$$
335 \leq t \leq 392
$$

La métrique principale reste la PR-AUC.

Trois situations sont possibles :

1. **PR-AUC du contrôle nettement inférieure au Dynamic GNN**  
   La mémoire temporelle apporte une information discriminante supplémentaire.

2. **PR-AUC proche du Dynamic GNN**  
   L'essentiel du gain provient du prédicteur neuronal et des Graph Core Edge.

3. **PR-AUC du contrôle supérieure au Dynamic GNN**  
   La mémoire temporelle actuelle dégrade le ranking et doit être repensée.

Aucun résultat du Test ne sera utilisé dans cette décision.

In [70]:
class NoMemoryControl(nn.Module):
    def __init__(
        self,
        edge_dim=13,
        memory_dim=8,
        hidden_dim=32,
    ):
        super().__init__()

        self.memory_dim = memory_dim

        # Etat partagé appris
        self.initial_state = nn.Parameter(
            torch.zeros(memory_dim)
        )

        # Même prédicteur que le modèle dynamique
        self.predictor = nn.Sequential(
            nn.Linear(
                edge_dim
                + 2 * memory_dim
                + 4,
                hidden_dim,
            ),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(
                hidden_dim,
                1,
            ),
        )

    def forward(self, x):
        n = len(x)

        # Même état partagé pour tous les comptes
        shared_state = (
            self.initial_state
            .unsqueeze(0)
            .expand(n, -1)
        )

        # Aucun signal temporel
        zeros = torch.zeros(
            n,
            4,
            dtype=x.dtype,
            device=x.device,
        )

        model_input = torch.cat(
            [
                x,
                shared_state,
                shared_state,
                zeros,
            ],
            dim=1,
        )

        return self.predictor(
            model_input
        ).squeeze(1)


torch.manual_seed(42)

control_model = NoMemoryControl(
    edge_dim=13,
    memory_dim=8,
    hidden_dim=32,
)

control_params = sum(
    p.numel()
    for p in control_model.parameters()
)

print(
    "Input predictor :",
    33,
)
print(
    "Memory historique :",
    False,
)
print(
    "Known/New :",
    False,
)
print(
    "Delta t :",
    False,
)
print(
    "GRU :",
    False,
)
print(
    "Paramètres entraînables :",
    f"{control_params:,}",
)

Input predictor : 33
Memory historique : False
Known/New : False
Delta t : False
GRU : False
Paramètres entraînables : 1,129


In [71]:
import time
import psutil

torch.manual_seed(42)
np.random.seed(42)

control_model.train()

control_optimizer = torch.optim.AdamW(
    control_model.parameters(),
    lr=1e-3,
    weight_decay=1e-5,
)

control_criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        1273.0686248331108,
        dtype=torch.float32,
    ),
    reduction="sum",
)

control_history = []

control_train_start = time.perf_counter()

for window_start in range(
    1,
    335,
    2,
):
    window_end = min(
        window_start + 1,
        334,
    )

    control_optimizer.zero_grad(
        set_to_none=True
    )

    loss_sum = None
    window_tx = 0
    window_fraud = 0

    window_time = time.perf_counter()

    for t in range(
        window_start,
        window_end + 1,
    ):
        batch = reader.read_step(t)

        x = torch.from_numpy(
            batch["x"]
        )

        y = torch.from_numpy(
            batch["label"].astype(
                np.float32
            )
        )

        logits = control_model(x)

        step_loss = control_criterion(
            logits,
            y,
        )

        if loss_sum is None:
            loss_sum = step_loss
        else:
            loss_sum = (
                loss_sum + step_loss
            )

        window_tx += len(y)
        window_fraud += int(
            y.sum().item()
        )

    loss = (
        loss_sum / window_tx
    )

    if not torch.isfinite(loss):
        raise RuntimeError(
            f"Loss non finie "
            f"{window_start}-{window_end}"
        )

    loss.backward()

    grad_norm = (
        torch.nn.utils.clip_grad_norm_(
            control_model.parameters(),
            max_norm=5.0,
        )
    )

    grad_value = float(
        grad_norm.detach()
    )

    if not np.isfinite(
        grad_value
    ):
        raise RuntimeError(
            f"Gradient non fini "
            f"{window_start}-{window_end}"
        )

    control_optimizer.step()

    runtime = (
        time.perf_counter()
        - window_time
    )

    control_history.append({
        "start_step": window_start,
        "end_step": window_end,
        "transactions": window_tx,
        "frauds": window_fraud,
        "loss": float(
            loss.detach()
        ),
        "grad_norm": grad_value,
        "runtime_seconds": runtime,
    })

    if (
        window_end % 20 == 0
        or window_end == 334
    ):
        print(
            f"Step {window_end:>3}/334 | "
            f"tx={window_tx:,} | "
            f"fraudes={window_fraud} | "
            f"loss={float(loss.detach()):.6f} | "
            f"grad={grad_value:.6f} | "
            f"runtime={runtime:.2f}s | "
            f"RAM libre="
            f"{psutil.virtual_memory().available / 1024**3:.2f} GB"
        )

print()
print(
    "Fenêtres :",
    len(control_history),
)
print(
    "Runtime Train contrôle :",
    f"{time.perf_counter() - control_train_start:.2f} s",
)

Step  20/334 | tx=91,977 | fraudes=15 | loss=0.852761 | grad=0.959784 | runtime=0.46s | RAM libre=3.47 GB
Step  40/334 | tx=57,661 | fraudes=26 | loss=1.045389 | grad=1.152266 | runtime=0.24s | RAM libre=3.45 GB
Step  60/334 | tx=32 | fraudes=32 | loss=621.790771 | grad=1616.147461 | runtime=0.01s | RAM libre=3.44 GB
Step  80/334 | tx=20 | fraudes=20 | loss=583.514465 | grad=1187.439331 | runtime=0.00s | RAM libre=3.45 GB
Step 100/334 | tx=14 | fraudes=14 | loss=385.968842 | grad=856.984863 | runtime=0.01s | RAM libre=3.45 GB
Step 120/334 | tx=8,884 | fraudes=22 | loss=1.672205 | grad=1.665260 | runtime=0.03s | RAM libre=3.45 GB
Step 140/334 | tx=83,066 | fraudes=18 | loss=0.975997 | grad=0.946660 | runtime=0.13s | RAM libre=3.43 GB
Step 160/334 | tx=55,326 | fraudes=36 | loss=1.065011 | grad=0.921107 | runtime=0.09s | RAM libre=3.43 GB
Step 180/334 | tx=62,015 | fraudes=18 | loss=0.992524 | grad=1.110153 | runtime=0.13s | RAM libre=3.43 GB
Step 200/334 | tx=18 | fraudes=18 | loss=149.

In [72]:
import time
import psutil

torch.manual_seed(42)
np.random.seed(42)

control_model.train()

control_optimizer = torch.optim.AdamW(
    control_model.parameters(),
    lr=1e-3,
    weight_decay=1e-5,
)

control_criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        1273.0686248331108,
        dtype=torch.float32,
    ),
    reduction="sum",
)

control_history = []

control_train_start = time.perf_counter()

for window_start in range(
    1,
    335,
    2,
):
    window_end = min(
        window_start + 1,
        334,
    )

    control_optimizer.zero_grad(
        set_to_none=True
    )

    loss_sum = None
    window_tx = 0
    window_fraud = 0

    window_time = time.perf_counter()

    for t in range(
        window_start,
        window_end + 1,
    ):
        batch = reader.read_step(t)

        x = torch.from_numpy(
            batch["x"]
        )

        y = torch.from_numpy(
            batch["label"].astype(
                np.float32
            )
        )

        logits = control_model(x)

        step_loss = control_criterion(
            logits,
            y,
        )

        if loss_sum is None:
            loss_sum = step_loss
        else:
            loss_sum = (
                loss_sum + step_loss
            )

        window_tx += len(y)
        window_fraud += int(
            y.sum().item()
        )

    loss = (
        loss_sum / window_tx
    )

    if not torch.isfinite(loss):
        raise RuntimeError(
            f"Loss non finie "
            f"{window_start}-{window_end}"
        )

    loss.backward()

    grad_norm = (
        torch.nn.utils.clip_grad_norm_(
            control_model.parameters(),
            max_norm=5.0,
        )
    )

    grad_value = float(
        grad_norm.detach()
    )

    if not np.isfinite(
        grad_value
    ):
        raise RuntimeError(
            f"Gradient non fini "
            f"{window_start}-{window_end}"
        )

    control_optimizer.step()

    runtime = (
        time.perf_counter()
        - window_time
    )

    control_history.append({
        "start_step": window_start,
        "end_step": window_end,
        "transactions": window_tx,
        "frauds": window_fraud,
        "loss": float(
            loss.detach()
        ),
        "grad_norm": grad_value,
        "runtime_seconds": runtime,
    })

    if (
        window_end % 20 == 0
        or window_end == 334
    ):
        print(
            f"Step {window_end:>3}/334 | "
            f"tx={window_tx:,} | "
            f"fraudes={window_fraud} | "
            f"loss={float(loss.detach()):.6f} | "
            f"grad={grad_value:.6f} | "
            f"runtime={runtime:.2f}s | "
            f"RAM libre="
            f"{psutil.virtual_memory().available / 1024**3:.2f} GB"
        )

print()
print(
    "Fenêtres :",
    len(control_history),
)
print(
    "Runtime Train contrôle :",
    f"{time.perf_counter() - control_train_start:.2f} s",
)

Step  20/334 | tx=91,977 | fraudes=15 | loss=0.942981 | grad=1.323212 | runtime=0.22s | RAM libre=3.38 GB
Step  40/334 | tx=57,661 | fraudes=26 | loss=0.999655 | grad=1.284643 | runtime=0.13s | RAM libre=3.37 GB
Step  60/334 | tx=32 | fraudes=32 | loss=71.732712 | grad=245.527084 | runtime=0.03s | RAM libre=3.37 GB
Step  80/334 | tx=20 | fraudes=20 | loss=42.738106 | grad=188.674850 | runtime=0.02s | RAM libre=3.37 GB
Step 100/334 | tx=14 | fraudes=14 | loss=26.494205 | grad=123.005630 | runtime=0.02s | RAM libre=3.36 GB
Step 120/334 | tx=8,884 | fraudes=22 | loss=1.166583 | grad=1.919050 | runtime=0.41s | RAM libre=3.43 GB
Step 140/334 | tx=83,066 | fraudes=18 | loss=1.516798 | grad=2.541728 | runtime=0.20s | RAM libre=3.32 GB
Step 160/334 | tx=55,326 | fraudes=36 | loss=1.521350 | grad=2.508330 | runtime=0.10s | RAM libre=3.31 GB
Step 180/334 | tx=62,015 | fraudes=18 | loss=1.410131 | grad=2.339793 | runtime=0.15s | RAM libre=3.40 GB
Step 200/334 | tx=18 | fraudes=18 | loss=8.672682 

In [73]:
control_tx = sum(
    item["transactions"]
    for item in control_history
)

control_frauds = sum(
    item["frauds"]
    for item in control_history
)

control_losses = np.array(
    [
        item["loss"]
        for item in control_history
    ],
    dtype=np.float64,
)

print(
    "Transactions exactes :",
    control_tx == 4_771_387,
    f"({control_tx:,})",
)

print(
    "Fraudes exactes :",
    control_frauds == 3_745,
    f"({control_frauds:,})",
)

print(
    "Losses finies :",
    bool(
        np.isfinite(
            control_losses
        ).all()
    ),
)

print(
    "Loss min/médiane/max :",
    f"{control_losses.min():.6f}",
    "/",
    f"{np.median(control_losses):.6f}",
    "/",
    f"{control_losses.max():.6f}",
)

Transactions exactes : True (4,771,387)
Fraudes exactes : True (3,745)
Losses finies : True
Loss min/médiane/max : 0.633784 / 1.329096 / 94.857002


In [74]:
control_model.eval()

for parameter in control_model.parameters():
    parameter.requires_grad_(False)

control_val_scores_blocks = []
control_val_labels_blocks = []

control_val_start = time.perf_counter()

with torch.no_grad():

    for t in range(
        335,
        393,
    ):
        batch = reader.read_step(t)

        x = torch.from_numpy(
            batch["x"]
        )

        logits = control_model(x)

        scores = torch.sigmoid(
            logits
        ).cpu().numpy().astype(
            np.float32,
            copy=False,
        )

        control_val_scores_blocks.append(
            scores
        )

        control_val_labels_blocks.append(
            batch["label"].astype(
                np.int8,
                copy=True,
            )
        )

control_val_scores = np.concatenate(
    control_val_scores_blocks
)

control_val_labels = np.concatenate(
    control_val_labels_blocks
)

control_val_pr_auc = (
    average_precision_score(
        control_val_labels,
        control_val_scores,
    )
)

control_val_roc_auc = (
    roc_auc_score(
        control_val_labels,
        control_val_scores,
    )
)

print(
    "Transactions Validation :",
    f"{len(control_val_scores):,}",
)

print(
    "Fraudes Validation :",
    f"{int(control_val_labels.sum()):,}",
)

print(
    "Scores finis :",
    bool(
        np.isfinite(
            control_val_scores
        ).all()
    ),
)

print(
    "PR-AUC contrôle :",
    f"{control_val_pr_auc:.6f}",
)

print(
    "ROC-AUC contrôle :",
    f"{control_val_roc_auc:.6f}",
)

print(
    "PR-AUC Dynamic :",
    f"{val_pr_auc:.6f}",
)

print(
    "Delta PR-AUC Dynamic - contrôle :",
    f"{val_pr_auc - control_val_pr_auc:+.6f}",
)

print(
    "Runtime Validation :",
    f"{time.perf_counter() - control_val_start:.2f} s",
)

Transactions Validation : 792,406
Fraudes Validation : 638
Scores finis : True
PR-AUC contrôle : 0.606575
ROC-AUC contrôle : 0.990075
PR-AUC Dynamic : 0.547378
Delta PR-AUC Dynamic - contrôle : -0.059197
Runtime Validation : 5.24 s


In [75]:
control_model_path = (
    dynamic_artifact_dir
    / "no_memory_control_train_final.pt"
)

control_results_path = (
    dynamic_artifact_dir
    / "no_memory_control_validation.json"
)

torch.save(
    {
        "model_state_dict":
            control_model.state_dict(),
        "train_history":
            control_history,
        "config": {
            "feature_set":
                "graph_core_edge",
            "memory": False,
            "known_new": False,
            "delta_t": False,
            "train_steps": [1, 334],
            "validation_steps":
                [335, 392],
            "seed": 42,
            "pos_weight":
                1273.0686248331108,
        },
    },
    control_model_path,
)

control_results = {
    "transactions":
        int(len(control_val_scores)),
    "frauds":
        int(control_val_labels.sum()),
    "pr_auc":
        float(control_val_pr_auc),
    "roc_auc":
        float(control_val_roc_auc),
    "dynamic_pr_auc":
        float(val_pr_auc),
    "delta_pr_auc":
        float(
            val_pr_auc
            - control_val_pr_auc
        ),
    "test_used": False,
}

with open(
    control_results_path,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        control_results,
        f,
        indent=2,
        ensure_ascii=False,
    )

print(
    "Modèle contrôle sauvegardé :",
    control_model_path.exists(),
)

print(
    "Résultats sauvegardés :",
    control_results_path.exists(),
)

print(
    "Test utilisé :",
    control_results["test_used"],
)

Modèle contrôle sauvegardé : True
Résultats sauvegardés : True
Test utilisé : False


## 11.4 Résultats de l'ablation de la mémoire temporelle

Le contrôle sans mémoire a été entraîné sur exactement le même split Train que le modèle dynamique :

$$
1 \leq t \leq 334
$$

avec :

- **4 771 387 transactions** ;
- **3 745 fraudes** ;
- **167 fenêtres d'optimisation** ;
- le même `pos_weight` ;
- le même learning rate ;
- le même weight decay ;
- le même gradient clipping ;
- la même seed ;
- les mêmes 13 Graph Core Edge.

Toutes les losses sont restées finies.

Le contrôle ne possède :

- aucune mémoire historique spécifique aux comptes ;
- aucune GRU ;
- aucun module de message ;
- aucun indicateur Known/New ;
- aucun $\Delta t$.

### Résultats sur la Validation

Le contrôle obtient :

$$
PR\text{-}AUC_{Control}
=
\mathbf{0.606575}
$$

et :

$$
ROC\text{-}AUC_{Control}
=
0.990075
$$

Le candidat dynamique obtenait :

$$
PR\text{-}AUC_{Dynamic}
=
0.547378
$$

et :

$$
ROC\text{-}AUC_{Dynamic}
=
0.980523
$$

La différence de PR-AUC est donc :

$$
\Delta PR\text{-}AUC
=
PR\text{-}AUC_{Dynamic}
-
PR\text{-}AUC_{Control}
$$

$$
=
0.547378-0.606575
=
\mathbf{-0.059197}
$$

La mémoire temporelle telle qu'elle est actuellement implémentée **dégrade donc la performance de ranking sur la Validation**.

### Comparaison avec GraphSAGE

Le GraphSAGE Core Edge du notebook 06 obtenait :

$$
PR\text{-}AUC_{GraphSAGE}
=
0.240136
$$

Le contrôle neuronal sans mémoire atteint :

$$
PR\text{-}AUC_{Control}
=
0.606575
$$

Une grande partie du gain observé précédemment entre GraphSAGE et Dynamic GNN ne peut donc pas être attribuée directement à la mémoire temporelle.

Le prédicteur neuronal entraîné directement sur les Graph Core Edge constitue déjà une baseline nettement plus forte que le GraphSAGE spatial.

Cette observation est importante pour l'interprétation scientifique du projet.

La comparaison pertinente devient désormais :

| Modèle | Validation PR-AUC |
| --- | ---: |
| GraphSAGE Core Edge | 0.240136 |
| Dynamic GNN avec mémoire | 0.547378 |
| **Contrôle neuronal sans mémoire** | **0.606575** |
| LightGBM Core | 0.929704 |

### Coût computationnel

Le contrôle sans mémoire termine son entraînement en environ :

$$
14.72 \text{ secondes}
$$

contre :

$$
202.33 \text{ secondes}
$$

pour le candidat dynamique.

La mémoire temporelle actuelle introduit donc un coût computationnel important sans améliorer la PR-AUC.

### Conclusion de l'ablation

Le résultat ne remet pas en cause l'intérêt général des modèles temporels pour la détection de fraude.

Il montre plus précisément que **la première formulation de la mémoire dynamique testée dans ce notebook n'est pas adaptée de manière optimale à la structure particulière de PaySim**.

Plusieurs éléments peuvent contribuer à ce comportement :

- environ 99,7 % des sources sont nouvelles ;
- une grande partie des transactions ne dispose donc pas d'un historique bilatéral ;
- l'information structurelle du graphe est extrêmement peu profonde ;
- les gradients temporels sont absents dans 40,12 % des fenêtres TBPTT ;
- le TBPTT de deux steps limite la portée du signal temporel ;
- une mémoire destination-centric apprise peut introduire du bruit lorsque l'historique disponible n'est pas suffisamment discriminant.

Le candidat dynamique actuel ne doit donc pas être sélectionné comme modèle final uniquement parce qu'il contient une composante temporelle.

**Section 11 validée.**

# 12. Diagnostic comparatif du signal temporel

## 12.1 Objectif

L'ablation précédente montre que le contrôle sans mémoire surpasse globalement le candidat dynamique sur la Validation.

Cependant, une métrique globale ne permet pas d'identifier précisément les situations dans lesquelles la mémoire temporelle aide ou dégrade la prédiction.

L'objectif est donc de comparer les deux modèles sur exactement les mêmes sous-populations :

- `CASH_OUT` ;
- `TRANSFER` ;
- `N→K` ;
- `N→N`.

Les catégories Known/New sont déterminées uniquement à partir de l'historique causal du graphe et ne dépendent pas des scores du modèle.

Elles peuvent donc être réutilisées pour analyser le contrôle sans mémoire.

Pour chaque sous-population, nous calculerons :

$$
\Delta AP
=
AP_{Dynamic}
-
AP_{Control}
$$

Trois cas sont possibles :

- $\Delta AP > 0$ : la dynamique apporte un gain local ;
- $\Delta AP \approx 0$ : la mémoire apporte peu d'information supplémentaire ;
- $\Delta AP < 0$ : la mémoire dégrade le ranking sur cette population.

Cette analyse permettra de décider s'il faut abandonner complètement la mémoire actuelle ou seulement modifier la manière dont elle intervient dans la prédiction.

In [76]:
# Comparer Dynamic et contrôle par type
comparison_type_rows = []

for tx_type in type_categories:

    mask = (
        val_type_name == tx_type
    )

    labels_subset = val_labels[mask]

    dynamic_subset = val_scores[mask]
    control_subset = control_val_scores[mask]

    n = len(labels_subset)
    frauds = int(labels_subset.sum())

    if frauds == 0 or frauds == n:
        dynamic_ap = np.nan
        control_ap = np.nan
        delta_ap = np.nan
    else:
        dynamic_ap = average_precision_score(
            labels_subset,
            dynamic_subset,
        )

        control_ap = average_precision_score(
            labels_subset,
            control_subset,
        )

        delta_ap = (
            dynamic_ap
            - control_ap
        )

    comparison_type_rows.append({
        "type": tx_type,
        "transactions": n,
        "frauds": frauds,
        "dynamic_pr_auc": dynamic_ap,
        "control_pr_auc": control_ap,
        "delta_dynamic_control": delta_ap,
    })

comparison_type = pd.DataFrame(
    comparison_type_rows
)

print(
    comparison_type.to_string(
        index=False,
        formatters={
            "dynamic_pr_auc":
                lambda x: (
                    "NA"
                    if np.isnan(x)
                    else f"{x:.6f}"
                ),
            "control_pr_auc":
                lambda x: (
                    "NA"
                    if np.isnan(x)
                    else f"{x:.6f}"
                ),
            "delta_dynamic_control":
                lambda x: (
                    "NA"
                    if np.isnan(x)
                    else f"{x:+.6f}"
                ),
        },
    )
)

    type  transactions  frauds dynamic_pr_auc control_pr_auc delta_dynamic_control
 CASH_IN        174986       0            NaN            NaN                   NaN
CASH_OUT        274426     319       0.524411       0.566440             -0.042030
   DEBIT          5989       0            NaN            NaN                   NaN
 PAYMENT        270349       0            NaN            NaN                   NaN
TRANSFER         66656     319       0.592375       0.667300             -0.074926


In [77]:
# Comparer Dynamic et contrôle par régime cold-start
comparison_cold_rows = []

for category in [
    "K→K",
    "N→K",
    "K→N",
    "N→N",
]:

    mask = (
        cold_category == category
    )

    labels_subset = val_labels[mask]

    dynamic_subset = val_scores[mask]
    control_subset = control_val_scores[mask]

    n = len(labels_subset)
    frauds = int(labels_subset.sum())

    if frauds == 0 or frauds == n:
        dynamic_ap = np.nan
        control_ap = np.nan
        delta_ap = np.nan
    else:
        dynamic_ap = average_precision_score(
            labels_subset,
            dynamic_subset,
        )

        control_ap = average_precision_score(
            labels_subset,
            control_subset,
        )

        delta_ap = (
            dynamic_ap
            - control_ap
        )

    comparison_cold_rows.append({
        "category": category,
        "transactions": n,
        "frauds": frauds,
        "dynamic_pr_auc": dynamic_ap,
        "control_pr_auc": control_ap,
        "delta_dynamic_control": delta_ap,
    })

comparison_cold = pd.DataFrame(
    comparison_cold_rows
)

print(
    comparison_cold.to_string(
        index=False,
        formatters={
            "dynamic_pr_auc":
                lambda x: (
                    "NA"
                    if np.isnan(x)
                    else f"{x:.6f}"
                ),
            "control_pr_auc":
                lambda x: (
                    "NA"
                    if np.isnan(x)
                    else f"{x:.6f}"
                ),
            "delta_dynamic_control":
                lambda x: (
                    "NA"
                    if np.isnan(x)
                    else f"{x:+.6f}"
                ),
        },
    )
)

category  transactions  frauds dynamic_pr_auc control_pr_auc delta_dynamic_control
     K→K           104       0            NaN            NaN                   NaN
     N→K        451111     241       0.420695       0.483460             -0.062765
     K→N            80       0            NaN            NaN                   NaN
     N→N        341111     397       0.634929       0.715360             -0.080431


In [78]:
control_predictions_path = (
    dynamic_artifact_dir
    / "no_memory_control_validation_predictions.npz"
)

np.savez_compressed(
    control_predictions_path,
    scores=control_val_scores,
    labels=control_val_labels,
)

print(
    "Prédictions contrôle sauvegardées :",
    control_predictions_path.exists(),
)

print(
    "Taille :",
    f"{control_predictions_path.stat().st_size / 1024**2:.2f} MiB",
)

Prédictions contrôle sauvegardées : True
Taille : 2.70 MiB


## 12.2 Résultats du diagnostic comparatif

La comparaison entre le Dynamic GNN et le contrôle sans mémoire a été réalisée sur exactement les mêmes transactions de Validation.

### Comparaison par type de transaction

| Type | Fraudes | Dynamic PR-AUC | Contrôle PR-AUC | Δ Dynamic − Contrôle |
| --- | ---: | ---: | ---: | ---: |
| CASH_OUT | 319 | 0.524411 | **0.566440** | −0.042030 |
| TRANSFER | 319 | 0.592375 | **0.667300** | −0.074926 |

Le contrôle sans mémoire surpasse le modèle dynamique pour les deux mécanismes de fraude.

La dégradation est particulièrement importante sur `TRANSFER` :

$$
\Delta AP_{TRANSFER}=-0.074926
$$

Elle est également présente sur `CASH_OUT` :

$$
\Delta AP_{CASH\_OUT}=-0.042030
$$

La mémoire actuelle ne produit donc pas un gain spécialisé sur l'un des deux types frauduleux.

### Comparaison selon le régime Known/New

| Régime | Fraudes | Dynamic PR-AUC | Contrôle PR-AUC | Δ Dynamic − Contrôle |
| --- | ---: | ---: | ---: | ---: |
| N→K | 241 | 0.420695 | **0.483460** | −0.062765 |
| N→N | 397 | 0.634929 | **0.715360** | −0.080431 |

Le résultat du régime `N→K` est particulièrement important.

Dans cette catégorie, la destination dispose d'un état temporel historique. Il s'agit donc du scénario dans lequel la mémoire destination-centric devait théoriquement apporter le plus de valeur.

Pourtant :

$$
AP_{Dynamic}^{N\rightarrow K}
=
0.420695
$$

contre :

$$
AP_{Control}^{N\rightarrow K}
=
0.483460
$$

soit :

$$
\Delta AP_{N\rightarrow K}
=
-0.062765
$$

La mémoire actuelle ne fournit donc aucun gain local observable, même lorsque l'historique destination est disponible.

### Conclusion

Le contrôle sans mémoire surpasse le Dynamic GNN dans toutes les sous-populations pour lesquelles la PR-AUC peut être calculée.

Il serait donc injustifié de complexifier immédiatement cette architecture avec un mécanisme de gate ou une fusion supplémentaire de la même mémoire.

Les prédictions du contrôle ont été sauvegardées afin de permettre la poursuite des analyses sans réexécuter son évaluation.

Cependant, un dernier point d'implémentation doit être diagnostiqué avant de conclure que le concept de mémoire temporelle est lui-même inadapté.

Le modèle dynamique a été entraîné avec :

$$
TBPTT = 2
$$

Cette fenêtre ne permet au gradient temporel de traverser qu'une distance extrêmement courte dans le flux.

De plus, avec les fenêtres non chevauchantes utilisées, seules les mises à jour du premier step de chaque fenêtre peuvent recevoir directement un signal provenant du step suivant.

Il faut donc vérifier si la fréquence réelle de réapparition des destinations est compatible avec cette portée temporelle.

**Section 12 validée.**

# 13. Diagnostic de la portée temporelle du TBPTT

## 13.1 Motivation

L'ablation montre que la mémoire temporelle actuelle dégrade les performances.

Cependant, cette mémoire a été entraînée avec une fenêtre très courte :

$$
TBPTT = 2
$$

et des fenêtres non chevauchantes :

$$
(1,2),\ (3,4),\ (5,6),\ldots
$$

Le gradient issu d'une prédiction ne peut donc traverser qu'une transition temporelle très courte avant que l'état ne soit détaché.

Cette limitation peut être importante si les destinations récurrentes réapparaissent généralement après plusieurs steps.

L'objectif est de mesurer, exclusivement sur le Train, la distribution réelle de :

$$
\Delta t_v
=
t-last\_update_v
$$

pour les destinations possédant déjà un historique.

Nous examinerons notamment la proportion des réapparitions vérifiant :

$$
\Delta t \leq 1
$$

$$
\Delta t \leq 2
$$

$$
\Delta t \leq 4
$$

$$
\Delta t \leq 8
$$

$$
\Delta t \leq 16
$$

ainsi que les quantiles de la distribution.

Le même diagnostic sera calculé séparément pour les transactions frauduleuses du Train.

Cette analyse permettra de déterminer si la portée temporelle utilisée lors du premier entraînement était cohérente avec les récurrences réellement présentes dans PaySim.

In [79]:
id="qn7k52"
# Index vide pour le diagnostic
gap_index = SparseTemporalIndex(
    num_nodes=9_073_900,
    initial_capacity=2_100_000,
    growth_size=100_000,
)

known_gap_blocks = []
fraud_known_gap_blocks = []

known_events = 0
fraud_known_events = 0

for t in range(1, 335):

    start = int(offsets[t - 1])
    end = int(offsets[t])

    dst_t = np.asarray(
        dst[start:end]
    )

    label_t = np.asarray(
        label[start:end]
    )

    # Etat strictement antérieur
    _, known_t, dt_t = (
        gap_index.lookup(
            dst_t,
            t=t,
        )
    )

    if known_t.any():

        gaps = dt_t[known_t].astype(
            np.int16,
            copy=True,
        )

        known_gap_blocks.append(
            gaps
        )

        known_events += len(gaps)

        fraud_known_mask = (
            known_t
            & (label_t == 1)
        )

        if fraud_known_mask.any():

            fraud_gaps = dt_t[
                fraud_known_mask
            ].astype(
                np.int16,
                copy=True,
            )

            fraud_known_gap_blocks.append(
                fraud_gaps
            )

            fraud_known_events += len(
                fraud_gaps
            )

    # Intégration après le lookup
    gap_index.integrate_destinations(
        dst_t,
        t=t,
    )


train_known_gaps = np.concatenate(
    known_gap_blocks
)

if fraud_known_gap_blocks:
    train_fraud_known_gaps = np.concatenate(
        fraud_known_gap_blocks
    )
else:
    train_fraud_known_gaps = np.array(
        [],
        dtype=np.int16,
    )


print(
    "Réapparitions destinations connues :",
    f"{len(train_known_gaps):,}",
)

print(
    "Fraudes avec destination connue :",
    f"{len(train_fraud_known_gaps):,}",
)

print(
    "Slots finaux :",
    f"{gap_index.size:,}",
)

Réapparitions destinations connues : 2,689,915
Fraudes avec destination connue : 1,374
Slots finaux : 2,035,605


In [80]:
id="dq6u6p"
# Résumer une distribution de delta t
def print_gap_summary(name, gaps):

    print(name)
    print(
        "  observations :",
        f"{len(gaps):,}",
    )

    if len(gaps) == 0:
        print()
        return

    print(
        "  min/max :",
        int(gaps.min()),
        "/",
        int(gaps.max()),
    )

    for q in [
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
    ]:
        print(
            f"  q={q:.2f} :",
            f"{np.quantile(gaps, q):.0f}",
        )

    print()

    for horizon in [
        1,
        2,
        4,
        8,
        16,
        32,
    ]:
        rate = (
            100
            * np.mean(
                gaps <= horizon
            )
        )

        print(
            f"  Δt <= {horizon:<2} : "
            f"{rate:.2f} %"
        )

    print()


print_gap_summary(
    "Toutes les destinations connues",
    train_known_gaps,
)

print_gap_summary(
    "Fraudes avec destination connue",
    train_fraud_known_gaps,
)

Toutes les destinations connues
  observations : 2,689,915
  min/max : 1 / 317
  q=0.25 : 2
  q=0.50 : 9
  q=0.75 : 25
  q=0.90 : 50
  q=0.95 : 81
  q=0.99 : 115

  Δt <= 1  : 17.90 %
  Δt <= 2  : 27.70 %
  Δt <= 4  : 39.12 %
  Δt <= 8  : 49.20 %
  Δt <= 16 : 59.06 %
  Δt <= 32 : 82.21 %

Fraudes avec destination connue
  observations : 1,374
  min/max : 1 / 267
  q=0.25 : 6
  q=0.50 : 18
  q=0.75 : 39
  q=0.90 : 69
  q=0.95 : 85
  q=0.99 : 135

  Δt <= 1  : 8.95 %
  Δt <= 2  : 13.97 %
  Δt <= 4  : 20.31 %
  Δt <= 8  : 30.35 %
  Δt <= 16 : 47.67 %
  Δt <= 32 : 69.65 %



In [81]:
id="m6u06k"
# Compter les réutilisations à distance 1
# réellement traversables par notre TBPTT actuel
gap_index_2 = SparseTemporalIndex(
    num_nodes=9_073_900,
    initial_capacity=2_100_000,
    growth_size=100_000,
)

known_total = 0
known_dt1 = 0
tbptt_reachable = 0

fraud_known_total = 0
fraud_dt1 = 0
fraud_tbptt_reachable = 0

for t in range(1, 335):

    start = int(offsets[t - 1])
    end = int(offsets[t])

    dst_t = np.asarray(
        dst[start:end]
    )

    label_t = np.asarray(
        label[start:end]
    )

    slots_t, known_t, dt_t = (
        gap_index_2.lookup(
            dst_t,
            t=t,
        )
    )

    known_total += int(
        known_t.sum()
    )

    dt1_mask = (
        known_t
        & (dt_t == 1)
    )

    known_dt1 += int(
        dt1_mask.sum()
    )

    # Le step précédent doit être impair :
    # (1→2), (3→4), ...
    reachable_mask = (
        dt1_mask
        & ((t - 1) % 2 == 1)
    )

    tbptt_reachable += int(
        reachable_mask.sum()
    )

    fraud_mask = (
        label_t == 1
    )

    fraud_known_total += int(
        (
            known_t
            & fraud_mask
        ).sum()
    )

    fraud_dt1 += int(
        (
            dt1_mask
            & fraud_mask
        ).sum()
    )

    fraud_tbptt_reachable += int(
        (
            reachable_mask
            & fraud_mask
        ).sum()
    )

    gap_index_2.integrate_destinations(
        dst_t,
        t=t,
    )


print("Toutes transactions")
print(
    "  destinations connues :",
    f"{known_total:,}",
)
print(
    "  réapparition Δt=1 :",
    f"{known_dt1:,}",
    f"({100 * known_dt1 / known_total:.2f} %)",
)
print(
    "  réellement traversables TBPTT=2 :",
    f"{tbptt_reachable:,}",
    f"({100 * tbptt_reachable / known_total:.2f} %)",
)

print()

print("Fraudes")
print(
    "  destinations connues :",
    f"{fraud_known_total:,}",
)

if fraud_known_total > 0:
    print(
        "  réapparition Δt=1 :",
        f"{fraud_dt1:,}",
        f"({100 * fraud_dt1 / fraud_known_total:.2f} %)",
    )

    print(
        "  réellement traversables TBPTT=2 :",
        f"{fraud_tbptt_reachable:,}",
        f"({100 * fraud_tbptt_reachable / fraud_known_total:.2f} %)",
    )

Toutes transactions
  destinations connues : 2,689,915
  réapparition Δt=1 : 481,488 (17.90 %)
  réellement traversables TBPTT=2 : 244,676 (9.10 %)

Fraudes
  destinations connues : 1,374
  réapparition Δt=1 : 123 (8.95 %)
  réellement traversables TBPTT=2 : 60 (4.37 %)


## 13.2 Résultats du diagnostic de portée temporelle

Le Train contient :

- **2 689 915 réapparitions de destinations connues** ;
- **1 374 transactions frauduleuses impliquant une destination connue**.

La distribution des intervalles de réapparition montre que l'historique utile s'étend sur une durée beaucoup plus longue que la fenêtre TBPTT utilisée par le premier candidat.

### Toutes les destinations connues

Pour l'ensemble des réapparitions :

| Statistique | $\Delta t$ |
| --- | ---: |
| 25 % | 2 |
| Médiane | **9** |
| 75 % | 25 |
| 90 % | 50 |
| 95 % | 81 |
| 99 % | 115 |
| Maximum | 317 |

Seulement :

$$
17.90\%
$$

des réapparitions interviennent dès le step suivant.

La couverture cumulée atteint :

| Horizon | Réapparitions couvertes |
| ---: | ---: |
| $\Delta t \leq 1$ | 17,90 % |
| $\Delta t \leq 2$ | 27,70 % |
| $\Delta t \leq 4$ | 39,12 % |
| $\Delta t \leq 8$ | 49,20 % |
| $\Delta t \leq 16$ | 59,06 % |
| $\Delta t \leq 32$ | 82,21 % |

La récurrence des destinations n'est donc pas principalement un phénomène à très court terme.

### Fraudes avec destination connue

La distribution est encore plus éloignée dans le temps pour les fraudes :

| Statistique | $\Delta t$ |
| --- | ---: |
| 25 % | 6 |
| Médiane | **18** |
| 75 % | 39 |
| 90 % | 69 |
| 95 % | 85 |
| 99 % | 135 |
| Maximum | 267 |

Seulement :

$$
8.95\%
$$

des fraudes avec destination connue réapparaissent après un seul step.

La couverture cumulée est :

| Horizon | Fraudes couvertes |
| ---: | ---: |
| $\Delta t \leq 1$ | 8,95 % |
| $\Delta t \leq 2$ | 13,97 % |
| $\Delta t \leq 4$ | 20,31 % |
| $\Delta t \leq 8$ | 30,35 % |
| $\Delta t \leq 16$ | 47,67 % |
| $\Delta t \leq 32$ | 69,65 % |

La médiane de **18 steps** est particulièrement importante : le signal temporel associé aux transactions frauduleuses connues se situe fréquemment bien au-delà de la portée du premier TBPTT.

### Couverture réelle du TBPTT=2

Le premier entraînement utilisait des fenêtres non chevauchantes :

$$
(1,2),\ (3,4),\ (5,6),\ldots
$$

Une réapparition à $\Delta t=1$ n'est donc pas automatiquement traversable par le gradient. Elle doit également se produire à l'intérieur de la même fenêtre.

Sur les **2 689 915** transactions avec destination connue :

- **481 488** présentent $\Delta t=1$ ;
- seulement **244 676** sont réellement connectées dans une fenêtre TBPTT=2.

La couverture effective est donc :

$$
\frac{244\,676}{2\,689\,915}
=
\mathbf{9.10\%}
$$

Pour les fraudes :

- destinations connues : **1 374** ;
- réapparitions à $\Delta t=1$ : **123** ;
- réellement traversables : **60**.

La couverture effective tombe à :

$$
\frac{60}{1\,374}
=
\mathbf{4.37\%}
$$

### Interprétation

Le premier candidat Dynamic GNN a donc été entraîné avec une portée temporelle très inférieure à la dynamique réelle du dataset.

La comparaison précédente reste factuellement valide :

$$
PR\text{-}AUC_{Dynamic,TBPTT=2}=0.547378
$$

contre :

$$
PR\text{-}AUC_{Control}=0.606575
$$

mais elle ne permet pas encore de conclure que toute mémoire temporelle est inutile.

Elle démontre plus précisément que :

> **la mémoire destination-centric entraînée avec TBPTT=2 n'est pas suffisamment efficace.**

Avant d'abandonner cette formulation, il est justifié de tester une portée temporelle mieux alignée sur les intervalles réellement observés.

Le choix de cette nouvelle portée sera effectué exclusivement à partir du Train et de la Validation.

Aucune donnée du Test ne sera utilisée.

In [82]:
# Longueurs candidates
tbptt_candidates = [2, 4, 8, 16, 32]

coverage = {
    h: {
        "known_total": 0,
        "compatible": 0,
        "reachable": 0,
        "fraud_known_total": 0,
        "fraud_compatible": 0,
        "fraud_reachable": 0,
    }
    for h in tbptt_candidates
}

# Rejouer uniquement l'index temporel
coverage_index = SparseTemporalIndex(
    num_nodes=9_073_900,
    initial_capacity=2_100_000,
    growth_size=100_000,
)

for t in range(1, 335):

    start = int(offsets[t - 1])
    end = int(offsets[t])

    dst_t = np.asarray(
        dst[start:end]
    )

    label_t = np.asarray(
        label[start:end]
    )

    _, known_t, dt_t = coverage_index.lookup(
        dst_t,
        t=t,
    )

    if known_t.any():

        # Step de la précédente observation
        previous_t = (
            t - dt_t
        )

        fraud_known = (
            known_t
            & (label_t == 1)
        )

        for h in tbptt_candidates:

            stats = coverage[h]

            stats["known_total"] += int(
                known_t.sum()
            )

            stats["fraud_known_total"] += int(
                fraud_known.sum()
            )

            # Distance théoriquement compatible
            compatible = (
                known_t
                & (dt_t < h)
            )

            fraud_compatible = (
                compatible
                & (label_t == 1)
            )

            # Même fenêtre TBPTT non chevauchante
            same_window = (
                ((previous_t - 1) // h)
                == ((t - 1) // h)
            )

            reachable = (
                known_t
                & same_window
            )

            fraud_reachable = (
                reachable
                & (label_t == 1)
            )

            stats["compatible"] += int(
                compatible.sum()
            )

            stats["reachable"] += int(
                reachable.sum()
            )

            stats["fraud_compatible"] += int(
                fraud_compatible.sum()
            )

            stats["fraud_reachable"] += int(
                fraud_reachable.sum()
            )

    # Intégrer uniquement après les mesures
    coverage_index.integrate_destinations(
        dst_t,
        t=t,
    )

In [83]:
print("Couverture TBPTT réelle")
print()

for h in tbptt_candidates:

    stats = coverage[h]

    known_rate = (
        100
        * stats["reachable"]
        / stats["known_total"]
    )

    compatible_rate = (
        100
        * stats["compatible"]
        / stats["known_total"]
    )

    fraud_rate = (
        100
        * stats["fraud_reachable"]
        / stats["fraud_known_total"]
    )

    fraud_compatible_rate = (
        100
        * stats["fraud_compatible"]
        / stats["fraud_known_total"]
    )

    print(f"TBPTT = {h}")
    print(
        "  Toutes destinations"
    )
    print(
        "    distance compatible :",
        f"{stats['compatible']:,}",
        f"({compatible_rate:.2f} %)",
    )
    print(
        "    réellement traversable :",
        f"{stats['reachable']:,}",
        f"({known_rate:.2f} %)",
    )

    print(
        "  Fraudes"
    )
    print(
        "    distance compatible :",
        f"{stats['fraud_compatible']:,}",
        f"({fraud_compatible_rate:.2f} %)",
    )
    print(
        "    réellement traversable :",
        f"{stats['fraud_reachable']:,}",
        f"({fraud_rate:.2f} %)",
    )

    print()

Couverture TBPTT réelle

TBPTT = 2
  Toutes destinations
    distance compatible : 481,488 (17.90 %)
    réellement traversable : 244,676 (9.10 %)
  Fraudes
    distance compatible : 123 (8.95 %)
    réellement traversable : 60 (4.37 %)

TBPTT = 4
  Toutes destinations
    distance compatible : 922,891 (34.31 %)
    réellement traversable : 568,642 (21.14 %)
  Fraudes
    distance compatible : 247 (17.98 %)
    réellement traversable : 129 (9.39 %)

TBPTT = 8
  Toutes destinations
    distance compatible : 1,280,543 (47.61 %)
    réellement traversable : 890,148 (33.09 %)
  Fraudes
    distance compatible : 387 (28.17 %)
    réellement traversable : 259 (18.85 %)

TBPTT = 16
  Toutes destinations
    distance compatible : 1,536,281 (57.11 %)
    réellement traversable : 1,170,310 (43.51 %)
  Fraudes
    distance compatible : 627 (45.63 %)
    réellement traversable : 407 (29.62 %)

TBPTT = 32
  Toutes destinations
    distance compatible : 2,200,515 (81.81 %)
    réellement traversable

## 13.3 Sélection d'une nouvelle portée TBPTT

La couverture réelle du gradient a été calculée pour plusieurs longueurs de TBPTT en tenant compte à la fois :

- de la distance entre deux observations d'une destination ;
- des frontières des fenêtres TBPTT non chevauchantes.

Les résultats sont :

| TBPTT | Destinations réellement traversables | Fraudes réellement traversables |
| ---: | ---: | ---: |
| 2 | 9,10 % | 4,37 % |
| 4 | 21,14 % | 9,39 % |
| 8 | 33,09 % | 18,85 % |
| 16 | **43,51 %** | **29,62 %** |
| 32 | 57,93 % | 47,31 % |

Le passage de `TBPTT=2` à `TBPTT=16` multiplie fortement la couverture temporelle effective.

Pour les fraudes avec destination connue :

$$
4.37\%
\rightarrow
29.62\%
$$

Le modèle pourrait donc recevoir un signal d'apprentissage temporel sur une part beaucoup plus représentative des récurrences frauduleuses.

`TBPTT=32` offre une couverture encore supérieure, mais son coût en mémoire d'autograd pourrait devenir important sur la machine locale, notamment lorsque plusieurs steps volumineux appartiennent à une même fenêtre.

Le candidat retenu pour le prochain benchmark est donc :

$$
\boxed{TBPTT=16}
$$

Ce choix ne constitue pas encore la configuration finale.

Il doit d'abord passer un benchmark mémoire sur une fenêtre Train représentative et volumineuse.

Aucune donnée de Validation ou de Test n'intervient dans cette décision.

# 14. Benchmark mémoire avec TBPTT=16

## 14.1 Objectif

L'augmentation de la portée temporelle améliore fortement la couverture des récurrences, mais elle augmente également la quantité d'informations conservées dans le graphe d'autograd avant chaque `backward()`.

Avec :

$$
TBPTT=16
$$

le modèle doit conserver les dépendances temporelles de seize steps consécutifs.

Sur PaySim, certains steps contiennent plus de 40 000 transactions.

Une fenêtre de seize steps peut donc représenter plusieurs centaines de milliers de transactions avant le détachement de la mémoire.

L'objectif de cette section est de vérifier si une telle fenêtre reste compatible avec :

- la RAM disponible ;
- le runtime CPU ;
- la stabilité des gradients ;
- la structure actuelle de l'overlay différentiable.

Le benchmark sera effectué uniquement sur le Train.

Avant de construire le graphe d'autograd, les fenêtres de seize steps réellement alignées avec le protocole TBPTT seront analysées afin de sélectionner une fenêtre exigeante mais représentative.

In [84]:
# Construire les fenêtres TBPTT=16 alignées
tbptt_16_windows = []

for start_step in range(1, 335, 16):

    end_step = min(
        start_step + 15,
        334,
    )

    start_offset = int(
        offsets[start_step - 1]
    )

    end_offset = int(
        offsets[end_step]
    )

    n_tx = (
        end_offset - start_offset
    )

    frauds = int(
        np.asarray(
            label[start_offset:end_offset]
        ).sum()
    )

    tbptt_16_windows.append({
        "start": start_step,
        "end": end_step,
        "transactions": n_tx,
        "frauds": frauds,
    })


# Trier par volume
sorted_windows = sorted(
    tbptt_16_windows,
    key=lambda x: x["transactions"],
    reverse=True,
)

print(
    "Nombre de fenêtres TBPTT=16 :",
    len(tbptt_16_windows),
)

print()
print(
    "10 fenêtres alignées les plus volumineuses"
)

for item in sorted_windows[:10]:

    print(
        f"Steps {item['start']:>3}-{item['end']:<3} | "
        f"tx={item['transactions']:,} | "
        f"fraudes={item['frauds']}"
    )

print()

window_tx_values = np.array(
    [
        item["transactions"]
        for item in tbptt_16_windows
    ],
    dtype=np.int64,
)

print("Distribution volume fenêtre")

for q in [
    0.50,
    0.75,
    0.90,
    0.95,
    1.00,
]:

    print(
        f"q={q:.2f} : "
        f"{np.quantile(window_tx_values, q):,.0f}"
    )

Nombre de fenêtres TBPTT=16 : 21

10 fenêtres alignées les plus volumineuses
Steps  33-48  | tx=453,119 | fraudes=231
Steps 177-192 | tx=447,578 | fraudes=186
Steps 129-144 | tx=439,423 | fraudes=152
Steps 225-240 | tx=389,534 | fraudes=202
Steps 321-334 | tx=382,599 | fraudes=136
Steps   1-16  | tx=348,191 | fraudes=197
Steps 273-288 | tx=347,291 | fraudes=208
Steps 241-256 | tx=248,744 | fraudes=170
Steps 193-208 | tx=245,981 | fraudes=151
Steps  17-32  | tx=228,183 | fraudes=152

Distribution volume fenêtre
q=0.50 : 226,753
q=0.75 : 348,191
q=0.90 : 439,423
q=0.95 : 447,578
q=1.00 : 453,119


## 14.2 Sélection de la fenêtre de benchmark

Le Train contient **21 fenêtres alignées** lorsque le TBPTT est fixé à 16 steps.

La distribution de leur volume est :

| Quantile | Transactions |
| ---: | ---: |
| 50 % | 226 753 |
| 75 % | 348 191 |
| 90 % | 439 423 |
| 95 % | 447 578 |
| Maximum | 453 119 |

La fenêtre la plus volumineuse est `33–48`, avec 453 119 transactions.

Cependant, le benchmark doit également représenter un régime dans lequel la mémoire temporelle est déjà suffisamment développée.

La fenêtre retenue est donc :

$$
\boxed{177 \leq t \leq 192}
$$

Elle contient :

- **447 578 transactions** ;
- **186 fraudes**.

Son volume correspond approximativement au 95e percentile des fenêtres TBPTT=16.

Cette fenêtre constitue donc un scénario exigeant, mais représentatif d'un entraînement en régime temporel établi.

Avant son traitement, les steps 1 à 176 seront intégrés structurellement afin de reconstruire :

- les slots connus ;
- les statuts Known/New ;
- les `last_update`.

Les vecteurs neuronaux issus des steps précédents ne seront pas réentraînés pendant ce warm-up.

Le benchmark reste donc une expérience de **faisabilité computationnelle**, et non une mesure de performance prédictive.

Les 16 steps seront ensuite parcourus sans détachement intermédiaire.

Un seul `backward()` sera exécuté à la fin de la fenêtre :

$$
177
\rightarrow
178
\rightarrow
\cdots
\rightarrow
192
\rightarrow
Backward
$$

Le benchmark mesurera notamment :

- le runtime de chaque step ;
- le nombre de slots actifs ;
- la taille maximale de l'overlay différentiable ;
- la RAM disponible ;
- le RSS du processus ;
- le runtime du backward ;
- les gradients du prédicteur, du module de message et de la GRU.

In [85]:
import gc
import time
import psutil
import torch
import numpy as np

# Index dédié au benchmark TBPTT=16
benchmark16_index = SparseTemporalIndex(
    num_nodes=9_073_900,
    initial_capacity=2_100_000,
    growth_size=100_000,
)

warmup16_start = time.perf_counter()

for t in range(1, 177):
    start = int(offsets[t - 1])
    end = int(offsets[t])

    benchmark16_index.integrate_destinations(
        dst[start:end],
        t=t,
    )

warmup16_runtime = (
    time.perf_counter()
    - warmup16_start
)

active_last_update = (
    benchmark16_index.last_update[
        :benchmark16_index.size
    ]
)

print("Warm-up terminé : True")
print("Dernier step intégré : 176")
print(
    "Slots actifs :",
    f"{benchmark16_index.size:,}",
)
print(
    "last_update max :",
    int(active_last_update.max()),
)
print(
    "Runtime warm-up :",
    f"{warmup16_runtime:.2f} s",
)
print(
    "RAM disponible :",
    f"{psutil.virtual_memory().available / 1024**3:.2f} GB",
)

Warm-up terminé : True
Dernier step intégré : 176
Slots actifs : 822,281
last_update max : 176
Runtime warm-up : 0.27 s
RAM disponible : 2.91 GB


In [86]:
# Reproductibilité
torch.manual_seed(42)
np.random.seed(42)

benchmark16_model = PilotDynamicModel(
    edge_dim=13,
    memory_dim=8,
    hidden_dim=32,
)

benchmark16_model.train()

benchmark16_memory = PilotTemporalMemory(
    temporal_index=benchmark16_index,
    memory_dim=8,
    initial_capacity=2_100_000,
    growth_size=100_000,
)

benchmark16_criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        1273.0686248331108,
        dtype=torch.float32,
    ),
    reduction="sum",
)

benchmark16_optimizer = torch.optim.AdamW(
    benchmark16_model.parameters(),
    lr=1e-3,
    weight_decay=1e-5,
)

process = psutil.Process()

print("Fenêtre benchmark : 177-192")
print("TBPTT : 16")
print(
    "Slots avant fenêtre :",
    f"{benchmark16_index.size:,}",
)
print(
    "Tensor mémoire :",
    f"{benchmark16_memory.memory.nbytes / 1024**2:.2f} MiB",
)
print(
    "RSS processus :",
    f"{process.memory_info().rss / 1024**3:.2f} GB",
)
print(
    "RAM disponible :",
    f"{psutil.virtual_memory().available / 1024**3:.2f} GB",
)

Fenêtre benchmark : 177-192
TBPTT : 16
Slots avant fenêtre : 822,281
Tensor mémoire : 64.09 MiB
RSS processus : 0.79 GB
RAM disponible : 2.78 GB


In [87]:
benchmark16_optimizer.zero_grad(
    set_to_none=True
)

loss_sum = None

total_tx = 0
total_fraud = 0

max_overlay = 0
max_rss = 0.0
min_ram_available = float("inf")

step_benchmark16 = []

benchmark16_start = time.perf_counter()

for t in range(177, 193):

    step_start = time.perf_counter()

    batch = reader.read_step(t)

    logits, y, info = process_pilot_step(
        model=benchmark16_model,
        temporal_memory=benchmark16_memory,
        batch=batch,
    )

    step_loss = benchmark16_criterion(
        logits,
        y,
    )

    if loss_sum is None:
        loss_sum = step_loss
    else:
        loss_sum = (
            loss_sum + step_loss
        )

    total_tx += info["transactions"]
    total_fraud += info["frauds"]

    overlay_size = len(
        benchmark16_memory.overlay
    )

    rss_gib = (
        process.memory_info().rss
        / 1024**3
    )

    ram_available = (
        psutil.virtual_memory().available
        / 1024**3
    )

    max_overlay = max(
        max_overlay,
        overlay_size,
    )

    max_rss = max(
        max_rss,
        rss_gib,
    )

    min_ram_available = min(
        min_ram_available,
        ram_available,
    )

    runtime = (
        time.perf_counter()
        - step_start
    )

    step_benchmark16.append({
        "step": t,
        "transactions": info["transactions"],
        "frauds": info["frauds"],
        "known_src": info["known_src"],
        "known_dst": info["known_dst"],
        "active_slots": info["active_slots"],
        "overlay": overlay_size,
        "runtime": runtime,
        "rss_gib": rss_gib,
        "ram_available_gib": ram_available,
    })

    print(
        f"Step {t} | "
        f"tx={info['transactions']:,} | "
        f"fraudes={info['frauds']} | "
        f"known_dst={info['known_dst']:,} | "
        f"overlay={overlay_size:,} | "
        f"runtime={runtime:.2f}s | "
        f"RSS={rss_gib:.2f} GB | "
        f"RAM libre={ram_available:.2f} GB"
    )

    # Garde-fou mémoire
    if ram_available < 0.75:
        raise MemoryError(
            "Benchmark arrêté : "
            "RAM disponible < 0.75 GB."
        )


forward_runtime = (
    time.perf_counter()
    - benchmark16_start
)

loss = (
    loss_sum / total_tx
)

print()
print(
    "Forward 16 steps terminé :",
    True,
)
print(
    "Transactions :",
    f"{total_tx:,}",
)
print(
    "Fraudes :",
    f"{total_fraud:,}",
)
print(
    "Loss avant backward :",
    f"{float(loss.detach()):.6f}",
)
print(
    "Overlay maximal :",
    f"{max_overlay:,}",
)
print(
    "RSS maximal avant backward :",
    f"{max_rss:.2f} GB",
)
print(
    "RAM libre minimale :",
    f"{min_ram_available:.2f} GB",
)
print(
    "Runtime forward :",
    f"{forward_runtime:.2f} s",
)

# Backward unique sur 16 steps
backward_start = time.perf_counter()

loss.backward()

backward_runtime = (
    time.perf_counter()
    - backward_start
)

predictor_grad = module_grad_norm(
    benchmark16_model.predictor
)

message_grad = module_grad_norm(
    benchmark16_model.message_module
)

gru_grad = module_grad_norm(
    benchmark16_model.gru
)

total_grad = (
    torch.nn.utils.clip_grad_norm_(
        benchmark16_model.parameters(),
        max_norm=5.0,
    )
)

rss_after_backward = (
    process.memory_info().rss
    / 1024**3
)

ram_after_backward = (
    psutil.virtual_memory().available
    / 1024**3
)

print()
print("Backward terminé : True")
print(
    "Runtime backward :",
    f"{backward_runtime:.2f} s",
)
print(
    "Gradient prédicteur :",
    f"{predictor_grad:.8f}",
)
print(
    "Gradient message :",
    f"{message_grad:.8f}",
)
print(
    "Gradient GRU :",
    f"{gru_grad:.8f}",
)
print(
    "Gradient total pré-clipping :",
    f"{float(total_grad):.8f}",
)
print(
    "RSS après backward :",
    f"{rss_after_backward:.2f} GB",
)
print(
    "RAM libre après backward :",
    f"{ram_after_backward:.2f} GB",
)

# Tester également la fin normale d'une fenêtre
benchmark16_optimizer.step()
benchmark16_memory.commit()

print()
print(
    "Overlay vide après commit :",
    len(benchmark16_memory.overlay) == 0,
)
print(
    "Mémoire finie :",
    bool(
        torch.isfinite(
            benchmark16_memory.memory[
                :benchmark16_index.size
            ]
        ).all()
    ),
)
print(
    "Slots finaux :",
    f"{benchmark16_index.size:,}",
)
print(
    "Runtime total benchmark :",
    f"{time.perf_counter() - benchmark16_start:.2f} s",
)

Step 177 | tx=17,336 | fraudes=18 | known_dst=9,751 | overlay=16,992 | runtime=1.30s | RSS=0.82 GB | RAM libre=2.91 GB
Step 178 | tx=28,204 | fraudes=14 | known_dst=16,278 | overlay=43,092 | runtime=1.17s | RSS=0.88 GB | RAM libre=2.82 GB
Step 179 | tx=30,534 | fraudes=8 | known_dst=18,015 | overlay=69,469 | runtime=1.85s | RSS=0.95 GB | RAM libre=2.65 GB
Step 180 | tx=31,481 | fraudes=10 | known_dst=18,515 | overlay=94,927 | runtime=1.25s | RSS=1.02 GB | RAM libre=2.51 GB
Step 181 | tx=31,749 | fraudes=11 | known_dst=18,743 | overlay=119,034 | runtime=0.82s | RSS=1.11 GB | RAM libre=2.38 GB
Step 182 | tx=31,900 | fraudes=8 | known_dst=18,652 | overlay=141,990 | runtime=1.49s | RSS=1.28 GB | RAM libre=2.14 GB
Step 183 | tx=31,199 | fraudes=16 | known_dst=18,126 | overlay=163,342 | runtime=0.63s | RSS=1.37 GB | RAM libre=2.05 GB
Step 184 | tx=32,175 | fraudes=8 | known_dst=18,997 | overlay=184,201 | runtime=0.67s | RSS=1.48 GB | RAM libre=1.95 GB
Step 185 | tx=37,323 | fraudes=16 | know

## 14.3 Résultats du benchmark TBPTT=16

Le benchmark a été réalisé sur la fenêtre Train :

$$
177 \leq t \leq 192
$$

Cette fenêtre contient :

- **447 578 transactions** ;
- **186 fraudes** ;
- un volume proche du 95e percentile des fenêtres alignées de 16 steps.

Avant son traitement, les steps 1 à 176 ont été intégrés structurellement.

L'état initial du benchmark contient :

- **822 281 slots actifs** ;
- un `last_update` maximal égal à **176** ;
- un tenseur de mémoire de **64,09 MiB**.

### Evolution de l'overlay

Les 16 steps ont été parcourus sans détachement intermédiaire.

L'overlay différentiable augmente progressivement avec les nouvelles destinations rencontrées dans la fenêtre et atteint au maximum :

$$
309\,712
$$

états temporels.

Cette croissance reste compatible avec les ressources disponibles.

### Consommation mémoire

Le RSS du processus avant le benchmark est de :

$$
0.79 \text{ GB}
$$

et atteint au maximum :

$$
2.03 \text{ GB}
$$

avant le backward.

La RAM disponible minimale observée pendant le forward est :

$$
1.58 \text{ GB}
$$

Le garde-fou fixé à 0,75 GB n'a donc jamais été approché.

Après le backward :

- RSS du processus : **1,77 GB** ;
- RAM disponible : **2,76 GB**.

La mémoire est correctement libérée après le calcul des gradients.

### Runtime

Le forward des 16 steps nécessite :

$$
15.59 \text{ secondes}
$$

Le backward unique sur la fenêtre nécessite :

$$
28.03 \text{ secondes}
$$

Le runtime total du benchmark est :

$$
47.37 \text{ secondes}
$$

Le backward constitue donc la partie la plus coûteuse de l'augmentation de la portée temporelle.

### Gradients

Les normes observées sont :

| Composant | Norme du gradient |
| --- | ---: |
| Prédicteur | 2.00110208 |
| Module de message | 0.00355068 |
| GRU | 0.00623089 |
| Total avant clipping | 2.00133061 |

Les gradients du module de message et de la GRU sont strictement non nuls.

Le signal d'apprentissage temporel traverse donc effectivement plusieurs steps dans cette fenêtre plus longue.

### Etat après la fenêtre

Après l'optimisation et le commit :

- overlay vide : **True** ;
- mémoire finie : **True** ;
- slots actifs : **1 011 019**.

Le protocole TBPTT=16 est donc compatible avec :

- la RAM disponible ;
- le CPU local ;
- la mémoire destination-centric ;
- le mécanisme d'overlay différentiable ;
- la mise à jour GRU.

### Décision

Le candidat :

$$
\boxed{TBPTT=16}
$$

est retenu pour un second entraînement dynamique complet.

Aucune autre longueur TBPTT ne sera testée avant son évaluation sur la Validation.

Cette décision limite volontairement le nombre d'expériences et évite une recherche d'hyperparamètres inutile.

**Section 14 validée.**

# 15. Entraînement du Dynamic GNN avec TBPTT=16

## 15.1 Objectif

Le premier candidat dynamique utilisait :

$$
TBPTT=2
$$

L'analyse du Train a montré que cette portée ne permettait au gradient d'atteindre que :

$$
4.37\%
$$

des fraudes impliquant une destination connue.

Le candidat `TBPTT=16` augmente cette couverture effective à :

$$
29.62\%
$$

Le benchmark mémoire réalisé sur une fenêtre de 447 578 transactions confirme que cette configuration reste compatible avec la machine locale.

L'objectif est maintenant d'entraîner un second candidat dynamique complet afin de répondre à la question suivante :

> Une portée temporelle mieux adaptée aux réapparitions des destinations permet-elle à la mémoire dynamique d'apporter une valeur prédictive réelle ?

## 15.2 Expérience contrôlée

Toutes les caractéristiques du premier Dynamic GNN sont conservées :

| Paramètre | Valeur |
| --- | ---: |
| Features | Graph Core Edge |
| Dimension Edge | 13 |
| Dimension mémoire | 8 |
| Hidden dimension | 32 |
| Agrégation intra-step | Mean |
| Learning rate | $10^{-3}$ |
| Weight decay | $10^{-5}$ |
| Gradient clipping | 5.0 |
| `pos_weight` | 1273.0686 |
| Seed | 42 |
| Epochs | 1 |
| Device | CPU |

Une seule variable expérimentale change :

$$
TBPTT=16
$$

au lieu de :

$$
TBPTT=2
$$

Le Train reste exactement :

$$
1 \leq t \leq 334
$$

Aucune donnée de Validation ou de Test n'intervient dans l'apprentissage.

## 15.3 Fenêtres temporelles

Les fenêtres sont alignées comme suit :

$$
1\text{–}16
$$

$$
17\text{–}32
$$

$$
33\text{–}48
$$

$$
\ldots
$$

jusqu'à :

$$
321\text{–}334
$$

Le modèle effectue donc **21 updates d'optimisation** sur l'ensemble du Train.

À l'intérieur d'une fenêtre :

1. les états temporels restent différentiables ;
2. aucune mise à jour de paramètres n'est réalisée avant la fin de la fenêtre ;
3. les losses des différents steps sont agrégées ;
4. un seul `backward()` est exécuté ;
5. les gradients sont clippés ;
6. l'optimiseur met à jour les paramètres ;
7. la mémoire est commitée puis détachée.

## 15.4 Checkpointing

Un checkpoint de reprise sera sauvegardé après chaque fenêtre TBPTT complète.

Chaque checkpoint correspond donc à un état causal propre :

$$
State_{\leq t}
$$

avec un overlay vide.

Un checkpoint final indépendant sera créé après le step 334.

Les artefacts de ce candidat sont séparés de ceux du modèle `TBPTT=2`.

In [88]:
# Configuration du second candidat
train16_config = dict(train_config)

train16_config.update({
    "candidate": "dynamic_core_edge_tbptt16",
    "tbptt_steps": 16,
})

checkpoint16_path = (
    dynamic_artifact_dir
    / "dynamic_core_edge_tbptt16_latest.pt"
)

final_checkpoint16_path = (
    dynamic_artifact_dir
    / "dynamic_core_edge_tbptt16_train_final.pt"
)

history16_path = (
    dynamic_artifact_dir
    / "dynamic_core_edge_tbptt16_train_history.json"
)

config16_path = (
    dynamic_artifact_dir
    / "dynamic_core_edge_tbptt16_config.json"
)

with open(
    config16_path,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        train16_config,
        f,
        indent=2,
        ensure_ascii=False,
    )

print(
    "Candidat :",
    train16_config["candidate"],
)
print(
    "TBPTT :",
    train16_config["tbptt_steps"],
)
print(
    "Train :",
    f"{train16_config['train_start_step']}"
    f"-{train16_config['train_end_step']}",
)
print(
    "Configuration sauvegardée :",
    config16_path.exists(),
)
print(
    "Test utilisé : False"
)

Candidat : dynamic_core_edge_tbptt16
TBPTT : 16
Train : 1-334
Configuration sauvegardée : True
Test utilisé : False


In [89]:
# Reproductibilité initiale
torch.manual_seed(
    train16_config["random_seed"]
)
np.random.seed(
    train16_config["random_seed"]
)

memory_dim = train16_config[
    "memory_dim"
]

# Modèle
train16_model = PilotDynamicModel(
    edge_dim=train16_config["edge_dim"],
    memory_dim=memory_dim,
    hidden_dim=train16_config["hidden_dim"],
)

# Optimiseur
train16_optimizer = torch.optim.AdamW(
    train16_model.parameters(),
    lr=train16_config["learning_rate"],
    weight_decay=train16_config["weight_decay"],
)

# Loss
train16_criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        train16_config["pos_weight"],
        dtype=torch.float32,
    ),
    reduction="sum",
)

if checkpoint16_path.exists():

    restored16 = load_training_checkpoint(
        path=checkpoint16_path,
        model=train16_model,
        optimizer=train16_optimizer,
        memory_dim=memory_dim,
    )

    saved_config16 = restored16[
        "config"
    ]

    critical_keys16 = [
        "feature_set",
        "edge_dim",
        "memory_dim",
        "hidden_dim",
        "aggregation",
        "tbptt_steps",
        "learning_rate",
        "weight_decay",
        "pos_weight",
        "preprocessing_sha256",
    ]

    config16_match = all(
        saved_config16[key]
        == train16_config[key]
        for key in critical_keys16
    )

    if not config16_match:
        raise RuntimeError(
            "Checkpoint TBPTT=16 incompatible."
        )

    train16_memory = restored16[
        "temporal_memory"
    ]

    train16_history = restored16[
        "history"
    ]

    completed16_step = restored16[
        "completed_step"
    ]

    start16_step = (
        completed16_step + 1
    )

    print("Mode : reprise")

else:

    train16_index = SparseTemporalIndex(
        num_nodes=9_073_900,
        initial_capacity=2_100_000,
        growth_size=100_000,
    )

    train16_memory = PilotTemporalMemory(
        temporal_index=train16_index,
        memory_dim=memory_dim,
        initial_capacity=2_100_000,
        growth_size=100_000,
    )

    train16_history = []

    completed16_step = 0
    start16_step = 1

    print("Mode : nouvel entraînement")


print(
    "Prochain step :",
    start16_step,
)
print(
    "Fenêtres terminées :",
    len(train16_history),
)
print(
    "Slots actifs :",
    f"{train16_memory.index.size:,}",
)
print(
    "Overlay vide :",
    len(train16_memory.overlay) == 0,
)
print(
    "RAM disponible :",
    f"{psutil.virtual_memory().available / 1024**3:.2f} GB",
)

Mode : nouvel entraînement
Prochain step : 1
Fenêtres terminées : 0
Slots actifs : 0
Overlay vide : True
RAM disponible : 3.10 GB


In [90]:
tbptt16 = 16

process16 = psutil.Process()

session16_start = time.perf_counter()

if start16_step > 334:

    print(
        "Train TBPTT=16 déjà terminé au step :",
        completed16_step,
    )

else:

    for window_start in range(
        start16_step,
        335,
        tbptt16,
    ):

        window_end = min(
            window_start + tbptt16 - 1,
            334,
        )

        train16_optimizer.zero_grad(
            set_to_none=True
        )

        loss_sum = None

        window_tx = 0
        window_fraud = 0
        window_known_src = 0
        window_known_dst = 0

        max_overlay = 0
        max_rss = 0.0
        min_ram = float("inf")

        window_start_time = (
            time.perf_counter()
        )

        # Forward causal de toute la fenêtre
        for t in range(
            window_start,
            window_end + 1,
        ):

            batch = reader.read_step(t)

            logits, y, info = (
                process_pilot_step(
                    model=train16_model,
                    temporal_memory=train16_memory,
                    batch=batch,
                )
            )

            step_loss = train16_criterion(
                logits,
                y,
            )

            if loss_sum is None:
                loss_sum = step_loss
            else:
                loss_sum = (
                    loss_sum + step_loss
                )

            window_tx += info[
                "transactions"
            ]

            window_fraud += info[
                "frauds"
            ]

            window_known_src += info[
                "known_src"
            ]

            window_known_dst += info[
                "known_dst"
            ]

            overlay_size = len(
                train16_memory.overlay
            )

            rss_gib = (
                process16.memory_info().rss
                / 1024**3
            )

            ram_gib = (
                psutil.virtual_memory().available
                / 1024**3
            )

            max_overlay = max(
                max_overlay,
                overlay_size,
            )

            max_rss = max(
                max_rss,
                rss_gib,
            )

            min_ram = min(
                min_ram,
                ram_gib,
            )

            # Garde-fou
            if ram_gib < 0.70:
                raise MemoryError(
                    f"RAM critique au step {t}: "
                    f"{ram_gib:.2f} GB libres."
                )

        loss = (
            loss_sum / window_tx
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Loss non finie "
                f"{window_start}-{window_end}"
            )

        # Backward
        backward_start = (
            time.perf_counter()
        )

        loss.backward()

        backward_runtime = (
            time.perf_counter()
            - backward_start
        )

        predictor_grad = module_grad_norm(
            train16_model.predictor
        )

        message_grad = module_grad_norm(
            train16_model.message_module
        )

        gru_grad = module_grad_norm(
            train16_model.gru
        )

        total_grad = (
            torch.nn.utils.clip_grad_norm_(
                train16_model.parameters(),
                max_norm=train16_config[
                    "gradient_clip"
                ],
            )
        )

        total_grad_value = float(
            total_grad.detach()
        )

        gradient_values = [
            predictor_grad,
            message_grad,
            gru_grad,
            total_grad_value,
        ]

        if not all(
            np.isfinite(value)
            for value in gradient_values
        ):
            raise RuntimeError(
                f"Gradient non fini "
                f"{window_start}-{window_end}"
            )

        # Update
        train16_optimizer.step()

        # Fin de fenêtre TBPTT
        train16_memory.commit()

        completed16_step = window_end

        window_runtime = (
            time.perf_counter()
            - window_start_time
        )

        history16_item = {
            "start_step": int(
                window_start
            ),
            "end_step": int(
                window_end
            ),
            "transactions": int(
                window_tx
            ),
            "frauds": int(
                window_fraud
            ),
            "loss": float(
                loss.detach()
            ),
            "predictor_grad": float(
                predictor_grad
            ),
            "message_grad": float(
                message_grad
            ),
            "gru_grad": float(
                gru_grad
            ),
            "total_grad_preclip": float(
                total_grad_value
            ),
            "known_src": int(
                window_known_src
            ),
            "known_dst": int(
                window_known_dst
            ),
            "active_slots": int(
                train16_memory.index.size
            ),
            "max_overlay": int(
                max_overlay
            ),
            "max_rss_gib": float(
                max_rss
            ),
            "min_ram_available_gib": float(
                min_ram
            ),
            "backward_seconds": float(
                backward_runtime
            ),
            "runtime_seconds": float(
                window_runtime
            ),
        }

        train16_history.append(
            history16_item
        )

        print(
            f"Steps {window_start:>3}-{window_end:<3} | "
            f"tx={window_tx:,} | "
            f"fraudes={window_fraud} | "
            f"loss={float(loss.detach()):.6f}"
        )

        print(
            f"  grad predictor={predictor_grad:.6f} | "
            f"message={message_grad:.6f} | "
            f"GRU={gru_grad:.6f}"
        )

        print(
            f"  overlay max={max_overlay:,} | "
            f"backward={backward_runtime:.2f}s | "
            f"window={window_runtime:.2f}s"
        )

        print(
            f"  slots={train16_memory.index.size:,} | "
            f"RSS max={max_rss:.2f} GB | "
            f"RAM min={min_ram:.2f} GB"
        )

        # Libérer le graphe avant sauvegarde
        del batch
        del logits
        del y
        del loss
        del loss_sum

        gc.collect()

        # Checkpoint après chaque fenêtre propre
        save_training_checkpoint(
            path=checkpoint16_path,
            model=train16_model,
            optimizer=train16_optimizer,
            temporal_memory=train16_memory,
            completed_step=completed16_step,
            history=train16_history,
            config=train16_config,
        )

        save_history_json(
            history16_path,
            train16_history,
        )

        print(
            f"  checkpoint sauvegardé "
            f"au step {completed16_step}"
        )
        print()


print(
    "Dernier step terminé :",
    completed16_step,
)

print(
    "Fenêtres TBPTT :",
    len(train16_history),
)

print(
    "Runtime session :",
    f"{time.perf_counter() - session16_start:.2f} s",
)

print(
    "RAM disponible :",
    f"{psutil.virtual_memory().available / 1024**3:.2f} GB",
)

Steps   1-16  | tx=348,191 | fraudes=197 | loss=1.298555
  grad predictor=1.646866 | message=0.006711 | GRU=0.011692
  overlay max=154,461 | backward=46.35s | window=67.77s
  slots=154,461 | RSS max=1.74 GB | RAM min=1.71 GB
  checkpoint sauvegardé au step 16

Steps  17-32  | tx=228,183 | fraudes=152 | loss=1.372294
  grad predictor=1.906167 | message=0.001812 | GRU=0.003926
  overlay max=128,101 | backward=22.92s | window=32.15s
  slots=250,739 | RSS max=1.26 GB | RAM min=3.02 GB
  checkpoint sauvegardé au step 32

Steps  33-48  | tx=453,119 | fraudes=231 | loss=1.238916
  grad predictor=2.786064 | message=0.005691 | GRU=0.009782
  overlay max=239,206 | backward=61.53s | window=80.19s
  slots=439,283 | RSS max=2.03 GB | RAM min=2.27 GB
  checkpoint sauvegardé au step 48

Steps  49-64  | tx=942 | fraudes=182 | loss=191.551193
  grad predictor=713.426429 | message=0.026672 | GRU=0.029816
  overlay max=941 | backward=0.01s | window=1.25s
  slots=439,900 | RSS max=0.62 GB | RAM min=3.45 G

In [91]:
train16_tx = sum(
    item["transactions"]
    for item in train16_history
)

train16_frauds = sum(
    item["frauds"]
    for item in train16_history
)

train16_losses = np.array(
    [
        item["loss"]
        for item in train16_history
    ],
    dtype=np.float64,
)

train16_message_grads = np.array(
    [
        item["message_grad"]
        for item in train16_history
    ],
    dtype=np.float64,
)

train16_gru_grads = np.array(
    [
        item["gru_grad"]
        for item in train16_history
    ],
    dtype=np.float64,
)

active16_size = (
    train16_memory.index.size
)

active16_last_update = (
    train16_memory.index.last_update[
        :active16_size
    ]
)

print(
    "Step 334 atteint :",
    completed16_step == 334,
)

print(
    "Transactions exactes :",
    train16_tx == 4_771_387,
    f"({train16_tx:,})",
)

print(
    "Fraudes exactes :",
    train16_frauds == 3_745,
    f"({train16_frauds:,})",
)

print(
    "Slots exacts :",
    active16_size == 2_035_605,
    f"({active16_size:,})",
)

print(
    "Losses finies :",
    bool(
        np.isfinite(
            train16_losses
        ).all()
    ),
)

print(
    "Gradients message finis :",
    bool(
        np.isfinite(
            train16_message_grads
        ).all()
    ),
)

print(
    "Gradients GRU finis :",
    bool(
        np.isfinite(
            train16_gru_grads
        ).all()
    ),
)

print(
    "Mémoire finie :",
    bool(
        torch.isfinite(
            train16_memory.memory[
                :active16_size
            ]
        ).all()
    ),
)

print(
    "last_update valide :",
    bool(
        np.all(
            active16_last_update >= 1
        )
        and np.all(
            active16_last_update <= 334
        )
    ),
)

print(
    "Overlay vide :",
    len(train16_memory.overlay) == 0,
)

print()
print(
    "Loss min/médiane/max :",
    f"{train16_losses.min():.6f}",
    "/",
    f"{np.median(train16_losses):.6f}",
    "/",
    f"{train16_losses.max():.6f}",
)

print(
    "Gradient message min/médiane/max :",
    f"{train16_message_grads.min():.8f}",
    "/",
    f"{np.median(train16_message_grads):.8f}",
    "/",
    f"{train16_message_grads.max():.8f}",
)

print(
    "Gradient GRU min/médiane/max :",
    f"{train16_gru_grads.min():.8f}",
    "/",
    f"{np.median(train16_gru_grads):.8f}",
    "/",
    f"{train16_gru_grads.max():.8f}",
)

Step 334 atteint : True
Transactions exactes : True (4,771,387)
Fraudes exactes : True (3,745)
Slots exacts : True (2,035,605)
Losses finies : True
Gradients message finis : True
Gradients GRU finis : True
Mémoire finie : True
last_update valide : True
Overlay vide : True

Loss min/médiane/max : 0.936243 / 1.298555 / 995.082153
Gradient message min/médiane/max : 0.00000000 / 0.00202211 / 0.02667243
Gradient GRU min/médiane/max : 0.00000000 / 0.00312474 / 0.02981632


## 15.5 Résultats de l'entraînement avec TBPTT=16

Le second candidat dynamique a été entraîné sur l'intégralité du Train avec une portée temporelle de :

$$
TBPTT=16
$$

L'entraînement couvre exactement :

- **334 steps** ;
- **4 771 387 transactions** ;
- **3 745 fraudes** ;
- **21 fenêtres TBPTT**.

Tous les contrôles d'intégrité sont satisfaits :

- step 334 atteint : **True** ;
- nombre de transactions exact : **True** ;
- nombre de fraudes exact : **True** ;
- nombre de slots exact : **2 035 605** ;
- losses finies : **True** ;
- gradients temporels finis : **True** ;
- mémoire finie : **True** ;
- `last_update` valide : **True** ;
- overlay vide : **True**.

Le modèle final correspond donc à un état causal :

$$
State_{\leq334}
$$

construit exclusivement à partir du Train.

### Stabilité de la loss

Les losses par fenêtre présentent :

| Statistique | Loss |
| --- | ---: |
| Minimum | 0.936243 |
| Médiane | 1.298555 |
| Maximum | 995.082153 |

Comme pour le premier candidat, les valeurs extrêmes correspondent aux fenêtres contenant très peu de transactions et une proportion exceptionnellement élevée de fraudes.

La loss reste néanmoins finie sur l'ensemble du parcours.

### Renforcement du signal temporel

Pour le module de message :

| Statistique | Gradient |
| --- | ---: |
| Minimum | 0.00000000 |
| Médiane | **0.00202211** |
| Maximum | 0.02667243 |

Pour la GRU :

| Statistique | Gradient |
| --- | ---: |
| Minimum | 0.00000000 |
| Médiane | **0.00312474** |
| Maximum | 0.02981632 |

À titre de comparaison, avec `TBPTT=2`, les médianes étaient :

$$
median(\nabla Message)=0.00025379
$$

et :

$$
median(\nabla GRU)=0.00042155
$$

Le passage à `TBPTT=16` augmente donc fortement l'intensité du signal transmis aux composants temporels.

La médiane du gradient du module de message est approximativement multipliée par 8, tandis que celle de la GRU est multipliée par environ 7,4.

Cette évolution est cohérente avec l'augmentation de la couverture temporelle effective observée précédemment.

### Coût computationnel

L'entraînement complet nécessite :

$$
499.05 \text{ secondes}
$$

soit environ :

$$
8 \text{ minutes } 19 \text{ secondes}
$$

sur CPU.

Le premier candidat `TBPTT=2` nécessitait environ 202,33 secondes.

L'augmentation de la portée temporelle multiplie donc le coût d'entraînement par environ :

$$
\frac{499.05}{202.33}
\approx
2.47
$$

Cette hausse reste compatible avec l'environnement local.

Aucune fenêtre n'a atteint le seuil critique de RAM défini lors du benchmark.

### Conclusion technique

Le second candidat satisfait désormais deux conditions qui n'étaient pas réunies simultanément avec `TBPTT=2` :

1. une portée temporelle nettement mieux alignée sur les réapparitions observées dans PaySim ;
2. une propagation beaucoup plus régulière et plus forte des gradients vers le module de message et la GRU.

La question restante n'est donc plus la faisabilité technique.

Elle devient exclusivement prédictive :

> Cette meilleure propagation temporelle améliore-t-elle réellement la PR-AUC sur la Validation ?

Cette question sera tranchée sur exactement les mêmes steps 335 à 392 que pour les candidats précédents.

In [92]:
message16_zero = (
    train16_message_grads <= 1e-12
)

gru16_zero = (
    train16_gru_grads <= 1e-12
)

both16_zero = (
    message16_zero
    & gru16_zero
)

print(
    "Fenêtres TBPTT=16 :",
    len(train16_history),
)

print(
    "Gradients temporels nuls :",
    int(both16_zero.sum()),
    f"({100 * both16_zero.mean():.2f} %)",
)

print(
    "Gradients temporels non nuls :",
    int((~both16_zero).sum()),
    f"({100 * (~both16_zero).mean():.2f} %)",
)

if both16_zero.any():
    print()
    print("Fenêtres concernées :")

    for item, is_zero in zip(
        train16_history,
        both16_zero,
    ):
        if is_zero:
            print(
                f"  {item['start_step']}-"
                f"{item['end_step']} | "
                f"tx={item['transactions']:,} | "
                f"fraudes={item['frauds']}"
            )

Fenêtres TBPTT=16 : 21
Gradients temporels nuls : 2 (9.52 %)
Gradients temporels non nuls : 19 (90.48 %)

Fenêtres concernées :
  65-80 | tx=210 | fraudes=210
  97-112 | tx=847 | fraudes=172


In [93]:
# Sauvegarder le checkpoint final TBPTT=16
save_training_checkpoint(
    path=final_checkpoint16_path,
    model=train16_model,
    optimizer=train16_optimizer,
    temporal_memory=train16_memory,
    completed_step=334,
    history=train16_history,
    config=train16_config,
)

# Calculer son SHA-256
sha256_16 = hashlib.sha256()

with open(
    final_checkpoint16_path,
    "rb",
) as f:
    for block in iter(
        lambda: f.read(
            8 * 1024 * 1024
        ),
        b"",
    ):
        sha256_16.update(block)

final_checkpoint16_sha256 = (
    sha256_16.hexdigest()
)

print(
    "Checkpoint final existe :",
    final_checkpoint16_path.exists(),
)

print(
    "Taille checkpoint :",
    f"{final_checkpoint16_path.stat().st_size / 1024**2:.2f} MiB",
)

print(
    "Completed step :",
    334,
)

print(
    "Historique existe :",
    history16_path.exists(),
)

print(
    "SHA-256 :",
    final_checkpoint16_sha256,
)

Checkpoint final existe : True
Taille checkpoint : 139.97 MiB
Completed step : 334
Historique existe : True
SHA-256 : 1974e607e00ba918431a9b529a61a9d215f1128824019c606516de22c9027350


## 15.6 Validation finale et artefact du candidat TBPTT=16

Le second candidat dynamique a terminé l'intégralité du Train jusqu'au step 334.

Sur les **21 fenêtres TBPTT**, les logs d'entraînement montrent que seules deux fenêtres présentent simultanément un gradient nul pour le module de message et la GRU :

- steps **65–80** ;
- steps **97–112**.

Cela représente :

$$
\frac{2}{21}\times100 \approx 9.52\%
$$

des fenêtres.

Ainsi, environ :

$$
90.48\%
$$

des fenêtres transmettent effectivement un signal d'apprentissage aux composants temporels.

Cette proportion est nettement supérieure à celle observée avec `TBPTT=2`, pour lequel **40,12 %** des fenêtres présentaient un gradient temporel nul.

Le checkpoint final du candidat `TBPTT=16` a été sauvegardé avec les caractéristiques suivantes :

- dernier step intégré : **334** ;
- taille : **139,97 MiB** ;
- historique d'entraînement sauvegardé : **True** ;
- SHA-256 :

```text
1974e607e00ba918431a9b529a61a9d215f1128824019c606516de22c9027350
```

Ce checkpoint représente exactement :

$$
State_{\leq334}
$$

et constitue le point de départ immuable de la Validation du second candidat.

**Entraînement TBPTT=16 validé.**

# 16. Validation du Dynamic GNN avec TBPTT=16

## 16.1 Objectif

Le second candidat dynamique a été entraîné avec une portée temporelle plus longue :

$$
TBPTT=16
$$

contre :

$$
TBPTT=2
$$

pour le premier candidat.

Cette modification augmente la proportion de fraudes avec destination connue réellement traversables par le gradient de :

$$
4.37\%
$$

à :

$$
29.62\%
$$

La Validation doit maintenant déterminer si cette amélioration de la propagation temporelle se traduit par un meilleur ranking des fraudes.

Le modèle est évalué sur exactement le même split :

$$
335 \leq t \leq 392
$$

contenant :

- **792 406 transactions** ;
- **638 fraudes**.

Les poids du modèle restent entièrement figés.

Seule la mémoire temporelle évolue causalement après chaque step.

La métrique principale reste :

$$
PR\text{-}AUC
$$

La comparaison principale sera :

$$
Dynamic_{TBPTT16}
\quad vs \quad
Dynamic_{TBPTT2}
\quad vs \quad
Control_{NoMemory}
$$

Aucune donnée du Test n'est utilisée.

In [94]:
# Recharger le candidat TBPTT=16
val16_model = PilotDynamicModel(
    edge_dim=13,
    memory_dim=8,
    hidden_dim=32,
)

val16_loader_optimizer = torch.optim.AdamW(
    val16_model.parameters(),
    lr=1e-3,
    weight_decay=1e-5,
)

val16_restored = load_training_checkpoint(
    path=final_checkpoint16_path,
    model=val16_model,
    optimizer=val16_loader_optimizer,
    memory_dim=8,
)

val16_memory = val16_restored[
    "temporal_memory"
]

del val16_loader_optimizer

# Figer les poids
val16_model.eval()

for parameter in val16_model.parameters():
    parameter.requires_grad_(False)

active16_size = val16_memory.index.size

last16 = val16_memory.index.last_update[
    :active16_size
]

print(
    "Completed step chargé :",
    val16_restored["completed_step"],
)

print(
    "Slots initiaux :",
    f"{active16_size:,}",
)

print(
    "last_update max :",
    int(last16.max()),
)

print(
    "Paramètres entraînables :",
    sum(
        p.requires_grad
        for p in val16_model.parameters()
    ),
)

print(
    "Overlay vide :",
    len(val16_memory.overlay) == 0,
)

print(
    "RAM disponible :",
    f"{psutil.virtual_memory().available / 1024**3:.2f} GB",
)

Completed step chargé : 334
Slots initiaux : 2,035,605
last_update max : 334
Paramètres entraînables : 0
Overlay vide : True
RAM disponible : 2.60 GB


In [95]:
val16_scores_blocks = []
val16_labels_blocks = []
val16_steps_blocks = []

val16_src_known_blocks = []
val16_dst_known_blocks = []

val16_start_time = time.perf_counter()

val16_tx = 0
val16_frauds = 0

# Snapshot des paramètres
val16_params_before = {
    name: parameter.detach().clone()
    for name, parameter
    in val16_model.named_parameters()
}

for t in range(335, 393):

    step_start = time.perf_counter()

    batch = reader.read_step(t)

    # Known/New strictement avant E_t
    _, src_known, _ = (
        val16_memory.index.lookup(
            batch["src"],
            t=t,
        )
    )

    _, dst_known, _ = (
        val16_memory.index.lookup(
            batch["dst"],
            t=t,
        )
    )

    with torch.no_grad():

        logits, y, info = process_pilot_step(
            model=val16_model,
            temporal_memory=val16_memory,
            batch=batch,
        )

        scores = torch.sigmoid(
            logits
        ).cpu().numpy().astype(
            np.float32,
            copy=False,
        )

    # Intégration après prédiction
    val16_memory.commit()

    val16_scores_blocks.append(
        scores
    )

    val16_labels_blocks.append(
        batch["label"].astype(
            np.int8,
            copy=True,
        )
    )

    val16_steps_blocks.append(
        np.full(
            len(scores),
            t,
            dtype=np.int16,
        )
    )

    val16_src_known_blocks.append(
        src_known.astype(
            np.bool_,
            copy=True,
        )
    )

    val16_dst_known_blocks.append(
        dst_known.astype(
            np.bool_,
            copy=True,
        )
    )

    val16_tx += len(scores)

    val16_frauds += int(
        batch["label"].sum()
    )

    if (
        t == 335
        or t % 10 == 0
        or t == 392
    ):
        print(
            f"Step {t}/392 | "
            f"tx={len(scores):,} | "
            f"fraudes={int(batch['label'].sum())} | "
            f"known_dst={int(dst_known.sum()):,} | "
            f"runtime="
            f"{time.perf_counter() - step_start:.2f}s | "
            f"RAM libre="
            f"{psutil.virtual_memory().available / 1024**3:.2f} GB"
        )

print()
print(
    "Transactions Validation :",
    f"{val16_tx:,}",
)

print(
    "Fraudes Validation :",
    f"{val16_frauds:,}",
)

print(
    "Runtime Validation :",
    f"{time.perf_counter() - val16_start_time:.2f} s",
)

Step 335/392 | tx=10,456 | fraudes=4 | known_dst=4,386 | runtime=1.70s | RAM libre=2.62 GB
Step 340/392 | tx=10 | fraudes=10 | known_dst=3 | runtime=0.01s | RAM libre=2.61 GB
Step 350/392 | tx=28,245 | fraudes=14 | known_dst=16,592 | runtime=1.42s | RAM libre=2.56 GB
Step 360/392 | tx=3,508 | fraudes=8 | known_dst=1,453 | runtime=0.09s | RAM libre=2.50 GB
Step 370/392 | tx=25,429 | fraudes=18 | known_dst=14,831 | runtime=0.43s | RAM libre=2.49 GB
Step 380/392 | tx=37,693 | fraudes=14 | known_dst=21,142 | runtime=0.70s | RAM libre=2.58 GB
Step 390/392 | tx=12 | fraudes=12 | known_dst=3 | runtime=0.00s | RAM libre=2.58 GB
Step 392/392 | tx=10 | fraudes=10 | known_dst=4 | runtime=0.00s | RAM libre=2.58 GB

Transactions Validation : 792,406
Fraudes Validation : 638
Runtime Validation : 19.50 s


In [96]:
# Consolider les prédictions
val16_scores = np.concatenate(
    val16_scores_blocks
)

val16_labels = np.concatenate(
    val16_labels_blocks
)

val16_steps = np.concatenate(
    val16_steps_blocks
)

val16_src_known = np.concatenate(
    val16_src_known_blocks
)

val16_dst_known = np.concatenate(
    val16_dst_known_blocks
)

# Vérifier les paramètres
val16_params_unchanged = all(
    torch.equal(
        val16_params_before[name],
        parameter.detach(),
    )
    for name, parameter
    in val16_model.named_parameters()
)

# Métriques
val16_pr_auc = average_precision_score(
    val16_labels,
    val16_scores,
)

val16_roc_auc = roc_auc_score(
    val16_labels,
    val16_scores,
)

print(
    "Transactions exactes :",
    len(val16_scores) == 792_406,
    f"({len(val16_scores):,})",
)

print(
    "Fraudes exactes :",
    int(val16_labels.sum()) == 638,
    f"({int(val16_labels.sum()):,})",
)

print(
    "Steps exacts :",
    (
        val16_steps.min() == 335
        and val16_steps.max() == 392
    ),
)

print(
    "Scores finis :",
    bool(
        np.isfinite(
            val16_scores
        ).all()
    ),
)

print(
    "Paramètres inchangés :",
    val16_params_unchanged,
)

print(
    "Overlay vide :",
    len(val16_memory.overlay) == 0,
)

print(
    "last_update max :",
    int(
        val16_memory.index.last_update[
            :val16_memory.index.size
        ].max()
    ),
)

print()
print(
    "PR-AUC TBPTT=16 :",
    f"{val16_pr_auc:.6f}",
)

print(
    "ROC-AUC TBPTT=16 :",
    f"{val16_roc_auc:.6f}",
)

print()
print(
    "PR-AUC TBPTT=2  :",
    f"{val_pr_auc:.6f}",
)

print(
    "PR-AUC NoMemory :",
    f"{control_val_pr_auc:.6f}",
)

print()
print(
    "Delta 16 - 2 :",
    f"{val16_pr_auc - val_pr_auc:+.6f}",
)

print(
    "Delta 16 - NoMemory :",
    f"{val16_pr_auc - control_val_pr_auc:+.6f}",
)

Transactions exactes : True (792,406)
Fraudes exactes : True (638)
Steps exacts : True
Scores finis : True
Paramètres inchangés : True
Overlay vide : True
last_update max : 392

PR-AUC TBPTT=16 : 0.395144
ROC-AUC TBPTT=16 : 0.939077

PR-AUC TBPTT=2  : 0.547378
PR-AUC NoMemory : 0.606575

Delta 16 - 2 : -0.152233
Delta 16 - NoMemory : -0.211431


In [97]:
val16_predictions_path = (
    dynamic_artifact_dir
    / "dynamic_core_edge_tbptt16_validation_predictions.npz"
)

val16_metrics_path = (
    dynamic_artifact_dir
    / "dynamic_core_edge_tbptt16_validation_metrics.json"
)

np.savez_compressed(
    val16_predictions_path,
    scores=val16_scores,
    labels=val16_labels,
    steps=val16_steps,
    src_known=val16_src_known,
    dst_known=val16_dst_known,
)

val16_metrics = {
    "split": "validation",
    "start_step": 335,
    "end_step": 392,
    "transactions": int(
        len(val16_scores)
    ),
    "frauds": int(
        val16_labels.sum()
    ),
    "pr_auc": float(
        val16_pr_auc
    ),
    "roc_auc": float(
        val16_roc_auc
    ),
    "tbptt": 16,
    "weights_frozen": bool(
        val16_params_unchanged
    ),
    "train_checkpoint_sha256":
        final_checkpoint16_sha256,
    "test_used": False,
}

with open(
    val16_metrics_path,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        val16_metrics,
        f,
        indent=2,
        ensure_ascii=False,
    )

print(
    "Prédictions sauvegardées :",
    val16_predictions_path.exists(),
)

print(
    "Métriques sauvegardées :",
    val16_metrics_path.exists(),
)

print(
    "Test utilisé :",
    val16_metrics["test_used"],
)

Prédictions sauvegardées : True
Métriques sauvegardées : True
Test utilisé : False


In [98]:
print(
    "Transactions exactes :",
    len(val16_scores) == 792_406,
    f"({len(val16_scores):,})",
)

print(
    "Fraudes exactes :",
    int(val16_labels.sum()) == 638,
    f"({int(val16_labels.sum()):,})",
)

print(
    "Scores finis :",
    bool(np.isfinite(val16_scores).all()),
)

print(
    "Paramètres inchangés :",
    val16_params_unchanged,
)

print(
    "Overlay vide :",
    len(val16_memory.overlay) == 0,
)

print(
    "last_update max :",
    int(
        val16_memory.index.last_update[
            :val16_memory.index.size
        ].max()
    ),
)

print()

print(
    "PR-AUC TBPTT=16 :",
    f"{val16_pr_auc:.6f}",
)

print(
    "ROC-AUC TBPTT=16 :",
    f"{val16_roc_auc:.6f}",
)

print(
    "PR-AUC TBPTT=2  :",
    f"{val_pr_auc:.6f}",
)

print(
    "PR-AUC NoMemory :",
    f"{control_val_pr_auc:.6f}",
)

print()

print(
    "Delta 16 - 2 :",
    f"{val16_pr_auc - val_pr_auc:+.6f}",
)

print(
    "Delta 16 - NoMemory :",
    f"{val16_pr_auc - control_val_pr_auc:+.6f}",
)

Transactions exactes : True (792,406)
Fraudes exactes : True (638)
Scores finis : True
Paramètres inchangés : True
Overlay vide : True
last_update max : 392

PR-AUC TBPTT=16 : 0.395144
ROC-AUC TBPTT=16 : 0.939077
PR-AUC TBPTT=2  : 0.547378
PR-AUC NoMemory : 0.606575

Delta 16 - 2 : -0.152233
Delta 16 - NoMemory : -0.211431


## 16.2 Résultat de la Validation TBPTT=16

Le candidat `TBPTT=16` a été évalué causalement sur exactement les mêmes steps de Validation que les modèles précédents :

$$
335 \leq t \leq 392
$$

L'évaluation couvre exactement :

- **792 406 transactions** ;
- **638 fraudes**.

Tous les contrôles d'intégrité sont satisfaits :

- transactions exactes : **True** ;
- fraudes exactes : **True** ;
- scores finis : **True** ;
- paramètres inchangés : **True** ;
- overlay vide : **True** ;
- `last_update` maximal : **392**.

Les poids sont donc restés totalement figés pendant la Validation.

### Performances

Le candidat obtient :

$$
PR\text{-}AUC_{TBPTT16}
=
0.395144
$$

et :

$$
ROC\text{-}AUC_{TBPTT16}
=
0.939077
$$

Les trois configurations neuronales peuvent désormais être comparées :

| Modèle | Validation PR-AUC | Validation ROC-AUC |
| --- | ---: | ---: |
| Dynamic GNN — TBPTT=16 | 0.395144 | 0.939077 |
| Dynamic GNN — TBPTT=2 | 0.547378 | 0.980523 |
| **Contrôle sans mémoire** | **0.606575** | **0.990075** |

L'augmentation de la portée temporelle entraîne une dégradation de :

$$
0.395144 - 0.547378
=
-0.152233
$$

par rapport au premier Dynamic GNN.

Par rapport au contrôle sans mémoire :

$$
0.395144 - 0.606575
=
-0.211431
$$

### Interprétation

Le passage de `TBPTT=2` à `TBPTT=16` a pourtant produit un signal temporel beaucoup plus régulier :

- meilleure couverture des réapparitions ;
- gradients du module de message plus importants ;
- gradients de la GRU plus importants ;
- seulement environ 9,52 % de fenêtres sans gradient temporel.

Cependant, cette amélioration de l'apprentissage de la mémoire ne se traduit pas par une amélioration prédictive.

Au contraire, la PR-AUC diminue fortement.

Ce résultat montre qu'une meilleure propagation du gradient dans le temps n'est pas suffisante lorsque l'état historique appris n'est pas suffisamment discriminant pour la tâche de fraude.

Plusieurs propriétés de PaySim peuvent expliquer cette difficulté :

- quasi-totalité des sources en cold-start ;
- topologie extrêmement peu profonde ;
- très faible recouvrement entre rôles source et destination ;
- historique principalement concentré sur les destinations ;
- nombreuses réapparitions de destinations non nécessairement liées au comportement frauduleux ;
- forte capacité discriminante des features transactionnelles elles-mêmes.

Ces éléments constituent des hypothèses d'interprétation et non des causalités démontrées individuellement.

### Conclusion expérimentale

Les résultats obtenus sur la Validation sont suffisamment cohérents pour arrêter l'exploration de cette formulation de mémoire.

Aucun test supplémentaire de :

- longueur TBPTT ;
- gate ;
- dimension mémoire ;
- profondeur de GRU ;
- variante de la même mémoire destination-centric

ne sera réalisé.

Multiplier ces expériences augmenterait le risque de sur-adaptation à la Validation sans apporter une question scientifique nouvelle.

La conclusion retenue est donc :

> **La mémoire temporelle destination-centric inspirée de TGN, dans la formulation testée sur PaySim, n'apporte pas de gain prédictif par rapport au contrôle neuronal sans mémoire.**

Le meilleur candidat dynamique reste le modèle `TBPTT=2` avec :

$$
PR\text{-}AUC_{Val}=0.547378
$$

Le meilleur modèle neuronal utilisant les Graph Core Edge est toutefois le contrôle sans mémoire :

$$
PR\text{-}AUC_{Val}=0.606575
$$

Cette conclusion négative sur la mémoire constitue un résultat scientifique du projet et sera conservée dans l'analyse finale.

**Section 16 validée.**

# 17. Sélection finale des modèles avant le Test

## 17.1 Décision basée exclusivement sur la Validation

Les expériences réalisées sur la Validation conduisent au classement suivant :

| Modèle | Rôle | PR-AUC Validation |
| --- | --- | ---: |
| GraphSAGE Core Edge | baseline GNN statique | 0.240136 |
| Dynamic TBPTT=16 | variante temporelle rejetée | 0.395144 |
| Dynamic TBPTT=2 | **meilleur candidat dynamique** | 0.547378 |
| NoMemory Control | **contrôle neuronal** | 0.606575 |
| LightGBM Core | référence tabulaire | 0.929704 |

Le candidat `TBPTT=16` est définitivement rejeté avant le Test.

Deux modèles seront conservés pour l'évaluation finale :

1. **Dynamic TBPTT=2**, car il constitue le meilleur modèle utilisant réellement une mémoire temporelle ;
2. **NoMemory Control**, car il permet de mesurer si l'effet observé du modèle dynamique se maintient hors Validation.

Ces rôles sont figés avant toute consultation du Test.

Les résultats Test ne serviront donc pas à sélectionner l'un des deux modèles.

## 17.2 Seuils

Le Dynamic GNN `TBPTT=2` possède déjà un seuil sélectionné exclusivement sur la Validation :

$$
\tau^*_{Dynamic}
=
0.9800333381
$$

Le contrôle sans mémoire doit maintenant recevoir son propre seuil, également déterminé exclusivement sur la Validation et avec exactement le même critère :

$$
\underset{\tau}{\operatorname{argmax}}\;F_1(\tau)
$$

Une fois ce seuil sauvegardé, aucune décision de modèle, de seuil ou d'architecture ne pourra être modifiée à partir du Test.

In [99]:
# Courbe Precision-Recall du contrôle
control_precision_curve, control_recall_curve, control_thresholds = (
    precision_recall_curve(
        control_val_labels,
        control_val_scores,
    )
)

control_precision_thr = (
    control_precision_curve[:-1]
)

control_recall_thr = (
    control_recall_curve[:-1]
)

control_denominator = (
    control_precision_thr
    + control_recall_thr
)

control_f1_curve = np.divide(
    2
    * control_precision_thr
    * control_recall_thr,
    control_denominator,
    out=np.zeros_like(
        control_denominator
    ),
    where=control_denominator > 0,
)

control_best_idx = int(
    np.argmax(
        control_f1_curve
    )
)

control_threshold = float(
    control_thresholds[
        control_best_idx
    ]
)

control_best_precision = float(
    control_precision_thr[
        control_best_idx
    ]
)

control_best_recall = float(
    control_recall_thr[
        control_best_idx
    ]
)

control_best_f1 = float(
    control_f1_curve[
        control_best_idx
    ]
)

print(
    "Seuil contrôle :",
    f"{control_threshold:.10f}",
)

print(
    "Precision :",
    f"{control_best_precision:.6f}",
)

print(
    "Recall :",
    f"{control_best_recall:.6f}",
)

print(
    "F1 :",
    f"{control_best_f1:.6f}",
)

Seuil contrôle : 0.9987996817
Precision : 0.734513
Recall : 0.520376
F1 : 0.609174


In [100]:
control_val_pred = (
    control_val_scores
    >= control_threshold
).astype(np.int8)

control_tn, control_fp, control_fn, control_tp = (
    confusion_matrix(
        control_val_labels,
        control_val_pred,
        labels=[0, 1],
    ).ravel()
)

print("Validation contrôle")
print()

print(
    "TP :",
    f"{control_tp:,}",
)
print(
    "FP :",
    f"{control_fp:,}",
)
print(
    "FN :",
    f"{control_fn:,}",
)
print(
    "TN :",
    f"{control_tn:,}",
)

print()

print(
    "Precision :",
    f"{precision_score(control_val_labels, control_val_pred):.6f}",
)

print(
    "Recall :",
    f"{recall_score(control_val_labels, control_val_pred):.6f}",
)

print(
    "F1 :",
    f"{f1_score(control_val_labels, control_val_pred):.6f}",
)

print(
    "PR-AUC :",
    f"{control_val_pr_auc:.6f}",
)

Validation contrôle

TP : 332
FP : 120
FN : 306
TN : 791,648

Precision : 0.734513
Recall : 0.520376
F1 : 0.609174
PR-AUC : 0.606575


In [101]:
model_selection_path = (
    dynamic_artifact_dir
    / "dynamic_gnn_model_selection.json"
)

control_threshold_path = (
    dynamic_artifact_dir
    / "no_memory_control_threshold.json"
)

control_threshold_metadata = {
    "selection_split": "validation",
    "selection_metric": "f1",
    "threshold": float(
        control_threshold
    ),
    "precision": float(
        control_best_precision
    ),
    "recall": float(
        control_best_recall
    ),
    "f1": float(
        control_best_f1
    ),
    "pr_auc": float(
        control_val_pr_auc
    ),
    "test_used_for_selection": False,
}

with open(
    control_threshold_path,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        control_threshold_metadata,
        f,
        indent=2,
        ensure_ascii=False,
    )


model_selection = {
    "selection_split": "validation",

    "best_dynamic_model":
        "dynamic_core_edge_tbptt2",

    "best_dynamic_val_pr_auc":
        float(val_pr_auc),

    "dynamic_threshold":
        float(val_threshold),

    "control_model":
        "no_memory_control",

    "control_val_pr_auc":
        float(control_val_pr_auc),

    "control_threshold":
        float(control_threshold),

    "rejected_dynamic_model":
        "dynamic_core_edge_tbptt16",

    "rejected_dynamic_val_pr_auc":
        float(val16_pr_auc),

    "test_models": [
        "dynamic_core_edge_tbptt2",
        "no_memory_control",
    ],

    "test_used_for_selection": False,
}

with open(
    model_selection_path,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        model_selection,
        f,
        indent=2,
        ensure_ascii=False,
    )

print(
    "Seuil contrôle sauvegardé :",
    control_threshold_path.exists(),
)

print(
    "Sélection sauvegardée :",
    model_selection_path.exists(),
)

print(
    "Meilleur Dynamic :",
    model_selection[
        "best_dynamic_model"
    ],
)

print(
    "Modèles autorisés au Test :",
    model_selection[
        "test_models"
    ],
)

print(
    "Test utilisé pour sélection :",
    model_selection[
        "test_used_for_selection"
    ],
)

Seuil contrôle sauvegardé : True
Sélection sauvegardée : True
Meilleur Dynamic : dynamic_core_edge_tbptt2
Modèles autorisés au Test : ['dynamic_core_edge_tbptt2', 'no_memory_control']
Test utilisé pour sélection : False


## 17.3 Décision finale avant le Test

La sélection expérimentale est désormais entièrement terminée sur la Validation.

Les modèles conservés sont :

| Modèle | Rôle | PR-AUC Validation | Seuil Validation |
| --- | --- | ---: | ---: |
| Dynamic GNN — TBPTT=2 | meilleur candidat dynamique | 0.547378 | 0.9800333381 |
| NoMemory Control | contrôle neuronal sans mémoire | **0.606575** | **0.9987996817** |

Le candidat `TBPTT=16` est rejeté avant toute consultation du Test.

Le seuil du contrôle a été sélectionné exclusivement sur la Validation en maximisant le F1-score :

$$
\tau^*_{Control}=0.9987996817
$$

Il obtient sur Validation :

- Precision : **0.734513** ;
- Recall : **0.520376** ;
- F1 : **0.609174** ;
- TP : **332** ;
- FP : **120** ;
- FN : **306**.

Le Dynamic GNN conserve son seuil précédemment sélectionné :

$$
\tau^*_{Dynamic}=0.9800333381
$$

Les rôles des deux modèles sont maintenant figés :

1. le Dynamic GNN mesure la généralisation du meilleur modèle utilisant effectivement une mémoire temporelle ;
2. le contrôle sans mémoire constitue l'ablation de référence permettant d'évaluer la valeur réelle de cette mémoire.

Le fichier de sélection confirme :

`Test utilisé pour sélection : False`

À partir de maintenant, aucune architecture, aucun seuil et aucun hyperparamètre ne pourra être modifié à partir des résultats du Test.

**Section 17 validée.**

# 18. Évaluation finale sur le Test

## 18.1 Objectif

Le Test constitue la dernière phase d'évaluation du notebook.

Il couvre :

$$
393 \leq t \leq 743
$$

soit :

$$
798\,827
$$

transactions et :

$$
3\,830
$$

fraudes.

Cette phase intervient uniquement après :

- la sélection des architectures ;
- les ablations ;
- le choix du meilleur candidat dynamique ;
- le choix du contrôle ;
- la sélection des seuils sur Validation.

Aucune décision expérimentale ne sera prise à partir du Test.

## 18.2 Modèles évalués

Deux modèles préspécifiés sont évalués :

### Dynamic GNN — TBPTT=2

Il constitue le meilleur candidat utilisant réellement une mémoire temporelle.

Son seuil est figé à :

$$
\tau^*_{Dynamic}=0.9800333381
$$

### NoMemory Control

Il constitue le contrôle neuronal sans mémoire temporelle.

Son seuil est figé à :

$$
\tau^*_{Control}=0.9987996817
$$

## 18.3 Continuité temporelle du Dynamic GNN

Le checkpoint d'entraînement du Dynamic GNN correspond à :

$$
State_{\leq334}
$$

Cependant, au début du Test, l'historique disponible doit comprendre également toute la Validation.

La mémoire doit donc évoluer, avec des poids figés, selon :

$$
State_{\leq334}
\rightarrow
E_{335}
\rightarrow
\cdots
\rightarrow
E_{392}
\rightarrow
State_{\leq392}
$$

puis :

$$
State_{\leq392}
\rightarrow
Prediction(E_{393})
$$

Les labels de Validation ne sont jamais utilisés pour mettre à jour la mémoire.

Seules les transactions observées et leurs features alimentent le mécanisme temporel.

## 18.4 Interdictions

Pendant le Test :

- aucun `backward()` ;
- aucun `optimizer.step()` ;
- aucun recalcul de seuil ;
- aucune sélection de modèle ;
- aucune modification d'architecture ;
- aucune adaptation à partir des métriques Test.

Les résultats sont uniquement reportés et analysés.

In [102]:
# Recharger les décisions pré-Test depuis les artefacts
with open(
    model_selection_path,
    "r",
    encoding="utf-8",
) as f:
    frozen_selection = json.load(f)

with open(
    threshold_path,
    "r",
    encoding="utf-8",
) as f:
    frozen_dynamic_threshold = json.load(f)

with open(
    control_threshold_path,
    "r",
    encoding="utf-8",
) as f:
    frozen_control_threshold = json.load(f)

dynamic_test_threshold = float(
    frozen_dynamic_threshold["threshold"]
)

control_test_threshold = float(
    frozen_control_threshold["threshold"]
)

print(
    "Meilleur Dynamic figé :",
    frozen_selection["best_dynamic_model"],
)

print(
    "Contrôle figé :",
    frozen_selection["control_model"],
)

print(
    "Seuil Dynamic :",
    f"{dynamic_test_threshold:.10f}",
)

print(
    "Seuil contrôle :",
    f"{control_test_threshold:.10f}",
)

print(
    "Test utilisé pour sélection :",
    frozen_selection["test_used_for_selection"],
)

print(
    "Modèles autorisés :",
    frozen_selection["test_models"],
)

Meilleur Dynamic figé : dynamic_core_edge_tbptt2
Contrôle figé : no_memory_control
Seuil Dynamic : 0.9800333381
Seuil contrôle : 0.9987996817
Test utilisé pour sélection : False
Modèles autorisés : ['dynamic_core_edge_tbptt2', 'no_memory_control']


In [103]:
# Recharger le meilleur Dynamic depuis State_<=334
test_dynamic_model = PilotDynamicModel(
    edge_dim=13,
    memory_dim=8,
    hidden_dim=32,
)

test_loader_optimizer = torch.optim.AdamW(
    test_dynamic_model.parameters(),
    lr=1e-3,
    weight_decay=1e-5,
)

test_dynamic_restored = load_training_checkpoint(
    path=final_checkpoint_path,
    model=test_dynamic_model,
    optimizer=test_loader_optimizer,
    memory_dim=8,
)

test_dynamic_memory = test_dynamic_restored[
    "temporal_memory"
]

del test_loader_optimizer

test_dynamic_model.eval()

for parameter in test_dynamic_model.parameters():
    parameter.requires_grad_(False)

# Snapshot des poids
test_dynamic_params_before = {
    name: parameter.detach().clone()
    for name, parameter
    in test_dynamic_model.named_parameters()
}

# Rejouer causalement toute la Validation
replay_start = time.perf_counter()

for t in range(335, 393):

    batch = reader.read_step(t)

    with torch.no_grad():
        _, _, _ = process_pilot_step(
            model=test_dynamic_model,
            temporal_memory=test_dynamic_memory,
            batch=batch,
        )

    test_dynamic_memory.commit()

replay_params_unchanged = all(
    torch.equal(
        test_dynamic_params_before[name],
        parameter.detach(),
    )
    for name, parameter
    in test_dynamic_model.named_parameters()
)

replay_last_update = int(
    test_dynamic_memory.index.last_update[
        :test_dynamic_memory.index.size
    ].max()
)

print(
    "Checkpoint initial :",
    test_dynamic_restored["completed_step"],
)

print(
    "Validation rejouée : True",
)

print(
    "Slots après replay :",
    f"{test_dynamic_memory.index.size:,}",
)

print(
    "last_update max :",
    replay_last_update,
)

print(
    "Paramètres inchangés :",
    replay_params_unchanged,
)

print(
    "Overlay vide :",
    len(test_dynamic_memory.overlay) == 0,
)

print(
    "Runtime replay Validation :",
    f"{time.perf_counter() - replay_start:.2f} s",
)

Checkpoint initial : 334
Validation rejouée : True
Slots après replay : 2,375,723
last_update max : 392
Paramètres inchangés : True
Overlay vide : True
Runtime replay Validation : 17.36 s


In [104]:
dynamic_test_scores_blocks = []
test_labels_blocks = []
test_steps_blocks = []
test_type_blocks = []
test_src_known_blocks = []
test_dst_known_blocks = []

dynamic_test_start = time.perf_counter()

for t in range(393, 744):

    batch = reader.read_step(t)

    # Etat strictement antérieur
    _, src_known, _ = (
        test_dynamic_memory.index.lookup(
            batch["src"],
            t=t,
        )
    )

    _, dst_known, _ = (
        test_dynamic_memory.index.lookup(
            batch["dst"],
            t=t,
        )
    )

    with torch.no_grad():

        logits, y, info = process_pilot_step(
            model=test_dynamic_model,
            temporal_memory=test_dynamic_memory,
            batch=batch,
        )

        scores = torch.sigmoid(
            logits
        ).cpu().numpy().astype(
            np.float32,
            copy=False,
        )

    test_dynamic_memory.commit()

    type_idx = np.argmax(
        batch["x"][:, 2:7],
        axis=1,
    ).astype(np.int8)

    dynamic_test_scores_blocks.append(
        scores
    )

    test_labels_blocks.append(
        batch["label"].astype(
            np.int8,
            copy=True,
        )
    )

    test_steps_blocks.append(
        np.full(
            len(scores),
            t,
            dtype=np.int16,
        )
    )

    test_type_blocks.append(
        type_idx
    )

    test_src_known_blocks.append(
        src_known.astype(
            np.bool_,
            copy=True,
        )
    )

    test_dst_known_blocks.append(
        dst_known.astype(
            np.bool_,
            copy=True,
        )
    )

    if (
        t == 393
        or t % 50 == 0
        or t == 743
    ):
        print(
            f"Step {t}/743 | "
            f"tx={len(scores):,} | "
            f"fraudes={int(batch['label'].sum())} | "
            f"known_dst={int(dst_known.sum()):,} | "
            f"RAM libre="
            f"{psutil.virtual_memory().available / 1024**3:.2f} GB"
        )

dynamic_test_scores = np.concatenate(
    dynamic_test_scores_blocks
)

test_labels = np.concatenate(
    test_labels_blocks
)

test_steps = np.concatenate(
    test_steps_blocks
)

test_type_idx = np.concatenate(
    test_type_blocks
)

test_src_known = np.concatenate(
    test_src_known_blocks
)

test_dst_known = np.concatenate(
    test_dst_known_blocks
)

print()
print(
    "Transactions Test :",
    f"{len(dynamic_test_scores):,}",
)

print(
    "Fraudes Test :",
    f"{int(test_labels.sum()):,}",
)

print(
    "Runtime Dynamic Test :",
    f"{time.perf_counter() - dynamic_test_start:.2f} s",
)

Step 393/743 | tx=19,269 | fraudes=4 | known_dst=10,920 | RAM libre=3.28 GB
Step 400/743 | tx=30,910 | fraudes=16 | known_dst=18,415 | RAM libre=3.28 GB
Step 450/743 | tx=16 | fraudes=16 | known_dst=7 | RAM libre=3.32 GB
Step 500/743 | tx=6,108 | fraudes=4 | known_dst=3,348 | RAM libre=3.31 GB
Step 550/743 | tx=4,804 | fraudes=8 | known_dst=2,133 | RAM libre=3.27 GB
Step 600/743 | tx=18 | fraudes=18 | known_dst=5 | RAM libre=3.46 GB
Step 650/743 | tx=16 | fraudes=16 | known_dst=7 | RAM libre=3.46 GB
Step 700/743 | tx=6 | fraudes=6 | known_dst=3 | RAM libre=3.36 GB
Step 743/743 | tx=8 | fraudes=8 | known_dst=4 | RAM libre=3.36 GB

Transactions Test : 798,827
Fraudes Test : 3,830
Runtime Dynamic Test : 23.58 s


In [105]:
# Recharger le contrôle final
control_payload = torch.load(
    control_model_path,
    map_location="cpu",
    weights_only=False,
)

test_control_model = NoMemoryControl(
    edge_dim=13,
    memory_dim=8,
    hidden_dim=32,
)

test_control_model.load_state_dict(
    control_payload["model_state_dict"]
)

test_control_model.eval()

for parameter in test_control_model.parameters():
    parameter.requires_grad_(False)

control_test_scores_blocks = []

control_test_start = time.perf_counter()

with torch.no_grad():

    for t in range(393, 744):

        batch = reader.read_step(t)

        x = torch.from_numpy(
            batch["x"]
        )

        logits = test_control_model(
            x
        )

        scores = torch.sigmoid(
            logits
        ).cpu().numpy().astype(
            np.float32,
            copy=False,
        )

        control_test_scores_blocks.append(
            scores
        )

control_test_scores = np.concatenate(
    control_test_scores_blocks
)

print(
    "Transactions contrôle :",
    f"{len(control_test_scores):,}",
)

print(
    "Scores finis :",
    bool(
        np.isfinite(
            control_test_scores
        ).all()
    ),
)

print(
    "Runtime contrôle Test :",
    f"{time.perf_counter() - control_test_start:.2f} s",
)

Transactions contrôle : 798,827
Scores finis : True
Runtime contrôle Test : 1.93 s


In [106]:
# Invariants Test
test_tx_ok = (
    len(test_labels) == 798_827
)

test_fraud_ok = (
    int(test_labels.sum()) == 3_830
)

test_steps_ok = (
    test_steps.min() == 393
    and test_steps.max() == 743
)

# Ranking
dynamic_test_pr_auc = average_precision_score(
    test_labels,
    dynamic_test_scores,
)

dynamic_test_roc_auc = roc_auc_score(
    test_labels,
    dynamic_test_scores,
)

control_test_pr_auc = average_precision_score(
    test_labels,
    control_test_scores,
)

control_test_roc_auc = roc_auc_score(
    test_labels,
    control_test_scores,
)

# Seuils figés
dynamic_test_pred = (
    dynamic_test_scores
    >= dynamic_test_threshold
).astype(np.int8)

control_test_pred = (
    control_test_scores
    >= control_test_threshold
).astype(np.int8)

# Confusions
d_tn, d_fp, d_fn, d_tp = confusion_matrix(
    test_labels,
    dynamic_test_pred,
    labels=[0, 1],
).ravel()

c_tn, c_fp, c_fn, c_tp = confusion_matrix(
    test_labels,
    control_test_pred,
    labels=[0, 1],
).ravel()

print(
    "Transactions exactes :",
    test_tx_ok,
    f"({len(test_labels):,})",
)

print(
    "Fraudes exactes :",
    test_fraud_ok,
    f"({int(test_labels.sum()):,})",
)

print(
    "Steps exacts :",
    test_steps_ok,
)

print()
print("DYNAMIC TBPTT=2")
print(
    "PR-AUC :",
    f"{dynamic_test_pr_auc:.6f}",
)
print(
    "ROC-AUC :",
    f"{dynamic_test_roc_auc:.6f}",
)
print(
    "Precision :",
    f"{precision_score(test_labels, dynamic_test_pred, zero_division=0):.6f}",
)
print(
    "Recall :",
    f"{recall_score(test_labels, dynamic_test_pred, zero_division=0):.6f}",
)
print(
    "F1 :",
    f"{f1_score(test_labels, dynamic_test_pred, zero_division=0):.6f}",
)
print(
    "TP / FP / FN :",
    f"{d_tp:,} / {d_fp:,} / {d_fn:,}",
)

print()
print("NO MEMORY CONTROL")
print(
    "PR-AUC :",
    f"{control_test_pr_auc:.6f}",
)
print(
    "ROC-AUC :",
    f"{control_test_roc_auc:.6f}",
)
print(
    "Precision :",
    f"{precision_score(test_labels, control_test_pred, zero_division=0):.6f}",
)
print(
    "Recall :",
    f"{recall_score(test_labels, control_test_pred, zero_division=0):.6f}",
)
print(
    "F1 :",
    f"{f1_score(test_labels, control_test_pred, zero_division=0):.6f}",
)
print(
    "TP / FP / FN :",
    f"{c_tp:,} / {c_fp:,} / {c_fn:,}",
)

print()
print(
    "Delta PR-AUC Dynamic - contrôle :",
    f"{dynamic_test_pr_auc - control_test_pr_auc:+.6f}",
)

Transactions exactes : True (798,827)
Fraudes exactes : True (3,830)
Steps exacts : True

DYNAMIC TBPTT=2
PR-AUC : 0.689456
ROC-AUC : 0.978403
Precision : 0.945424
Recall : 0.434204
F1 : 0.595098
TP / FP / FN : 1,663 / 96 / 2,167

NO MEMORY CONTROL
PR-AUC : 0.749854
ROC-AUC : 0.987748
Precision : 0.937348
Recall : 0.503916
F1 : 0.655459
TP / FP / FN : 1,930 / 129 / 1,900

Delta PR-AUC Dynamic - contrôle : -0.060397


In [107]:
test_predictions_path = (
    dynamic_artifact_dir
    / "dynamic_gnn_final_test_predictions.npz"
)

test_metrics_path = (
    dynamic_artifact_dir
    / "dynamic_gnn_final_test_metrics.json"
)

np.savez_compressed(
    test_predictions_path,
    labels=test_labels,
    steps=test_steps,
    type_idx=test_type_idx,
    src_known=test_src_known,
    dst_known=test_dst_known,
    dynamic_scores=dynamic_test_scores,
    control_scores=control_test_scores,
)

test_metrics = {
    "split": "test",
    "start_step": 393,
    "end_step": 743,
    "transactions": int(
        len(test_labels)
    ),
    "frauds": int(
        test_labels.sum()
    ),
    "dynamic": {
        "model":
            "dynamic_core_edge_tbptt2",
        "threshold":
            dynamic_test_threshold,
        "pr_auc":
            float(dynamic_test_pr_auc),
        "roc_auc":
            float(dynamic_test_roc_auc),
        "precision":
            float(
                precision_score(
                    test_labels,
                    dynamic_test_pred,
                    zero_division=0,
                )
            ),
        "recall":
            float(
                recall_score(
                    test_labels,
                    dynamic_test_pred,
                    zero_division=0,
                )
            ),
        "f1":
            float(
                f1_score(
                    test_labels,
                    dynamic_test_pred,
                    zero_division=0,
                )
            ),
        "tp": int(d_tp),
        "fp": int(d_fp),
        "fn": int(d_fn),
    },
    "control": {
        "model":
            "no_memory_control",
        "threshold":
            control_test_threshold,
        "pr_auc":
            float(control_test_pr_auc),
        "roc_auc":
            float(control_test_roc_auc),
        "precision":
            float(
                precision_score(
                    test_labels,
                    control_test_pred,
                    zero_division=0,
                )
            ),
        "recall":
            float(
                recall_score(
                    test_labels,
                    control_test_pred,
                    zero_division=0,
                )
            ),
        "f1":
            float(
                f1_score(
                    test_labels,
                    control_test_pred,
                    zero_division=0,
                )
            ),
        "tp": int(c_tp),
        "fp": int(c_fp),
        "fn": int(c_fn),
    },
    "test_used_for_selection": False,
}

with open(
    test_metrics_path,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        test_metrics,
        f,
        indent=2,
        ensure_ascii=False,
    )

print(
    "Prédictions Test sauvegardées :",
    test_predictions_path.exists(),
)

print(
    "Métriques Test sauvegardées :",
    test_metrics_path.exists(),
)

print(
    "Test utilisé pour sélection :",
    test_metrics[
        "test_used_for_selection"
    ],
)

Prédictions Test sauvegardées : True
Métriques Test sauvegardées : True
Test utilisé pour sélection : False


## 18.5 Résultats globaux sur le Test

L'évaluation finale a été réalisée sur l'intégralité du Test :

$$
393 \leq t \leq 743
$$

soit :

$$
798\,827
$$

transactions et :

$$
3\,830
$$

fraudes.

Les décisions expérimentales avaient été entièrement figées avant cette évaluation.

Le Test n'a été utilisé ni pour :

- sélectionner un modèle ;
- modifier une architecture ;
- ajuster un seuil ;
- choisir un hyperparamètre.

### Continuité temporelle du Dynamic GNN

Le Dynamic GNN a été restauré depuis :

$$
State_{\leq334}
$$

puis la Validation a été rejouée causalement, avec des poids figés, afin de reconstruire :

$$
State_{\leq392}
$$

Avant le premier événement Test :

- slots actifs : **2 375 723** ;
- `last_update` maximal : **392** ;
- paramètres inchangés : **True** ;
- overlay vide : **True**.

Le Test commence donc exactement à partir de l'historique qui aurait été disponible au step 393.

### Résultats du Dynamic GNN

Le meilleur candidat dynamique, `TBPTT=2`, obtient :

| Métrique | Test |
| --- | ---: |
| PR-AUC | **0.689456** |
| ROC-AUC | 0.978403 |
| Precision | 0.945424 |
| Recall | 0.434204 |
| F1 | 0.595098 |
| TP | 1 663 |
| FP | 96 |
| FN | 2 167 |

Le seuil appliqué est exactement celui sélectionné sur Validation :

$$
\tau^*_{Dynamic}=0.9800333381
$$

### Résultats du contrôle sans mémoire

Le contrôle neuronal obtient :

| Métrique | Test |
| --- | ---: |
| PR-AUC | **0.749854** |
| ROC-AUC | 0.987748 |
| Precision | 0.937348 |
| Recall | 0.503916 |
| F1 | 0.655459 |
| TP | 1 930 |
| FP | 129 |
| FN | 1 900 |

Son seuil est également celui sélectionné exclusivement sur Validation :

$$
\tau^*_{Control}=0.9987996817
$$

### Comparaison Dynamic vs contrôle

La différence de PR-AUC sur le Test est :

$$
0.689456-0.749854
=
\mathbf{-0.060397}
$$

Le résultat est remarquablement proche de celui obtenu sur Validation :

$$
\Delta AP_{Val}
=
-0.059197
$$

Cette stabilité hors échantillon renforce la conclusion de l'ablation.

La mémoire temporelle destination-centric utilisée dans le Dynamic GNN ne produit pas de gain de ranking par rapport au prédicteur neuronal sans mémoire.

Le contrôle détecte également :

$$
1\,930-1\,663
=
267
$$

fraudes supplémentaires, au prix de seulement :

$$
129-96
=
33
$$

faux positifs supplémentaires.

Son F1 est donc supérieur :

$$
0.655459
>
0.595098
$$

### Comparaison avec les modèles précédents

Les PR-AUC Test disponibles sont :

| Modèle | PR-AUC Test |
| --- | ---: |
| GraphSAGE Core Edge | 0.352779 |
| Dynamic GNN TBPTT=2 | 0.689456 |
| NoMemory Control | **0.749854** |
| LightGBM Core | **0.979749** |

Le Dynamic GNN améliore fortement le GraphSAGE statique :

$$
0.689456-0.352779
=
+0.336677
$$

Le contrôle sans mémoire l'améliore encore davantage :

$$
0.749854-0.352779
=
+0.397075
$$

Cependant, le LightGBM Core reste nettement supérieur aux deux modèles neuronaux.

### Effet du drift de prévalence

Le taux de fraude du Test est :

$$
\frac{3\,830}{798\,827}
\approx
0.47945\%
$$

contre environ :

$$
0.08051\%
$$

sur la Validation.

La prévalence est donc environ **5,95 fois plus élevée** sur le Test.

Comme la PR-AUC dépend de la prévalence de la classe positive, les valeurs absolues de PR-AUC entre Validation et Test ne doivent pas être comparées comme si les deux splits avaient la même difficulté.

La comparaison la plus fiable reste celle entre modèles évalués sur exactement le même split.

### Conclusion globale

Le Test confirme les conclusions prises avant son ouverture :

1. le message passing spatial du GraphSAGE est peu adapté à la structure de PaySim ;
2. un prédicteur neuronal directement entraîné sur les Graph Core Edge constitue une baseline bien plus forte ;
3. la mémoire temporelle testée améliore le GraphSAGE, mais n'améliore pas le prédicteur neuronal sans mémoire ;
4. cette absence de gain de la mémoire est observée à la fois sur Validation et sur Test ;
5. le LightGBM Core demeure la référence la plus performante parmi les modèles Core étudiés.

Les résultats Test sont sauvegardés et aucune décision expérimentale supplémentaire ne sera prise à partir de ces métriques.

In [108]:
# Décoder les types du Test
test_type_name = type_names[
    test_type_idx
]

test_type_rows = []

for tx_type in type_categories:

    mask = (
        test_type_name == tx_type
    )

    y_true = test_labels[mask]

    dynamic_scores_subset = (
        dynamic_test_scores[mask]
    )

    control_scores_subset = (
        control_test_scores[mask]
    )

    dynamic_pred_subset = (
        dynamic_test_pred[mask]
    )

    control_pred_subset = (
        control_test_pred[mask]
    )

    frauds = int(
        y_true.sum()
    )

    if frauds == 0 or frauds == len(y_true):
        dynamic_ap = np.nan
        control_ap = np.nan
    else:
        dynamic_ap = average_precision_score(
            y_true,
            dynamic_scores_subset,
        )

        control_ap = average_precision_score(
            y_true,
            control_scores_subset,
        )

    dynamic_tp = int(
        (
            (y_true == 1)
            & (dynamic_pred_subset == 1)
        ).sum()
    )

    control_tp = int(
        (
            (y_true == 1)
            & (control_pred_subset == 1)
        ).sum()
    )

    dynamic_fp = int(
        (
            (y_true == 0)
            & (dynamic_pred_subset == 1)
        ).sum()
    )

    control_fp = int(
        (
            (y_true == 0)
            & (control_pred_subset == 1)
        ).sum()
    )

    test_type_rows.append({
        "type": tx_type,
        "transactions": int(mask.sum()),
        "frauds": frauds,
        "dynamic_pr_auc": dynamic_ap,
        "control_pr_auc": control_ap,
        "delta_pr_auc": (
            dynamic_ap - control_ap
            if not np.isnan(dynamic_ap)
            else np.nan
        ),
        "dynamic_tp": dynamic_tp,
        "control_tp": control_tp,
        "dynamic_fp": dynamic_fp,
        "control_fp": control_fp,
    })

test_type_analysis = pd.DataFrame(
    test_type_rows
)

print(
    test_type_analysis.to_string(
        index=False,
        formatters={
            "dynamic_pr_auc":
                lambda x: (
                    "NA"
                    if np.isnan(x)
                    else f"{x:.6f}"
                ),
            "control_pr_auc":
                lambda x: (
                    "NA"
                    if np.isnan(x)
                    else f"{x:.6f}"
                ),
            "delta_pr_auc":
                lambda x: (
                    "NA"
                    if np.isnan(x)
                    else f"{x:+.6f}"
                ),
        },
    )
)

    type  transactions  frauds dynamic_pr_auc control_pr_auc delta_pr_auc  dynamic_tp  control_tp  dynamic_fp  control_fp
 CASH_IN        178958       0            NaN            NaN          NaN           0           0           0           0
CASH_OUT        271051    1915       0.582895       0.659195    -0.076299         741         876           0          12
   DEBIT          5739       0            NaN            NaN          NaN           0           0           0           0
 PAYMENT        272255       0            NaN            NaN          NaN           0           0           0           0
TRANSFER         70824    1915       0.797322       0.853889    -0.056567         922        1054          96         117


In [109]:
# Construire les catégories Test
test_cold_category = np.full(
    len(test_labels),
    "",
    dtype="<U3",
)

test_cold_category[
    test_src_known
    & test_dst_known
] = "K→K"

test_cold_category[
    ~test_src_known
    & test_dst_known
] = "N→K"

test_cold_category[
    test_src_known
    & ~test_dst_known
] = "K→N"

test_cold_category[
    ~test_src_known
    & ~test_dst_known
] = "N→N"


test_cold_rows = []

for category in [
    "K→K",
    "N→K",
    "K→N",
    "N→N",
]:

    mask = (
        test_cold_category == category
    )

    y_true = test_labels[mask]

    dynamic_scores_subset = (
        dynamic_test_scores[mask]
    )

    control_scores_subset = (
        control_test_scores[mask]
    )

    dynamic_pred_subset = (
        dynamic_test_pred[mask]
    )

    control_pred_subset = (
        control_test_pred[mask]
    )

    frauds = int(
        y_true.sum()
    )

    if frauds == 0 or frauds == len(y_true):
        dynamic_ap = np.nan
        control_ap = np.nan
    else:
        dynamic_ap = average_precision_score(
            y_true,
            dynamic_scores_subset,
        )

        control_ap = average_precision_score(
            y_true,
            control_scores_subset,
        )

    dynamic_tp = int(
        (
            (y_true == 1)
            & (dynamic_pred_subset == 1)
        ).sum()
    )

    control_tp = int(
        (
            (y_true == 1)
            & (control_pred_subset == 1)
        ).sum()
    )

    dynamic_fp = int(
        (
            (y_true == 0)
            & (dynamic_pred_subset == 1)
        ).sum()
    )

    control_fp = int(
        (
            (y_true == 0)
            & (control_pred_subset == 1)
        ).sum()
    )

    test_cold_rows.append({
        "category": category,
        "transactions": int(mask.sum()),
        "frauds": frauds,
        "dynamic_pr_auc": dynamic_ap,
        "control_pr_auc": control_ap,
        "delta_pr_auc": (
            dynamic_ap - control_ap
            if not np.isnan(dynamic_ap)
            else np.nan
        ),
        "dynamic_tp": dynamic_tp,
        "control_tp": control_tp,
        "dynamic_fp": dynamic_fp,
        "control_fp": control_fp,
    })

test_cold_analysis = pd.DataFrame(
    test_cold_rows
)

print(
    test_cold_analysis.to_string(
        index=False,
        formatters={
            "dynamic_pr_auc":
                lambda x: (
                    "NA"
                    if np.isnan(x)
                    else f"{x:.6f}"
                ),
            "control_pr_auc":
                lambda x: (
                    "NA"
                    if np.isnan(x)
                    else f"{x:.6f}"
                ),
            "delta_pr_auc":
                lambda x: (
                    "NA"
                    if np.isnan(x)
                    else f"{x:+.6f}"
                ),
        },
    )
)

category  transactions  frauds dynamic_pr_auc control_pr_auc delta_pr_auc  dynamic_tp  control_tp  dynamic_fp  control_fp
     K→K            97       0            NaN            NaN          NaN           0           0           0           0
     N→K        451490    1402       0.516675       0.597966    -0.081292         495         608          40          79
     K→N            76       0            NaN            NaN          NaN           0           0           0           0
     N→N        347164    2428       0.791796       0.847579    -0.055783        1168        1322          56          50


In [110]:
# Construire les catégories Test
test_cold_category = np.full(
    len(test_labels),
    "",
    dtype="<U3",
)

test_cold_category[
    test_src_known
    & test_dst_known
] = "K→K"

test_cold_category[
    ~test_src_known
    & test_dst_known
] = "N→K"

test_cold_category[
    test_src_known
    & ~test_dst_known
] = "K→N"

test_cold_category[
    ~test_src_known
    & ~test_dst_known
] = "N→N"


test_cold_rows = []

for category in [
    "K→K",
    "N→K",
    "K→N",
    "N→N",
]:

    mask = (
        test_cold_category == category
    )

    y_true = test_labels[mask]

    dynamic_scores_subset = (
        dynamic_test_scores[mask]
    )

    control_scores_subset = (
        control_test_scores[mask]
    )

    dynamic_pred_subset = (
        dynamic_test_pred[mask]
    )

    control_pred_subset = (
        control_test_pred[mask]
    )

    frauds = int(
        y_true.sum()
    )

    if frauds == 0 or frauds == len(y_true):
        dynamic_ap = np.nan
        control_ap = np.nan
    else:
        dynamic_ap = average_precision_score(
            y_true,
            dynamic_scores_subset,
        )

        control_ap = average_precision_score(
            y_true,
            control_scores_subset,
        )

    dynamic_tp = int(
        (
            (y_true == 1)
            & (dynamic_pred_subset == 1)
        ).sum()
    )

    control_tp = int(
        (
            (y_true == 1)
            & (control_pred_subset == 1)
        ).sum()
    )

    dynamic_fp = int(
        (
            (y_true == 0)
            & (dynamic_pred_subset == 1)
        ).sum()
    )

    control_fp = int(
        (
            (y_true == 0)
            & (control_pred_subset == 1)
        ).sum()
    )

    test_cold_rows.append({
        "category": category,
        "transactions": int(mask.sum()),
        "frauds": frauds,
        "dynamic_pr_auc": dynamic_ap,
        "control_pr_auc": control_ap,
        "delta_pr_auc": (
            dynamic_ap - control_ap
            if not np.isnan(dynamic_ap)
            else np.nan
        ),
        "dynamic_tp": dynamic_tp,
        "control_tp": control_tp,
        "dynamic_fp": dynamic_fp,
        "control_fp": control_fp,
    })

test_cold_analysis = pd.DataFrame(
    test_cold_rows
)

print(
    test_cold_analysis.to_string(
        index=False,
        formatters={
            "dynamic_pr_auc":
                lambda x: (
                    "NA"
                    if np.isnan(x)
                    else f"{x:.6f}"
                ),
            "control_pr_auc":
                lambda x: (
                    "NA"
                    if np.isnan(x)
                    else f"{x:.6f}"
                ),
            "delta_pr_auc":
                lambda x: (
                    "NA"
                    if np.isnan(x)
                    else f"{x:+.6f}"
                ),
        },
    )
)

category  transactions  frauds dynamic_pr_auc control_pr_auc delta_pr_auc  dynamic_tp  control_tp  dynamic_fp  control_fp
     K→K            97       0            NaN            NaN          NaN           0           0           0           0
     N→K        451490    1402       0.516675       0.597966    -0.081292         495         608          40          79
     K→N            76       0            NaN            NaN          NaN           0           0           0           0
     N→N        347164    2428       0.791796       0.847579    -0.055783        1168        1322          56          50


## 19.3 Analyse finale par type de transaction et régime Known/New

L'analyse détaillée du Test confirme que l'écart entre le Dynamic GNN et le contrôle sans mémoire n'est pas concentré dans un seul sous-groupe.

Il apparaît dans toutes les catégories où des fraudes sont effectivement présentes.

### Analyse par type de transaction

Les fraudes du Test sont réparties uniquement entre :

- `CASH_OUT` : **1 915 fraudes** ;
- `TRANSFER` : **1 915 fraudes**.

Les autres types ne contiennent aucune fraude.

| Type | Fraudes | PR-AUC Dynamic | PR-AUC Control | $\Delta$ Dynamic-Control |
| --- | ---: | ---: | ---: | ---: |
| CASH_OUT | 1 915 | 0.582895 | **0.659195** | **-0.076299** |
| TRANSFER | 1 915 | 0.797322 | **0.853889** | **-0.056567** |

Le contrôle sans mémoire est donc supérieur sur les deux types frauduleux.

#### CASH_OUT

Avec le seuil figé sur Validation :

- Dynamic : **741 TP**, 0 FP ;
- Control : **876 TP**, 12 FP.

Le rappel passe ainsi approximativement de :

$$
38.69\%
$$

à :

$$
45.74\%
$$

avec le contrôle.

Il détecte donc **135 fraudes CASH_OUT supplémentaires**, au prix de seulement 12 faux positifs.

#### TRANSFER

Pour `TRANSFER` :

- Dynamic : **922 TP**, 96 FP ;
- Control : **1 054 TP**, 117 FP.

Le rappel passe approximativement de :

$$
48.15\%
$$

à :

$$
55.04\%
$$

Le contrôle récupère donc **132 fraudes supplémentaires**, pour 21 faux positifs supplémentaires.

Le Dynamic GNN ne présente ainsi aucun avantage local sur les deux types où la détection de fraude est réellement évaluée.

### Analyse par régime Known/New

Le caractère inductif du dataset reste extrêmement marqué.

Sur le Test :

| Régime | Transactions | Fraudes |
| --- | ---: | ---: |
| K→K | 97 | 0 |
| N→K | 451 490 | 1 402 |
| K→N | 76 | 0 |
| N→N | 347 164 | 2 428 |

Les catégories impliquant une source déjà connue sont pratiquement inexistantes.

Les fraudes se concentrent donc exclusivement dans :

$$
N\rightarrow K
$$

et :

$$
N\rightarrow N
$$

#### Régime N→K

Ce régime constitue le cas théoriquement le plus favorable à la mémoire destination-centric :

- la source est nouvelle ;
- la destination possède déjà un historique.

Pourtant :

$$
PR\text{-}AUC_{Dynamic}=0.516675
$$

contre :

$$
PR\text{-}AUC_{Control}=0.597966
$$

soit :

$$
\Delta=-0.081292
$$

Il s'agit même de l'écart le plus défavorable observé parmi les sous-groupes principaux.

Au seuil figé :

- Dynamic : **495 TP**, 40 FP ;
- Control : **608 TP**, 79 FP.

Le rappel passe approximativement de :

$$
35.31\%
$$

à :

$$
43.37\%
$$

avec le contrôle.

Ce résultat est particulièrement important :

> même lorsque la destination dispose d'un historique exploitable, la mémoire temporelle apprise n'améliore pas la détection.

#### Régime N→N

Lorsque les deux comptes sont nouveaux :

$$
PR\text{-}AUC_{Dynamic}=0.791796
$$

contre :

$$
PR\text{-}AUC_{Control}=0.847579
$$

soit :

$$
\Delta=-0.055783
$$

Le contrôle reste donc également supérieur dans le régime où aucune mémoire historique n'est disponible au moment de la transaction.

Au seuil figé :

- Dynamic : **1 168 TP**, 56 FP ;
- Control : **1 322 TP**, 50 FP.

Le contrôle obtient ici un résultat particulièrement favorable : il détecte **154 fraudes supplémentaires tout en produisant 6 faux positifs de moins**.

### Lecture scientifique

Ces résultats rendent peu plausible l'hypothèse selon laquelle la faiblesse globale du Dynamic GNN serait uniquement due au poids des transactions cold-start.

Si la mémoire destination-centric contenait un signal prédictif utile, un avantage aurait normalement dû apparaître au minimum dans :

$$
N\rightarrow K
$$

où la destination est connue au moment de la prédiction.

Or c'est précisément dans cette catégorie que le Dynamic GNN présente son plus grand déficit de PR-AUC par rapport au contrôle.

La conclusion ne doit cependant pas être généralisée à tous les modèles temporels ou à tous les graphes transactionnels.

Elle concerne spécifiquement :

- PaySim ;
- la représentation transactionnelle utilisée ;
- les Graph Core Edge features ;
- la mémoire destination-centric inspirée de TGN testée dans ce notebook ;
- le protocole causal et inductif retenu.

### Conclusion de l'analyse Test

Le Test confirme simultanément trois observations :

1. le Dynamic GNN surpasse largement le GraphSAGE statique ;
2. le contrôle neuronal sans mémoire surpasse systématiquement le Dynamic GNN ;
3. cet avantage du contrôle persiste dans chaque sous-groupe frauduleux mesurable.

La mémoire temporelle testée n'apporte donc pas de valeur prédictive supplémentaire dans ce setting.

**Section 19 validée.**

# 20. Conclusion générale

## 20.1 Question étudiée

Ce notebook avait pour objectif d'évaluer si une modélisation temporelle explicite du graphe transactionnel pouvait améliorer la détection de fraude Mobile Money par rapport :

- à un GNN statique ;
- à un prédicteur neuronal sans mémoire ;
- à la référence tabulaire LightGBM Core.

L'expérience a été menée dans un cadre strictement :

- temporel ;
- causal ;
- inductif ;
- sans fuite d'information ;
- sans utilisation du Test pour la sélection.

## 20.2 Approche dynamique développée

Une architecture légère inspirée de TGN a été conçue afin de respecter les contraintes mémoire de la machine locale.

La représentation utilise :

- des features transactionnelles `Graph Core Edge` ;
- un état temporel partagé initialement pour les nouveaux nœuds ;
- une mémoire sparse concentrée sur les destinations observées ;
- un module de message ;
- une agrégation intra-step par moyenne ;
- une GRU pour la mise à jour temporelle ;
- un prédicteur d'arêtes.

La prédiction respecte systématiquement :

$$
State_{<t}
\rightarrow
Prediction(E_t)
\rightarrow
Update(E_t)
$$

Aucune transaction du step courant n'est utilisée pour prédire une autre transaction du même step.

## 20.3 Résultats principaux

Les performances principales sont :

| Modèle | PR-AUC Validation | PR-AUC Test |
| --- | ---: | ---: |
| GraphSAGE Core Edge | 0.240136 | 0.352779 |
| Dynamic GNN TBPTT=16 | 0.395144 | non retenu |
| Dynamic GNN TBPTT=2 | 0.547378 | 0.689456 |
| NoMemory Control | **0.606575** | **0.749854** |
| LightGBM Core | **0.929704** | **0.979749** |

Le Dynamic GNN améliore fortement le GraphSAGE statique.

Cependant, le contrôle neuronal sans mémoire reste supérieur au modèle dynamique :

Validation :

$$
0.547378-0.606575
=
-0.059197
$$

Test :

$$
0.689456-0.749854
=
-0.060397
$$

La stabilité de cet écart entre Validation et Test renforce fortement la robustesse de l'ablation.

## 20.4 Expérience TBPTT

L'analyse des réapparitions a montré que le premier `TBPTT=2` était très court par rapport à la temporalité réelle.

Seulement :

$$
4.37\%
$$

des fraudes avec destination connue étaient réellement traversables par le gradient.

Avec :

$$
TBPTT=16
$$

cette couverture atteint :

$$
29.62\%
$$

Les gradients du module de message et de la GRU deviennent également beaucoup plus réguliers.

Malgré cela, la PR-AUC Validation chute à :

$$
0.395144
$$

Ce résultat démontre que la faiblesse du modèle ne provenait pas uniquement d'une portée insuffisante du gradient.

Une mémoire mieux entraînée n'est pas nécessairement une mémoire plus prédictive.

## 20.5 Interprétation structurelle

Les résultats doivent être replacés dans la structure particulière de PaySim.

Le graphe présente notamment :

- environ 9,07 millions de nœuds ;
- 6,36 millions d'arêtes ;
- quasiment aucune répétition de couple dirigé ;
- aucune réciprocité ;
- aucune boucle ;
- une structure essentiellement composée de paires, étoiles et petits arbres ;
- environ 94,83 % des nœuds avec un degré total égal à 1 ;
- environ 99,7 % de nouvelles sources sur Validation et Test ;
- environ 43 % de nouvelles destinations.

Le graphe fournit donc relativement peu d'opportunités pour apprendre des représentations structurelles profondes ou des trajectoires récurrentes riches.

Le signal transactionnel local reste particulièrement important.

Cette configuration explique de manière plausible pourquoi :

- GraphSAGE est faible ;
- le contrôle neuronal fonctionne mieux ;
- la mémoire temporelle testée ne produit pas le gain attendu.

## 20.6 Réponse à la question scientifique du notebook

Dans les conditions expérimentales étudiées :

> **L'ajout d'une mémoire temporelle destination-centric à un prédicteur neuronal d'arêtes n'améliore pas la détection de fraude sur PaySim.**

Le meilleur résultat neuronal est obtenu sans mémoire temporelle explicite.

Le meilleur modèle global utilisant les features Core reste néanmoins LightGBM.

Cette conclusion ne signifie pas que les Dynamic GNN sont inefficaces pour la fraude financière en général.

Elle indique plutôt que :

> **la valeur d'un Graph Neural Network dépend fortement de la richesse relationnelle et temporelle réellement présente dans les données.**

Un dataset simulé dont les transactions possèdent très peu de récurrence relationnelle peut limiter fortement les bénéfices attendus du Graph Learning dynamique.

## 20.7 Contribution au mémoire

Ce résultat négatif constitue une contribution utile du PFE.

Le travail ne consiste pas simplement à montrer qu'un modèle sophistiqué fonctionne.

Il permet également de démontrer expérimentalement :

- quand le message passing spatial apporte peu ;
- quand une mémoire temporelle n'améliore pas un prédicteur plus simple ;
- pourquoi les propriétés structurelles du dataset doivent être étudiées avant de choisir une architecture Graph ML ;
- pourquoi une ablation sans mémoire est indispensable pour attribuer correctement les gains d'un modèle dynamique.

La complexité architecturale n'est donc pas une garantie de meilleure performance.

## 20.8 Limites

Plusieurs limites doivent être conservées pour l'interprétation finale :

1. **PaySim est un simulateur.**  
   Son graphe ne reproduit pas nécessairement toute la récurrence relationnelle d'un véritable réseau Mobile Money.

2. **Une seule famille de mémoire temporelle a été étudiée en profondeur.**  
   Les résultats ne permettent pas de rejeter TGN, TGAT, DyRep ou l'ensemble des Dynamic GNN.

3. **La mémoire est principalement destination-centric.**  
   Ce choix est motivé par le cold-start massif des sources et les contraintes mémoire.

4. **Les ressources computationnelles sont limitées.**  
   L'architecture a volontairement été conçue pour fonctionner sur CPU avec une RAM restreinte.

5. **Le modèle privilégie une expérimentation contrôlée.**  
   Une recherche exhaustive d'hyperparamètres aurait augmenté le risque de sur-adaptation à la Validation.

Ces limites encadrent la portée de la conclusion sans invalider les observations expérimentales.

## 20.9 Conclusion

Le notebook montre finalement que la modélisation dynamique est techniquement faisable sur le graphe PaySim complet avec des ressources locales limitées, mais que le signal temporel appris par la mémoire testée n'améliore pas la performance du prédicteur neuronal sans mémoire.

Le résultat central peut être résumé ainsi :

$$
GraphSAGE
<
Dynamic\ GNN
<
NoMemory\ Neural\ Control
<
LightGBM\ Core
$$

sur le critère principal de PR-AUC.

Cette hiérarchie fournit une base claire pour la discussion finale du mémoire et pour l'orientation vers des données transactionnelles réelles présentant une structure relationnelle plus riche.

# 21. Validation finale des artefacts

## 21.1 Objectif

Cette dernière section vérifie que les principaux artefacts produits par le notebook sont disponibles et cohérents.

Elle permet de garantir que les résultats du notebook pourront être :

- réutilisés dans le mémoire ;
- comparés aux notebooks précédents ;
- rechargés sans réentraîner les modèles ;
- exploités pour produire les tableaux et figures finales du projet.

Aucun nouvel entraînement ni aucune nouvelle sélection de modèle n'est effectué dans cette section.

In [112]:
# Artefacts essentiels du notebook
final_artifacts = {
    "preprocessing":
        dynamic_artifact_dir
        / "dynamic_gnn_preprocessing.json",

    "dynamic_tbptt2_checkpoint":
        final_checkpoint_path,

    "dynamic_tbptt2_validation_predictions":
        dynamic_artifact_dir
        / "dynamic_core_edge_validation_predictions.npz",

    "dynamic_tbptt2_validation_metrics":
        dynamic_artifact_dir
        / "dynamic_core_edge_validation_metrics.json",

    "dynamic_tbptt2_threshold":
        threshold_path,

    "dynamic_tbptt16_checkpoint":
        final_checkpoint16_path,

    "dynamic_tbptt16_validation_predictions":
        dynamic_artifact_dir
        / "dynamic_core_edge_tbptt16_validation_predictions.npz",

    "dynamic_tbptt16_validation_metrics":
        dynamic_artifact_dir
        / "dynamic_core_edge_tbptt16_validation_metrics.json",

    "no_memory_checkpoint":
        control_model_path,

    "no_memory_validation_predictions":
        dynamic_artifact_dir
        / "no_memory_control_validation_predictions.npz",

    "no_memory_validation_metrics":
        dynamic_artifact_dir
        / "no_memory_control_validation.json",

    "no_memory_threshold":
        control_threshold_path,

    "model_selection":
        model_selection_path,

    "final_test_predictions":
        test_predictions_path,

    "final_test_metrics":
        test_metrics_path,
}

all_artifacts_exist = all(
    path.exists()
    for path in final_artifacts.values()
)

print(
    "Tous les artefacts existent :",
    all_artifacts_exist,
)

print(
    "Nombre d'artefacts vérifiés :",
    len(final_artifacts),
)

print()

for name, path in final_artifacts.items():

    print(
        f"{name:<42} : "
        f"{path.exists()}"
    )

Tous les artefacts existent : True
Nombre d'artefacts vérifiés : 15

preprocessing                              : True
dynamic_tbptt2_checkpoint                  : True
dynamic_tbptt2_validation_predictions      : True
dynamic_tbptt2_validation_metrics          : True
dynamic_tbptt2_threshold                   : True
dynamic_tbptt16_checkpoint                 : True
dynamic_tbptt16_validation_predictions     : True
dynamic_tbptt16_validation_metrics         : True
no_memory_checkpoint                       : True
no_memory_validation_predictions           : True
no_memory_validation_metrics               : True
no_memory_threshold                        : True
model_selection                            : True
final_test_predictions                     : True
final_test_metrics                         : True


In [113]:
# Résumé scientifique final
dynamic_gnn_summary = {
    "notebook":
        "07_dynamic_gnn.ipynb",

    "status":
        "validated",

    "task":
        "temporal_edge_classification",

    "dataset":
        "PaySim",

    "protocol": {
        "train_steps": [1, 334],
        "validation_steps": [335, 392],
        "test_steps": [393, 743],
        "causal_prediction":
            "State_<t -> E_t -> Update(E_t)",
        "shuffle": False,
        "test_used_for_selection": False,
    },

    "best_dynamic_model": {
        "name":
            "dynamic_core_edge_tbptt2",
        "validation_pr_auc":
            float(val_pr_auc),
        "test_pr_auc":
            float(dynamic_test_pr_auc),
        "test_roc_auc":
            float(dynamic_test_roc_auc),
        "test_precision":
            float(
                precision_score(
                    test_labels,
                    dynamic_test_pred,
                    zero_division=0,
                )
            ),
        "test_recall":
            float(
                recall_score(
                    test_labels,
                    dynamic_test_pred,
                    zero_division=0,
                )
            ),
        "test_f1":
            float(
                f1_score(
                    test_labels,
                    dynamic_test_pred,
                    zero_division=0,
                )
            ),
        "threshold":
            float(dynamic_test_threshold),
    },

    "no_memory_control": {
        "validation_pr_auc":
            float(control_val_pr_auc),
        "test_pr_auc":
            float(control_test_pr_auc),
        "test_roc_auc":
            float(control_test_roc_auc),
        "test_precision":
            float(
                precision_score(
                    test_labels,
                    control_test_pred,
                    zero_division=0,
                )
            ),
        "test_recall":
            float(
                recall_score(
                    test_labels,
                    control_test_pred,
                    zero_division=0,
                )
            ),
        "test_f1":
            float(
                f1_score(
                    test_labels,
                    control_test_pred,
                    zero_division=0,
                )
            ),
        "threshold":
            float(control_test_threshold),
    },

    "tbptt16_ablation": {
        "validation_pr_auc":
            float(val16_pr_auc),
        "selected_for_test":
            False,
    },

    "references": {
        "graphsage_core_edge_validation_pr_auc":
            0.240136,

        "graphsage_core_edge_test_pr_auc":
            0.352779,

        "lightgbm_core_validation_pr_auc":
            0.929704,

        "lightgbm_core_test_pr_auc":
            0.979749,
    },

    "main_conclusion": (
        "The tested destination-centric temporal memory "
        "does not outperform the no-memory neural control "
        "on PaySim."
    ),
}

summary_path = (
    dynamic_artifact_dir
    / "dynamic_gnn_final_summary.json"
)

with open(
    summary_path,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        dynamic_gnn_summary,
        f,
        indent=2,
        ensure_ascii=False,
    )

print(
    "Résumé final sauvegardé :",
    summary_path.exists(),
)

print(
    "Statut :",
    dynamic_gnn_summary["status"],
)

print(
    "Test utilisé pour sélection :",
    dynamic_gnn_summary[
        "protocol"
    ][
        "test_used_for_selection"
    ],
)

print(
    "Meilleur Dynamic Test PR-AUC :",
    f"{dynamic_test_pr_auc:.6f}",
)

print(
    "Control Test PR-AUC :",
    f"{control_test_pr_auc:.6f}",
)

print(
    "LightGBM Core Test PR-AUC :",
    f"{0.979749:.6f}",
)

Résumé final sauvegardé : True
Statut : validated
Test utilisé pour sélection : False
Meilleur Dynamic Test PR-AUC : 0.689456
Control Test PR-AUC : 0.749854
LightGBM Core Test PR-AUC : 0.979749


In [114]:
# Vérification finale du notebook
final_checks = {
    "artifacts":
        all_artifacts_exist,

    "summary":
        summary_path.exists(),

    "train_complete":
        completed16_step == 334,

    "validation_complete":
        len(val16_scores) == 792_406,

    "test_complete":
        len(test_labels) == 798_827,

    "test_frauds":
        int(test_labels.sum()) == 3_830,

    "test_not_used_for_selection":
        not frozen_selection[
            "test_used_for_selection"
        ],
}

print("Validation finale")
print()

for name, value in final_checks.items():
    print(
        f"{name:<30} : {value}"
    )

print()

print(
    "07_dynamic_gnn.ipynb validated :",
    all(final_checks.values()),
)

print(
    "Artifacts ready for final report :",
    (
        all(final_checks.values())
        and summary_path.exists()
    ),
)

Validation finale

artifacts                      : True
summary                        : True
train_complete                 : True
validation_complete            : True
test_complete                  : True
test_frauds                    : True
test_not_used_for_selection    : True

07_dynamic_gnn.ipynb validated : True
Artifacts ready for final report : True
